# Knot invariant analyses: working notebook

Configure the project and data paths below. The original saved execution is in
`archive/paper_run_recorded.ipynb`. This working copy has no historical outputs.
For an inexpensive release check, open `release_check.ipynb` first.

**Do not use Run all as a release verification yet.** The historical geometry
block contains a failed balance gate, and some recovery cells require existing
artifacts. The failed gate is retained. See `../docs/REPRODUCIBILITY.md`.


In [ ]:
# Mount Drive only when running in Colab.
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

# Set KNOT_PROJECT_DIR, KNOT_DATA_DIR and KNOT_OUTPUT_DIR before execution,
# or edit these paths. Existing Colab defaults are retained.
PROJECT_DIR = Path(os.environ.get("KNOT_PROJECT_DIR",
    "/content/drive/MyDrive/consensus_hardness_refactored")).expanduser().resolve()
DATA_DIR = Path(os.environ.get("KNOT_DATA_DIR",
    "/content/drive/MyDrive/Colab Notebooks/data_invariants/Invariants")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("KNOT_OUTPUT_DIR",
    str(DATA_DIR / "processed_consensus_hardness/corrected_run_20260819"))).expanduser().resolve()
assert (PROJECT_DIR / "pyproject.toml").is_file(), PROJECT_DIR
os.environ.update(KNOT_PROJECT_DIR=str(PROJECT_DIR), KNOT_DATA_DIR=str(DATA_DIR),
                  KNOT_OUTPUT_DIR=str(OUTPUT_DIR), STAGE32_DATA_ROOT=str(DATA_DIR),
                  STAGE32_OUTPUT_ROOT=str(OUTPUT_DIR))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(PROJECT_DIR) + "[notebook,plots]"])
sys.path.insert(0, str(PROJECT_DIR / "src"))


Evidence collection is available after execution:
`python scripts/collect_release_evidence.py --root /path/to/corrected_run --out /path/to/evidence.zip`


The collector creates a local ZIP; download it through the notebook file browser.


In [ ]:
import consensus_hardness as ch
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG = ch.canonical_run_config()


## 00–01. Alignment, identity exclusion, and QC

In [ ]:
FILE_MAP = {
    'alex': 'Alexander_upto17.csv',
    'homfly': 'HomflyPt_upto15_MIRRORS.csv',
    'jones': 'Jones_upto17_MIRRORS.csv',
    'theta': 'theta_upto15.csv',
    'kh': 'even_KH_upto17.pkl',
}
REPRESENTATION_SPECS = {
    'Alexander': {'source': 'alex', 'feature_prefixes': ['A']},
    'Jones': {'source': 'jones', 'feature_prefixes': ['J']},
    'HOMFLY-PT': {'source': 'homfly', 'feature_prefixes': ['a']},
    'Theta': {'source': 'theta', 'feature_prefixes': ['T']},
    'Khovanov': {'source': 'kh', 'feature_prefixes': ['F_']},
}
aligned = ch.build_aligned_dataset(
    DATA_DIR, FILE_MAP, REPRESENTATION_SPECS,
    min_crossings=CONFIG.universe.min_crossings,
    max_crossings=CONFIG.universe.max_crossings,
    expected_n=CONFIG.universe.expected_n,
    expected_s_qc_corrections=CONFIG.universe.expected_s_qc_corrections,
    preferred_metadata_sources=['alex', 'jones', 'homfly', 'theta', 'kh'],
    output_dir=OUTPUT_DIR / '00_alignment',
)
meta, X_dict = aligned['meta'], aligned['X_dict']
aligned['universe_audit']

## 02–03. Primary PCA-SSE consensus and nested nulls

In [ ]:
primary = ch.run_primary_pca_analysis(
    meta, X_dict, OUTPUT_DIR, config=CONFIG, run_nulls=True
)
display(primary['summary'])
display(primary['null_summary'])

## 04+. Robustness modules

Run these after the primary artifacts are frozen: norm diagnostics and matching; EVR-99.9 and fixed-compression PCA; leave-one-representation-out and tail sensitivity; held-out PCA/AE; norm-conditioned and residualized hardness; continuous C_k and external baselines. Do not overwrite the primary tables.

In [ ]:
ROBUSTNESS_DIR = ch.stage_directory(
    OUTPUT_DIR,
    4,
    "pca_sensitivities",
)

loo_sets, loo_summary = ch.leave_one_representation_out(
    primary["hard_sets"],
    meta,
    s_col=CONFIG.s_col,
)

display(loo_summary)

loo_summary.to_csv(
    ROBUSTNESS_DIR / "leave_one_representation_out.csv",
    index=False,
)

print("Saved to:")
print(ROBUSTNESS_DIR)

# 05. PCA

Esta celda especifica se corre si cerró la sesión despues de correr 04+.

In [ ]:
import numpy as np
import pandas as pd

PRIMARY_DIR = OUTPUT_DIR / "02_primary_pca"
NULL_DIR = OUTPUT_DIR / "03_intersection_nulls"

# ------------------------------------------------------------
# Recover PCA scores
# ------------------------------------------------------------

saved_scores = np.load(
    PRIMARY_DIR / "primary_pca_scores.npz",
    allow_pickle=False,
)

dimensions = pd.read_csv(
    PRIMARY_DIR / "primary_pca_dimensions.csv"
).set_index("invariant")

pca_primary_results = {}

for name, X in X_dict.items():
    key = ch.safe_name(name)
    row = dimensions.loc[name]

    pca_primary_results[name] = {
        "sse": saved_scores[f"{key}__sse"],
        "mse": saved_scores[f"{key}__mse"],
        "nre": saved_scores[f"{key}__nre"],
        "k": int(row["k"]),
        "evr": float(row["actual_evr"]),
        "input_dim": int(row["input_dim"]),
        "compression_ratio": float(row["compression_ratio"]),
    }

hard_sets_saved = pd.read_csv(
    PRIMARY_DIR / "hard_sets_by_stable_id.csv"
)

pca_primary_hard_sets = {}

for invariant, group in hard_sets_saved.groupby("invariant"):
    knot_ids = set(group["knot_id_base"].astype(str))

    pca_primary_hard_sets[invariant] = ch.ids_to_indices(
        knot_ids,
        meta,
        id_col="knot_id_base",
    )

pca_primary_consensus = ch.consensus_from_hard_sets(
    pca_primary_hard_sets
)

assert len(pca_primary_consensus) == 292

print("Recovered primary consensus:", len(pca_primary_consensus))

primary_summary = pd.read_csv(
    PRIMARY_DIR / "primary_consensus_summary.csv"
).iloc[0].to_dict()

null_summary = pd.read_csv(
    NULL_DIR / "intersection_null_summary.csv"
)

primary = {
    "fixed_results": pca_primary_results,
    "hard_sets": pca_primary_hard_sets,
    "consensus": pca_primary_consensus,
    "summary": primary_summary,
    "null_summary": null_summary,
}

loo_sets, loo_summary = ch.leave_one_representation_out(
    pca_primary_hard_sets,
    meta,
    s_col=CONFIG.s_col,
)

display(loo_summary)

In [ ]:
# ============================================================
# EVR-99.9 sensitivity
# ============================================================

pca_999_results = ch.run_pca_fixed_k_for_representations(
    X_dict,
    CONFIG.pca.sensitivity_k_999,
)

pca_999_hard_sets = ch.build_hard_sets_from_fixed_results(
    pca_999_results,
    score_name="sse",
    tail_mass=CONFIG.pca.tail_mass,
    stable_ids=meta["knot_id_base"],
)

pca_999_consensus = ch.consensus_from_hard_sets(
    pca_999_hard_sets
)

pca_999_summary = ch.summarize_selected_set(
    "PCA EVR-99.9",
    pca_999_consensus,
    meta,
    s_col=CONFIG.s_col,
)

pca_99_vs_999 = ch.compare_consensus_sets(
    pca_primary_consensus,
    pca_999_consensus,
    name_a="PCA EVR-99",
    name_b="PCA EVR-99.9",
)

display(pca_999_summary)
display(pca_99_vs_999)

In [ ]:
# ============================================================
# Fixed-compression sensitivity
# ============================================================

pca_fixed_compression_results = (
    ch.run_pca_fixed_k_for_representations(
        X_dict,
        CONFIG.pca.fixed_compression_k,
    )
)

pca_fixed_compression_hard_sets = (
    ch.build_hard_sets_from_fixed_results(
        pca_fixed_compression_results,
        score_name="sse",
        tail_mass=CONFIG.pca.tail_mass,
        stable_ids=meta["knot_id_base"],
    )
)

pca_fixed_compression_consensus = (
    ch.consensus_from_hard_sets(
        pca_fixed_compression_hard_sets
    )
)

pca_fixed_compression_summary = ch.summarize_selected_set(
    "PCA fixed compression",
    pca_fixed_compression_consensus,
    meta,
    s_col=CONFIG.s_col,
)

pca_99_vs_fixed_compression = ch.compare_consensus_sets(
    pca_primary_consensus,
    pca_fixed_compression_consensus,
    name_a="PCA EVR-99",
    name_b="PCA fixed compression",
)

display(pca_fixed_compression_summary)
display(pca_99_vs_fixed_compression)

In [ ]:
# Summaries and overlaps
pd.DataFrame([pca_999_summary]).to_csv(
    ROBUSTNESS_DIR / "pca_evr999_summary.csv",
    index=False,
)

pd.DataFrame([pca_99_vs_999]).to_csv(
    ROBUSTNESS_DIR / "pca_99_vs_999_overlap.csv",
    index=False,
)

pd.DataFrame([pca_fixed_compression_summary]).to_csv(
    ROBUSTNESS_DIR / "pca_fixed_compression_summary.csv",
    index=False,
)

pd.DataFrame([pca_99_vs_fixed_compression]).to_csv(
    ROBUSTNESS_DIR / "pca_99_vs_fixed_compression_overlap.csv",
    index=False,
)

# Stable-ID hard sets
ch.save_hard_sets_by_id(
    pca_999_hard_sets,
    meta,
    ROBUSTNESS_DIR / "pca_evr999_hard_sets.csv",
)

ch.save_hard_sets_by_id(
    pca_fixed_compression_hard_sets,
    meta,
    ROBUSTNESS_DIR / "pca_fixed_compression_hard_sets.csv",
)

# Consensus members
ch.consensus_dataframe(
    pca_999_consensus,
    meta,
).to_csv(
    ROBUSTNESS_DIR / "pca_evr999_consensus_members.csv",
    index=False,
)

ch.consensus_dataframe(
    pca_fixed_compression_consensus,
    meta,
).to_csv(
    ROBUSTNESS_DIR / "pca_fixed_compression_consensus_members.csv",
    index=False,
)

## 05.1 Tail sensivity

In [ ]:
tail_sets_all5, tail_summary_all5 = ch.tail_mass_sensitivity(
    fixed_results=pca_primary_results,
    meta=meta,
    tail_masses=(0.005, 0.01, 0.02, 0.05),
    score_name="sse",
    s_col=CONFIG.s_col,
)

display(
    tail_summary_all5[
        [
            "tail_percent",
            "n",
            "median_sigma",
            "median_s",
            "sigma_ge_10_prop",
            "s_ge_10_prop",
            "s_ge_12_prop",
            "alternating_prop",
            "nonalt_s_gt_sigma_prop",
        ]
    ]
)

In [ ]:
NO_KH = (
    "Alexander",
    "Jones",
    "HOMFLY-PT",
    "Theta",
)

tail_sets_no_kh, tail_summary_no_kh = ch.tail_mass_sensitivity(
    fixed_results=pca_primary_results,
    meta=meta,
    tail_masses=(0.005, 0.01, 0.02, 0.05),
    include=NO_KH,
    score_name="sse",
    s_col=CONFIG.s_col,
)

display(
    tail_summary_no_kh[
        [
            "tail_percent",
            "n",
            "median_sigma",
            "median_s",
            "sigma_ge_10_prop",
            "s_ge_10_prop",
            "s_ge_12_prop",
            "alternating_prop",
            "nonalt_s_gt_sigma_prop",
        ]
    ]
)

In [ ]:
tail_summary_all5.to_csv(
    ROBUSTNESS_DIR / "tail_sensitivity_all5.csv",
    index=False,
)

tail_summary_no_kh.to_csv(
    ROBUSTNESS_DIR / "tail_sensitivity_without_khovanov.csv",
    index=False,
)

## 05.2 Enrichment and persistent membership

In [ ]:
# Primary and no-Khovanov dataframes
primary_consensus_df = ch.consensus_dataframe(
    pca_primary_consensus,
    meta,
)

no_kh_consensus = loo_sets["Without Khovanov"]

no_kh_consensus_df = ch.consensus_dataframe(
    no_kh_consensus,
    meta,
)

In [ ]:
# Enrichment tables
primary_enrichment = ch.standard_knot_enrichment_table(
    selected_df=primary_consensus_df,
    background_df=meta,
    s_col=CONFIG.s_col,
)

no_kh_enrichment = ch.standard_knot_enrichment_table(
    selected_df=no_kh_consensus_df,
    background_df=meta,
    s_col=CONFIG.s_col,
)

display(primary_enrichment)
display(no_kh_enrichment)

In [ ]:
persistent_rows = []

for minimum in (2, 3, 4, 5):
    selected_df, _ = ch.characterize_persistent_hard(
        pca_primary_hard_sets,
        meta,
        min_membership=minimum,
    )

    selected_indices = set(
        np.flatnonzero(
            selected_df["n_hard_invariant_tails"]
            .ge(minimum)
            .reindex(meta.index, fill_value=False)
            .to_numpy()
        )
    )

In [ ]:
membership_df, membership_summary = ch.membership_count_table(
    pca_primary_hard_sets,
    meta,
)

persistent_rows = []

for minimum in (2, 3, 4, 5):
    selected_indices = set(
        np.flatnonzero(
            membership_df["membership_count"].to_numpy() >= minimum
        )
    )

    row = ch.summarize_selected_set(
        f"At least {minimum}/5 tails",
        selected_indices,
        meta,
        s_col=CONFIG.s_col,
    )

    row["min_membership"] = minimum
    persistent_rows.append(row)

persistent_summary = pd.DataFrame(persistent_rows)

display(
    persistent_summary[
        [
            "min_membership",
            "n",
            "median_sigma",
            "median_s",
            "sigma_ge_10_prop",
            "s_ge_10_prop",
            "s_ge_12_prop",
            "alternating_prop",
            "nonalt_s_gt_sigma_prop",
        ]
    ]
)

# 06. Norms and descriptives models

In [ ]:
# ============================================================
# Standardized vector norms
# ============================================================

meta_norm, norm_results = ch.compute_standardized_norms(
    meta=meta,
    X_dict=X_dict,
)

meta_errors = ch.add_consensus_indicator(
    meta_norm,
    pca_primary_consensus,
    output_col="is_consensus_sse_099",
)

In [ ]:
pca_results_compat = {
    name: {
        0.99: result
    }
    for name, result in pca_primary_results.items()
}

meta_errors, pca_sse_cols, pca_nre_cols = (
    ch.add_pca_errors_to_metadata(
        meta=meta_errors,
        pca_results=pca_results_compat,
        X_dict=X_dict,
        alpha=0.99,
    )
)

In [ ]:
sq_norm_cols = [
    f"{ch.safe_name(name)}_sq_norm"
    for name in X_dict
]

log_sq_norm_cols = [
    f"{ch.safe_name(name)}_log_sq_norm"
    for name in X_dict
]

norm_compare_cols = (
    sq_norm_cols
    + log_sq_norm_cols
    + [
        "mean_sq_norm",
        "max_sq_norm",
        "mean_log_sq_norm",
        "max_log_sq_norm",
    ]
)

norm_consensus_comparison = (
    ch.compare_selected_vs_background(
        meta_errors,
        selected_col="is_consensus_sse_099",
        value_cols=norm_compare_cols,
    )
)

display(norm_consensus_comparison)

In [ ]:
norm_sse_correlations = ch.norm_error_correlations(
    meta_norm=meta_errors,
    pca_results=pca_results_compat,
    X_dict=X_dict,
    alpha=0.99,
    s_col=CONFIG.s_col,
)

display(norm_sse_correlations)

In [ ]:
predictors_structural = [
    "signature",
    CONFIG.s_col,
    "is_alternating",
    "number_of_crossings",
]

predictors_norm = [
    "mean_log_sq_norm",
    "max_log_sq_norm",
]

model_specs = {
    "structural": predictors_structural,
    "norm_only": predictors_norm,
    "structural_plus_norm": (
        predictors_structural + predictors_norm
    ),
}

cv_logistic_summary = ch.logistic_model_comparison(
    meta_errors,
    target="is_consensus_sse_099",
    model_specs=model_specs,
    n_splits=5,
    seed=42,
)

display(cv_logistic_summary)

In [ ]:
NORM_DIR = ch.stage_directory(
    OUTPUT_DIR,
    5,
    "norm_diagnostics",
)

norm_consensus_comparison.to_csv(
    NORM_DIR / "norm_consensus_comparison.csv",
    index=False,
)

norm_sse_correlations.to_csv(
    NORM_DIR / "norm_sse_correlations.csv",
    index=False,
)

cv_logistic_summary.to_csv(
    NORM_DIR / "cv_logistic_summary.csv",
    index=False,
)

np.savez_compressed(
    NORM_DIR / "standardized_norm_columns.npz",
    **{
        col: meta_errors[col].to_numpy()
        for col in norm_compare_cols
    },
)

# 07. PCA held-out

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Frozen target-free split
# ------------------------------------------------------------

assert np.array_equal(
    meta.index.to_numpy(),
    np.arange(len(meta)),
), "meta must have a positional RangeIndex aligned with X_dict"

train_idx, val_idx, test_idx = ch.make_train_val_test_indices(
    meta=meta,
    split_seed=42,
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    stratify_col=None,        # primary target-free split
)

assert not set(train_idx) & set(val_idx)
assert not set(train_idx) & set(test_idx)
assert not set(val_idx) & set(test_idx)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

print("\nTest signature distribution:")
display(
    meta.iloc[test_idx]["signature"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

print("\nTest s-invariant QC distribution:")
display(
    meta.iloc[test_idx][CONFIG.s_col]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

# ------------------------------------------------------------
# 2. Norms standardized using training data only
# ------------------------------------------------------------

meta_holdout, norm_objects_holdout = ch.compute_standardized_norms(
    meta=meta,
    X_dict=X_dict,
    fit_indices=train_idx,
)

# ------------------------------------------------------------
# 3. PCA fitted only on training knots
# ------------------------------------------------------------

pca_holdout_k99 = ch.run_holdout_pca_for_representations(
    X_dict=X_dict,
    k_dict=CONFIG.pca.primary_k,
    train_idx=train_idx,
    test_idx=test_idx,
)

pca_holdout_reconstruction = ch.summarize_holdout_reconstruction(
    pca_holdout_k99
)

display(pca_holdout_reconstruction)

# ------------------------------------------------------------
# 4. Hard sets defined only within test
# ------------------------------------------------------------

(
    pca_holdout_hard_sets,
    pca_holdout_consensus,
    pca_holdout_summary,
    pca_holdout_enrichment,
) = ch.summarize_heldout_run(
    result_dict=pca_holdout_k99,
    meta=meta,
    label="PCA held-out target-free k99",
    score_name="test_sse",
    tail_mass=CONFIG.pca.tail_mass,
)

print("\nHeld-out consensus summary:")
display(pd.DataFrame([pca_holdout_summary]))

print("\nHeld-out enrichment relative to test background:")
display(pca_holdout_enrichment)

print(
    "\nHeld-out consensus size:",
    len(pca_holdout_consensus),
)

# ------------------------------------------------------------
# 5. Add test errors to metadata for subsequent matching
# ------------------------------------------------------------

meta_holdout = ch.add_holdout_errors_to_meta(
    meta=meta_holdout,
    result_dict=pca_holdout_k99,
    score_prefix="pca_holdout_k99",
    test_idx=test_idx,
)

meta_holdout["is_pca_holdout_k99_consensus"] = False
meta_holdout.loc[
    sorted(pca_holdout_consensus),
    "is_pca_holdout_k99_consensus",
] = True

test_meta = meta_holdout.iloc[test_idx].copy()

print("\nTest metadata shape:", test_meta.shape)
print(
    "Consensus indicators in test:",
    int(test_meta["is_pca_holdout_k99_consensus"].sum()),
)

# ------------------------------------------------------------
# 6. Save checkpoint
# ------------------------------------------------------------

HELDOUT_DIR = OUTPUT_DIR / "06_heldout_pca_target_free"
HELDOUT_DIR.mkdir(parents=True, exist_ok=True)

split_labels = np.full(len(meta), "", dtype=object)
split_labels[train_idx] = "train"
split_labels[val_idx] = "validation"
split_labels[test_idx] = "test"

pd.DataFrame({
    "knot_id_base": meta["knot_id_base"].astype(str),
    "split": split_labels,
}).to_csv(
    HELDOUT_DIR / "frozen_split_seed42.csv",
    index=False,
)

pca_holdout_reconstruction.to_csv(
    HELDOUT_DIR / "reconstruction_summary.csv",
    index=False,
)

pd.DataFrame([pca_holdout_summary]).to_csv(
    HELDOUT_DIR / "consensus_summary.csv",
    index=False,
)

pca_holdout_enrichment.to_csv(
    HELDOUT_DIR / "consensus_enrichment.csv",
    index=False,
)

hard_id_rows = []

for invariant, indices in pca_holdout_hard_sets.items():
    ids = meta.iloc[sorted(indices)]["knot_id_base"].astype(str)

    hard_id_rows.append(
        pd.DataFrame({
            "invariant": invariant,
            "knot_id_base": ids.to_numpy(),
        })
    )

pd.concat(hard_id_rows, ignore_index=True).to_csv(
    HELDOUT_DIR / "hard_sets_by_stable_id.csv",
    index=False,
)

meta.iloc[sorted(pca_holdout_consensus)][
    ["knot_id_base", "signature", CONFIG.s_col,
     "number_of_crossings", "is_alternating"]
].to_csv(
    HELDOUT_DIR / "consensus_members.csv",
    index=False,
)

score_arrays = {}

for invariant, result_i in pca_holdout_k99.items():
    key = ch.safe_name(invariant)

    score_arrays[f"{key}_test_sse"] = result_i["test_sse"]
    score_arrays[f"{key}_test_mse"] = result_i["test_mse"]
    score_arrays[f"{key}_test_nre"] = result_i["test_nre"]

score_arrays["test_idx"] = test_idx

np.savez_compressed(
    HELDOUT_DIR / "heldout_scores.npz",
    **score_arrays,
)

joblib.dump(
    {
        invariant: {
            "scaler": result_i["scaler"],
            "pca": result_i["pca"],
            "k": result_i["k"],
        }
        for invariant, result_i in pca_holdout_k99.items()
    },
    HELDOUT_DIR / "heldout_pca_models.joblib",
)

print("\nSaved checkpoint to:")
print(HELDOUT_DIR)

## 07.1 overlap y matching condicionado por norma

In [ ]:
# ------------------------------------------------------------
# 1. Detailed held-out summary
# ------------------------------------------------------------

heldout_detailed_summary = pd.DataFrame([
    ch.summarize_selected_set(
        name="PCA held-out target-free k99",
        selected=pca_holdout_consensus,
        meta=meta,
        s_col=CONFIG.s_col,
    )
])

display(heldout_detailed_summary)

# ------------------------------------------------------------
# 2. Full-data consensus restricted to the frozen test set
# ------------------------------------------------------------

primary_consensus = set(primary["consensus"])
test_index_set = set(map(int, test_idx))

primary_consensus_in_test = (
    primary_consensus & test_index_set
)

heldout_vs_full_test = ch.compare_consensus_sets(
    primary_consensus_in_test,
    pca_holdout_consensus,
    name_a="Full-data PCA consensus restricted to test",
    name_b="Held-out PCA consensus",
)

display(pd.DataFrame([heldout_vs_full_test]))

heldout_detailed_summary.to_csv(
    HELDOUT_DIR / "detailed_consensus_summary.csv",
    index=False,
)

pd.DataFrame([heldout_vs_full_test]).to_csv(
    HELDOUT_DIR / "heldout_vs_full_test_overlap.csv",
    index=False,
)

In [ ]:
# ------------------------------------------------------------
# 3. Norm-nearest matched controls within the test set
# ------------------------------------------------------------

exact_cols = [
    "number_of_crossings",
    "is_alternating",
    "signature_bin",
]

norm_cols = [
    "mean_log_sq_norm",
    "max_log_sq_norm",
]

holdout_sse_cols = [
    f"{ch.safe_name(name)}_pca_holdout_k99_sse"
    for name in X_dict
]

holdout_nre_cols = [
    f"{ch.safe_name(name)}_pca_holdout_k99_nre"
    for name in X_dict
]

outcome_cols = (
    [
        "mean_pca_holdout_k99_sse",
        "max_pca_holdout_k99_sse",
        "mean_pca_holdout_k99_nre",
        "max_pca_holdout_k99_nre",
    ]
    + holdout_sse_cols
    + holdout_nre_cols
    + [
        "signature",
        CONFIG.s_col,
    ]
)

pca_caliper_summary, pca_caliper_results = (
    ch.run_caliper_sensitivity(
        df=test_meta,
        selected_col="is_pca_holdout_k99_consensus",
        exact_cols=exact_cols,
        norm_cols=norm_cols,
        outcome_cols=outcome_cols,
        ratio=5,
        calipers=(
            0.10,
            0.15,
            0.20,
            0.25,
            0.35,
            0.50,
            None,
        ),
        replace=True,
        seed=42,
    )
)

display(pca_caliper_summary)

pca_caliper_summary.to_csv(
    HELDOUT_DIR / "norm_matching_caliper_sensitivity.csv",
    index=False,
)

In [ ]:
# ------------------------------------------------------------
# Recover chosen PCA matching
# ------------------------------------------------------------

chosen_caliper_pca = 0.50

chosen = pca_caliper_results[chosen_caliper_pca]

pca_selected_nn = chosen["selected"]
pca_controls_nn = chosen["controls"]
pca_pairs_nn = chosen["pairs"]
pca_unmatched_nn = chosen["unmatched"]
pca_balance_nn = chosen["balance"]
pca_outcomes_nn = chosen["outcomes"].copy()

print("Selected matched:", len(pca_selected_nn))
print("Control rows:", len(pca_controls_nn))
print(
    "Unique controls:",
    pca_pairs_nn["control_index"].nunique(),
)
print("Pairs:", len(pca_pairs_nn))
print("Unmatched:", len(pca_unmatched_nn))

print("\nNorm balance:")
display(pca_balance_nn)

print("\nMatched outcomes:")
display(pca_outcomes_nn)

In [ ]:
from statsmodels.stats.multitest import multipletests

valid = pca_outcomes_nn["paired_wilcoxon_p"].notna()

pca_outcomes_nn["q_value"] = np.nan

_, q_values, _, _ = multipletests(
    pca_outcomes_nn.loc[
        valid,
        "paired_wilcoxon_p",
    ],
    method="fdr_bh",
)

pca_outcomes_nn.loc[valid, "q_value"] = q_values

pca_outcomes_nn["fdr_significant"] = (
    pca_outcomes_nn["q_value"] < 0.05
)

display(
    pca_outcomes_nn.sort_values("q_value")
)

In [ ]:
# ------------------------------------------------------------
# Concordance-gap outcome among matched non-alternating knots
# ------------------------------------------------------------

test_meta = test_meta.copy()

test_meta["s_minus_sigma_qc"] = (
    test_meta[CONFIG.s_col]
    - test_meta["signature"]
)

matched_nonalt_indices = pca_selected_nn.index[
    pca_selected_nn["is_alternating"].eq(0)
]

pca_nonalt_pairs_nn = pca_pairs_nn[
    pca_pairs_nn["selected_index"].isin(
        matched_nonalt_indices
    )
].copy()

pca_nonalt_gap_outcomes = (
    ch.compare_matched_outcomes_grouped(
        df=test_meta,
        matched_pairs=pca_nonalt_pairs_nn,
        outcome_cols=[
            "signature",
            CONFIG.s_col,
            "s_minus_sigma_qc",
        ],
    )
)

print(
    "Matched non-alternating selected knots:",
    pca_nonalt_pairs_nn["selected_index"].nunique(),
)

display(pca_nonalt_gap_outcomes)

In [ ]:
# ------------------------------------------------------------
# Matching audit
# ------------------------------------------------------------

print("Unmatched audit:")

if len(pca_unmatched_nn):
    display(
        pca_unmatched_nn
        .groupby(
            ["reason"] + exact_cols,
            dropna=False,
        )
        .size()
        .reset_index(name="n_unmatched")
    )

control_reuse = (
    pca_pairs_nn["control_index"]
    .value_counts()
    .rename_axis("control_index")
    .reset_index(name="times_used")
)

control_reuse["knot_id_base"] = (
    control_reuse["control_index"]
    .map(test_meta["knot_id_base"])
)

print("\nControl reuse:")
display(control_reuse.head(15))

print("\nControl-reuse distribution:")
display(
    control_reuse["times_used"]
    .value_counts()
    .sort_index()
    .rename_axis("times_used")
    .reset_index(name="n_unique_controls")
)

In [ ]:
pca_balance_nn.to_csv(
    HELDOUT_DIR / "chosen_caliper_050_balance.csv",
    index=False,
)

pca_outcomes_nn.to_csv(
    HELDOUT_DIR / "chosen_caliper_050_outcomes_fdr.csv",
    index=False,
)

pca_nonalt_gap_outcomes.to_csv(
    HELDOUT_DIR / "chosen_caliper_050_nonalt_gap.csv",
    index=False,
)

pca_pairs_nn.to_csv(
    HELDOUT_DIR / "chosen_caliper_050_pairs.csv",
    index=False,
)

pca_unmatched_nn.to_csv(
    HELDOUT_DIR / "chosen_caliper_050_unmatched.csv",
    index=False,
)

control_reuse.to_csv(
    HELDOUT_DIR / "chosen_caliper_050_control_reuse.csv",
    index=False,
)

pca_selected_nn.to_csv(
    HELDOUT_DIR / "chosen_caliper_050_selected.csv",
    index=False,
)

pca_controls_nn.to_csv(
    HELDOUT_DIR / "chosen_caliper_050_controls.csv",
    index=False,
)

## One-to-one matching without control replacement

In [ ]:
# ------------------------------------------------------------
# One-to-one matching without control replacement
# ------------------------------------------------------------

(
    pca_unique_caliper_summary,
    pca_unique_caliper_results,
) = ch.run_caliper_sensitivity(
    df=test_meta,
    selected_col="is_pca_holdout_k99_consensus",
    exact_cols=exact_cols,
    norm_cols=norm_cols,
    outcome_cols=outcome_cols,
    ratio=1,
    calipers=(
        0.20,
        0.25,
        0.35,
        0.50,
        0.75,
        None,
    ),
    replace=False,
    seed=42,
)

display(pca_unique_caliper_summary)

pca_unique_caliper_summary.to_csv(
    HELDOUT_DIR /
    "unique_control_caliper_sensitivity.csv",
    index=False,
)

In [ ]:
for caliper, result_i in pca_unique_caliper_results.items():
    pairs_i = result_i["pairs"]

    if len(pairs_i):
        assert pairs_i["control_index"].is_unique

print("All one-to-one matched controls are unique.")

In [ ]:
# ------------------------------------------------------------
# Chosen unique-control matching
# ------------------------------------------------------------

chosen_unique_caliper = 0.50

unique_result = pca_unique_caliper_results[
    chosen_unique_caliper
]

pca_unique_selected = unique_result["selected"]
pca_unique_controls = unique_result["controls"]
pca_unique_pairs = unique_result["pairs"]
pca_unique_unmatched = unique_result["unmatched"]
pca_unique_balance = unique_result["balance"]
pca_unique_outcomes = unique_result["outcomes"].copy()

assert pca_unique_pairs["control_index"].is_unique
assert len(pca_unique_selected) == len(pca_unique_controls)

print("Unique matched pairs:", len(pca_unique_pairs))
print("Unmatched:", len(pca_unique_unmatched))

display(pca_unique_balance)
display(pca_unique_outcomes)

In [ ]:
valid = pca_unique_outcomes[
    "paired_wilcoxon_p"
].notna()

pca_unique_outcomes["q_value"] = np.nan

_, q_values, _, _ = multipletests(
    pca_unique_outcomes.loc[
        valid,
        "paired_wilcoxon_p",
    ],
    method="fdr_bh",
)

pca_unique_outcomes.loc[
    valid,
    "q_value",
] = q_values

pca_unique_outcomes["fdr_significant"] = (
    pca_unique_outcomes["q_value"] < 0.05
)

display(
    pca_unique_outcomes.sort_values("q_value")
)

In [ ]:
# ------------------------------------------------------------
# Unique-control non-alternating gap
# ------------------------------------------------------------

unique_nonalt_selected_indices = (
    pca_unique_selected.index[
        pca_unique_selected[
            "is_alternating"
        ].eq(0)
    ]
)

pca_unique_nonalt_pairs = pca_unique_pairs[
    pca_unique_pairs["selected_index"].isin(
        unique_nonalt_selected_indices
    )
].copy()

pca_unique_nonalt_gap = (
    ch.compare_matched_outcomes_grouped(
        df=test_meta,
        matched_pairs=pca_unique_nonalt_pairs,
        outcome_cols=[
            "signature",
            CONFIG.s_col,
            "s_minus_sigma_qc",
        ],
    )
)

print(
    "Unique matched non-alternating pairs:",
    pca_unique_nonalt_pairs[
        "selected_index"
    ].nunique(),
)

display(pca_unique_nonalt_gap)

In [ ]:
pca_unique_balance.to_csv(
    HELDOUT_DIR /
    "unique_control_caliper_050_balance.csv",
    index=False,
)

pca_unique_outcomes.to_csv(
    HELDOUT_DIR /
    "unique_control_caliper_050_outcomes_fdr.csv",
    index=False,
)

pca_unique_nonalt_gap.to_csv(
    HELDOUT_DIR /
    "unique_control_caliper_050_nonalt_gap.csv",
    index=False,
)

pca_unique_pairs.to_csv(
    HELDOUT_DIR /
    "unique_control_caliper_050_pairs.csv",
    index=False,
)

pca_unique_unmatched.to_csv(
    HELDOUT_DIR /
    "unique_control_caliper_050_unmatched.csv",
    index=False,
)

# 08. Autoencoder held-out multisemilla

In [ ]:
import gc
import joblib
import tensorflow as tf

print("GPUs:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "Activate a GPU runtime before training the autoencoders."
    )

AE_DIR = ch.stage_directory(
    OUTPUT_DIR,
    7,
    "heldout_ae_target_free",
)

AE_MODEL_DIR = AE_DIR / "models"
AE_SCORE_DIR = AE_DIR / "scores"
AE_SCALER_DIR = AE_DIR / "scalers"

AE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
AE_SCORE_DIR.mkdir(parents=True, exist_ok=True)
AE_SCALER_DIR.mkdir(parents=True, exist_ok=True)

AE_SEEDS = (0, 1, 2, 3, 4)

# Store only the objects needed for summaries in memory.
ae_holdout_k99_runs = {}

for seed in AE_SEEDS:

    print("\n" + "#" * 80)
    print("AE HELD-OUT SEED:", seed)
    print("#" * 80)

    seed_runs = ch.train_heldout_autoencoders(
        X_dict=X_dict,
        latent_dims=CONFIG.pca.primary_k,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        seeds=(seed,),
        epochs=200,
        batch_size=4096,
        patience=20,
        learning_rate=1e-3,
        dropout=0.05,
        l2=1e-6,
        save_dir=AE_MODEL_DIR,
    )

    full_seed_result = seed_runs[seed]

    # Save reconstruction scores immediately.
    score_payload = {
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
    }

    for invariant, result_i in full_seed_result.items():
        key = ch.safe_name(invariant)

        score_payload[f"{key}_val_sse"] = result_i["val_sse"]
        score_payload[f"{key}_val_mse"] = result_i["val_mse"]
        score_payload[f"{key}_val_nre"] = result_i["val_nre"]

        score_payload[f"{key}_test_sse"] = result_i["test_sse"]
        score_payload[f"{key}_test_mse"] = result_i["test_mse"]
        score_payload[f"{key}_test_nre"] = result_i["test_nre"]

    np.savez_compressed(
        AE_SCORE_DIR / f"heldout_ae_seed_{seed}.npz",
        **score_payload,
    )

    # Save scalers separately.
    joblib.dump(
        {
            invariant: result_i["scaler"]
            for invariant, result_i
            in full_seed_result.items()
        },
        AE_SCALER_DIR / f"heldout_ae_scalers_seed_{seed}.joblib",
    )

    # Save training histories.
    for invariant, result_i in full_seed_result.items():
        history_df = pd.DataFrame(result_i["history"])

        history_df.to_csv(
            AE_DIR /
            (
                f"history_{ch.safe_name(invariant)}"
                f"_seed_{seed}.csv"
            ),
            index=False,
        )

    # Keep only lightweight information in memory.
    ae_holdout_k99_runs[seed] = {
        invariant: {
            "name": result_i["name"],
            "latent_dim": result_i["latent_dim"],
            "input_dim": result_i["input_dim"],
            "compression_ratio": result_i["compression_ratio"],
            "model_seed": result_i["model_seed"],
            "best_val_loss": result_i["best_val_loss"],
            "final_val_loss": result_i["final_val_loss"],
            "train_idx": result_i["train_idx"],
            "val_idx": result_i["val_idx"],
            "test_idx": result_i["test_idx"],
            "val_sse": result_i["val_sse"],
            "val_mse": result_i["val_mse"],
            "val_nre": result_i["val_nre"],
            "test_sse": result_i["test_sse"],
            "test_mse": result_i["test_mse"],
            "test_nre": result_i["test_nre"],
        }
        for invariant, result_i
        in full_seed_result.items()
    }

    # Completion marker.
    (
        AE_DIR / f"seed_{seed}_complete.txt"
    ).write_text(
        f"Completed seed {seed}\n",
        encoding="utf-8",
    )

    del seed_runs
    del full_seed_result

    tf.keras.backend.clear_session()
    gc.collect()

    print("Saved seed:", seed)

In [ ]:
(
    ae_holdout_summary,
    ae_holdout_enrichment,
    ae_holdout_consensus_sets,
    ae_holdout_hard_sets,
) = ch.summarize_heldout_ae_runs(
    ae_runs=ae_holdout_k99_runs,
    meta=meta,
    tail_mass=CONFIG.pca.tail_mass,
)

display(ae_holdout_summary)

ae_jaccard_matrix, ae_pairwise_jaccards = (
    ch.pairwise_jaccard_matrix(
        ae_holdout_consensus_sets
    )
)

display(ae_jaccard_matrix)
display(ae_pairwise_jaccards)

ae_holdout_summary.to_csv(
    AE_DIR / "ae_seed_summary.csv",
    index=False,
)

ae_holdout_enrichment.to_csv(
    AE_DIR / "ae_seed_enrichment.csv",
    index=False,
)

ae_jaccard_matrix.to_csv(
    AE_DIR / "ae_consensus_jaccard_matrix.csv"
)

ae_pairwise_jaccards.to_csv(
    AE_DIR / "ae_consensus_pairwise_jaccards.csv",
    index=False,
)

In [ ]:
# ------------------------------------------------------------
# AE consensus frequency across seeds
# ------------------------------------------------------------

ae_frequency = pd.Series(
    0,
    index=pd.Index(test_idx, name="row_index"),
    dtype=int,
)

for seed, consensus_set in ae_holdout_consensus_sets.items():
    ae_frequency.loc[list(consensus_set)] += 1

ae_frequency_distribution = (
    ae_frequency
    .value_counts()
    .sort_index()
    .rename_axis("n_ae_seeds")
    .reset_index(name="n_knots")
)

ae_frequency_distribution["percentage_test"] = (
    100
    * ae_frequency_distribution["n_knots"]
    / len(test_idx)
)

display(ae_frequency_distribution)

# Sets appearing in at least k of five seeds
ae_frequency_sets = {
    k: set(
        map(
            int,
            ae_frequency.index[
                ae_frequency >= k
            ].tolist(),
        )
    )
    for k in range(1, 6)
}

ae_frequency_summary = pd.DataFrame([
    ch.summarize_selected_set(
        name=f"AE held-out >= {k}/5 seeds",
        selected=ae_frequency_sets[k],
        meta=meta,
        s_col=CONFIG.s_col,
    )
    for k in range(1, 6)
])

display(ae_frequency_summary)


In [ ]:
# ------------------------------------------------------------
# Enrichment of stable AE sets
# ------------------------------------------------------------

test_background = meta.iloc[test_idx].copy()

ae_stable_enrichment_rows = []

for k in (3, 4, 5):

    selected_df = meta.iloc[
        sorted(ae_frequency_sets[k])
    ].copy()

    enrichment_i = (
        ch.standard_knot_enrichment_table(
            selected_df=selected_df,
            background_df=test_background,
            s_col=CONFIG.s_col,
        )
        .assign(min_seed_membership=k)
    )

    ae_stable_enrichment_rows.append(
        enrichment_i
    )

ae_stable_enrichment = pd.concat(
    ae_stable_enrichment_rows,
    ignore_index=True,
)

display(ae_stable_enrichment)

In [ ]:
# ------------------------------------------------------------
# PCA overlap with each AE seed and stable AE sets
# ------------------------------------------------------------

pca_ae_overlap_rows = []

for seed, ae_set in ae_holdout_consensus_sets.items():

    row = ch.compare_consensus_sets(
        pca_holdout_consensus,
        ae_set,
        name_a="PCA held-out",
        name_b=f"AE held-out seed {seed}",
    )

    row["analysis"] = f"AE seed {seed}"
    pca_ae_overlap_rows.append(row)

for k in (3, 4, 5):

    row = ch.compare_consensus_sets(
        pca_holdout_consensus,
        ae_frequency_sets[k],
        name_a="PCA held-out",
        name_b=f"AE held-out >= {k}/5",
    )

    row["analysis"] = f"AE >= {k}/5"
    pca_ae_overlap_rows.append(row)

pca_ae_overlap = pd.DataFrame(
    pca_ae_overlap_rows
)

display(pca_ae_overlap)

In [ ]:
ae_seed_sets = list(
    ae_holdout_consensus_sets.values()
)

ae_core_all_seeds = set.intersection(
    *ae_seed_sets
)

ae_union_all_seeds = set.union(
    *ae_seed_sets
)

print("AE strict core, 5/5:", len(ae_core_all_seeds))
print("AE majority, >=3/5:", len(ae_frequency_sets[3]))
print("AE union, >=1/5:", len(ae_union_all_seeds))

In [ ]:
ae_frequency_distribution.to_csv(
    AE_DIR / "ae_seed_membership_distribution.csv",
    index=False,
)

ae_frequency_summary.to_csv(
    AE_DIR / "ae_seed_membership_summary.csv",
    index=False,
)

ae_stable_enrichment.to_csv(
    AE_DIR / "ae_stable_consensus_enrichment.csv",
    index=False,
)

pca_ae_overlap.to_csv(
    AE_DIR / "pca_vs_ae_overlap.csv",
    index=False,
)

ae_frequency_members = (
    meta.iloc[test_idx][
        [
            "knot_id_base",
            "signature",
            CONFIG.s_col,
            "number_of_crossings",
            "is_alternating",
        ]
    ]
    .copy()
)

ae_frequency_members["ae_seed_membership"] = (
    ae_frequency.loc[
        ae_frequency_members.index
    ].to_numpy()
)

ae_frequency_members.query(
    "ae_seed_membership > 0"
).to_csv(
    AE_DIR / "ae_consensus_members_by_frequency.csv",
    index=False,
)

comparar reconstrucción AE contra PCA

In [ ]:
# ------------------------------------------------------------
# AE reconstruction performance across seeds
# ------------------------------------------------------------

ae_reconstruction_rows = []

for seed, seed_run in ae_holdout_k99_runs.items():

    for invariant, result_i in seed_run.items():

        ae_reconstruction_rows.append({
            "seed": seed,
            "invariant": invariant,
            "latent_dim": result_i["latent_dim"],
            "best_val_loss": result_i["best_val_loss"],
            "final_val_loss": result_i["final_val_loss"],
            "median_test_mse": float(
                np.median(result_i["test_mse"])
            ),
            "mean_test_mse": float(
                np.mean(result_i["test_mse"])
            ),
            "median_test_sse": float(
                np.median(result_i["test_sse"])
            ),
            "mean_test_sse": float(
                np.mean(result_i["test_sse"])
            ),
        })

ae_reconstruction_by_seed = pd.DataFrame(
    ae_reconstruction_rows
)

ae_reconstruction_summary = (
    ae_reconstruction_by_seed
    .groupby("invariant", as_index=False)
    .agg(
        ae_median_test_mse_mean=(
            "median_test_mse", "mean"
        ),
        ae_median_test_mse_sd=(
            "median_test_mse", "std"
        ),
        ae_mean_test_mse_mean=(
            "mean_test_mse", "mean"
        ),
        ae_mean_test_mse_sd=(
            "mean_test_mse", "std"
        ),
        ae_best_val_loss_mean=(
            "best_val_loss", "mean"
        ),
    )
)

ae_vs_pca_reconstruction = (
    ae_reconstruction_summary
    .merge(
        pca_holdout_reconstruction[
            [
                "invariant",
                "median_test_mse",
                "mean_test_mse",
            ]
        ].rename(columns={
            "median_test_mse":
                "pca_median_test_mse",
            "mean_test_mse":
                "pca_mean_test_mse",
        }),
        on="invariant",
        how="left",
    )
)

ae_vs_pca_reconstruction[
    "ae_to_pca_median_mse_ratio"
] = (
    ae_vs_pca_reconstruction[
        "ae_median_test_mse_mean"
    ]
    / ae_vs_pca_reconstruction[
        "pca_median_test_mse"
    ]
)

ae_vs_pca_reconstruction[
    "ae_to_pca_mean_mse_ratio"
] = (
    ae_vs_pca_reconstruction[
        "ae_mean_test_mse_mean"
    ]
    / ae_vs_pca_reconstruction[
        "pca_mean_test_mse"
    ]
)

display(ae_vs_pca_reconstruction)

ae_reconstruction_by_seed.to_csv(
    AE_DIR / "ae_reconstruction_by_seed.csv",
    index=False,
)

ae_vs_pca_reconstruction.to_csv(
    AE_DIR / "ae_vs_pca_reconstruction.csv",
    index=False,
)

Matching del consenso AE mayoritario

In [ ]:
# ------------------------------------------------------------
# Add seed-averaged AE errors to test metadata
# ------------------------------------------------------------

test_meta_ae = test_meta.copy()

ae_seeds = sorted(
    ae_holdout_k99_runs.keys()
)

ae_sse_cols = []
ae_nre_cols = []

for invariant in X_dict:

    key = ch.safe_name(invariant)

    for seed in ae_seeds:
        assert np.array_equal(
            ae_holdout_k99_runs[seed][invariant][
                "test_idx"
            ],
            test_idx,
        )

    seed_sse = np.stack([
        ae_holdout_k99_runs[seed][invariant][
            "test_sse"
        ]
        for seed in ae_seeds
    ])

    seed_nre = np.stack([
        ae_holdout_k99_runs[seed][invariant][
            "test_nre"
        ]
        for seed in ae_seeds
    ])

    sse_col = (
        f"{key}_ae_holdout_seedmean_sse"
    )

    nre_col = (
        f"{key}_ae_holdout_seedmean_nre"
    )

    test_meta_ae[sse_col] = seed_sse.mean(axis=0)
    test_meta_ae[nre_col] = seed_nre.mean(axis=0)

    ae_sse_cols.append(sse_col)
    ae_nre_cols.append(nre_col)

test_meta_ae[
    "mean_ae_holdout_seedmean_sse"
] = test_meta_ae[ae_sse_cols].mean(axis=1)

test_meta_ae[
    "max_ae_holdout_seedmean_sse"
] = test_meta_ae[ae_sse_cols].max(axis=1)

test_meta_ae[
    "mean_ae_holdout_seedmean_nre"
] = test_meta_ae[ae_nre_cols].mean(axis=1)

test_meta_ae[
    "max_ae_holdout_seedmean_nre"
] = test_meta_ae[ae_nre_cols].max(axis=1)

test_meta_ae[
    "is_ae_majority_consensus"
] = test_meta_ae.index.isin(
    ae_frequency_sets[3]
)

test_meta_ae["s_minus_sigma_qc"] = (
    test_meta_ae[CONFIG.s_col]
    - test_meta_ae["signature"]
)

assert (
    test_meta_ae[
        "is_ae_majority_consensus"
    ].sum()
    == 25
)

In [ ]:
ae_outcome_cols = (
    [
        "mean_ae_holdout_seedmean_sse",
        "max_ae_holdout_seedmean_sse",
        "mean_ae_holdout_seedmean_nre",
        "max_ae_holdout_seedmean_nre",
    ]
    + ae_sse_cols
    + ae_nre_cols
    + [
        "signature",
        CONFIG.s_col,
        "s_minus_sigma_qc",
    ]
)

# Five controls, allowing reuse
(
    ae_caliper_summary,
    ae_caliper_results,
) = ch.run_caliper_sensitivity(
    df=test_meta_ae,
    selected_col="is_ae_majority_consensus",
    exact_cols=exact_cols,
    norm_cols=norm_cols,
    outcome_cols=ae_outcome_cols,
    ratio=5,
    calipers=(
        0.20,
        0.25,
        0.35,
        0.50,
        0.75,
        None,
    ),
    replace=True,
    seed=42,
)

print("Matching with replacement:")
display(ae_caliper_summary)

# One unique control per selected knot
(
    ae_unique_caliper_summary,
    ae_unique_caliper_results,
) = ch.run_caliper_sensitivity(
    df=test_meta_ae,
    selected_col="is_ae_majority_consensus",
    exact_cols=exact_cols,
    norm_cols=norm_cols,
    outcome_cols=ae_outcome_cols,
    ratio=1,
    calipers=(
        0.20,
        0.25,
        0.35,
        0.50,
        0.75,
        None,
    ),
    replace=False,
    seed=42,
)

print("Unique-control matching:")
display(ae_unique_caliper_summary)

ae_caliper_summary.to_csv(
    AE_DIR /
    "ae_majority_caliper_sensitivity.csv",
    index=False,
)

ae_unique_caliper_summary.to_csv(
    AE_DIR /
    "ae_majority_unique_caliper_sensitivity.csv",
    index=False,
)

In [ ]:
display(ae_unique_caliper_summary)

In [ ]:
# ------------------------------------------------------------
# Recover chosen AE matchings
# ------------------------------------------------------------

AE_CALIPER = 0.50

ae_matched = ae_caliper_results[AE_CALIPER]
ae_unique = ae_unique_caliper_results[AE_CALIPER]

ae_selected_nn = ae_matched["selected"]
ae_controls_nn = ae_matched["controls"]
ae_pairs_nn = ae_matched["pairs"]
ae_unmatched_nn = ae_matched["unmatched"]
ae_balance_nn = ae_matched["balance"]
ae_outcomes_nn = ae_matched["outcomes"].copy()

ae_unique_selected = ae_unique["selected"]
ae_unique_controls = ae_unique["controls"]
ae_unique_pairs = ae_unique["pairs"]
ae_unique_unmatched = ae_unique["unmatched"]
ae_unique_balance = ae_unique["balance"]
ae_unique_outcomes = ae_unique["outcomes"].copy()

assert ae_unique_pairs["control_index"].is_unique

print("With replacement:", len(ae_selected_nn))
print("Unique control:", len(ae_unique_selected))

display(ae_balance_nn)
display(ae_unique_balance)

In [ ]:
def add_bh_fdr(table):
    out = table.copy()

    valid = out["paired_wilcoxon_p"].notna()
    out["q_value"] = np.nan

    if valid.any():
        _, qvals, _, _ = multipletests(
            out.loc[
                valid,
                "paired_wilcoxon_p",
            ],
            method="fdr_bh",
        )

        out.loc[valid, "q_value"] = qvals

    out["fdr_significant"] = (
        out["q_value"] < 0.05
    )

    return out


ae_outcomes_nn = add_bh_fdr(
    ae_outcomes_nn
)

ae_unique_outcomes = add_bh_fdr(
    ae_unique_outcomes
)

print("Outcomes with replacement:")
display(
    ae_outcomes_nn.sort_values("q_value")
)

print("Outcomes with unique controls:")
display(
    ae_unique_outcomes.sort_values("q_value")
)

In [ ]:
# ------------------------------------------------------------
# Non-alternating concordance gap
# ------------------------------------------------------------

def nonalternating_gap_analysis(
    selected_df,
    pairs_df,
    full_df,
):
    nonalt_indices = selected_df.index[
        selected_df["is_alternating"].eq(0)
    ]

    nonalt_pairs = pairs_df[
        pairs_df["selected_index"].isin(
            nonalt_indices
        )
    ].copy()

    outcomes = ch.compare_matched_outcomes_grouped(
        df=full_df,
        matched_pairs=nonalt_pairs,
        outcome_cols=[
            "signature",
            CONFIG.s_col,
            "s_minus_sigma_qc",
        ],
    )

    return nonalt_pairs, outcomes


ae_nonalt_pairs, ae_nonalt_gap = (
    nonalternating_gap_analysis(
        ae_selected_nn,
        ae_pairs_nn,
        test_meta_ae,
    )
)

(
    ae_unique_nonalt_pairs,
    ae_unique_nonalt_gap,
) = nonalternating_gap_analysis(
    ae_unique_selected,
    ae_unique_pairs,
    test_meta_ae,
)

print(
    "Matched non-alternating, replacement:",
    ae_nonalt_pairs[
        "selected_index"
    ].nunique(),
)

display(ae_nonalt_gap)

print(
    "Matched non-alternating, unique:",
    ae_unique_nonalt_pairs[
        "selected_index"
    ].nunique(),
)

display(ae_unique_nonalt_gap)

In [ ]:
def unmatched_audit(unmatched_df):
    if not len(unmatched_df):
        return pd.DataFrame()

    return (
        unmatched_df
        .groupby(
            ["reason"] + exact_cols,
            dropna=False,
        )
        .size()
        .reset_index(name="n_unmatched")
    )


ae_unmatched_audit = unmatched_audit(
    ae_unmatched_nn
)

ae_unique_unmatched_audit = unmatched_audit(
    ae_unique_unmatched
)

display(ae_unmatched_audit)
display(ae_unique_unmatched_audit)

In [ ]:
ae_outcomes_nn.to_csv(
    AE_DIR / "ae_majority_caliper_050_outcomes_fdr.csv",
    index=False,
)

ae_unique_outcomes.to_csv(
    AE_DIR /
    "ae_majority_unique_caliper_050_outcomes_fdr.csv",
    index=False,
)

ae_nonalt_gap.to_csv(
    AE_DIR / "ae_majority_caliper_050_nonalt_gap.csv",
    index=False,
)

ae_unique_nonalt_gap.to_csv(
    AE_DIR /
    "ae_majority_unique_caliper_050_nonalt_gap.csv",
    index=False,
)

ae_unmatched_audit.to_csv(
    AE_DIR / "ae_majority_caliper_050_unmatched_audit.csv",
    index=False,
)

ae_unique_unmatched_audit.to_csv(
    AE_DIR /
    "ae_majority_unique_caliper_050_unmatched_audit.csv",
    index=False,
)

# 09. hardness condicional por norma

In [ ]:
# ------------------------------------------------------------
# Conditional hardness within norm strata
# ------------------------------------------------------------

CONDITIONAL_DIR = ch.stage_directory(
    OUTPUT_DIR,
    8,
    "conditional_norm_hardness",
)

norm_col_map = {
    invariant: (
        f"{ch.safe_name(invariant)}_sq_norm"
    )
    for invariant in X_dict
}

for invariant, norm_col in norm_col_map.items():
    assert norm_col in meta_norm.columns
    assert len(meta_norm[norm_col]) == len(meta)

conditional_results_100, conditional_diagnostics_100 = (
    ch.norm_adjusted_scores(
        fixed_results=primary["fixed_results"],
        norm_meta=meta_norm,
        norm_col_map=norm_col_map,
        method="conditional_percentile",
        n_bins=CONFIG.norm_bins,  # 100
        seed=CONFIG.random_seed,
    )
)

display(conditional_diagnostics_100)

In [ ]:
conditional_hard_sets_100 = (
    ch.build_hard_sets_from_fixed_results(
        fixed_results=conditional_results_100,
        score_name="sse",
        tail_mass=CONFIG.pca.tail_mass,
        stable_ids=meta[
            CONFIG.universe.id_col
        ],
    )
)

print("Conditional hard-set sizes:")

for invariant, indices in (
    conditional_hard_sets_100.items()
):
    print(invariant, len(indices))

In [ ]:
INVARIANTS = tuple(X_dict.keys())

NO_KHOVANOV = tuple(
    name
    for name in INVARIANTS
    if name != "Khovanov"
)

POLYNOMIAL_ONLY = (
    "Alexander",
    "Jones",
    "HOMFLY-PT",
)

CONDITIONAL_FAMILIES = {
    "All 5": {
        "names": INVARIANTS,
        "thresholds": (3, 4, 5),
    },
    "No Khovanov": {
        "names": NO_KHOVANOV,
        "thresholds": (2, 3, 4),
    },
    "Polynomial only": {
        "names": POLYNOMIAL_ONLY,
        "thresholds": (2, 3),
    },
}

In [ ]:
INVARIANTS = tuple(X_dict.keys())

NO_KHOVANOV = tuple(
    name
    for name in INVARIANTS
    if name != "Khovanov"
)

POLYNOMIAL_ONLY = (
    "Alexander",
    "Jones",
    "HOMFLY-PT",
)

CONDITIONAL_FAMILIES = {
    "All 5": {
        "names": INVARIANTS,
        "thresholds": (3, 4, 5),
    },
    "No Khovanov": {
        "names": NO_KHOVANOV,
        "thresholds": (2, 3, 4),
    },
    "Polynomial only": {
        "names": POLYNOMIAL_ONLY,
        "thresholds": (2, 3),
    },
}

In [ ]:
conditional_membership_sets = {}
conditional_summary_rows = []

for family, specification in (
    CONDITIONAL_FAMILIES.items()
):

    names = specification["names"]
    m = len(names)

    for k in specification["thresholds"]:

        selected_set = ch.at_least_k_consensus(
            hard_sets=conditional_hard_sets_100,
            names=names,
            k=k,
            n_objects=len(meta),
        )

        conditional_membership_sets[
            (family, k)
        ] = selected_set

        row = ch.summarize_selected_set(
            name=f"{family}: >= {k}/{m}",
            selected=selected_set,
            meta=meta,
            s_col=CONFIG.s_col,
        )

        selected_df = meta.iloc[
            sorted(selected_set)
        ]

        nonalt = selected_df[
            selected_df["is_alternating"].eq(0)
        ]

        delta = (
            nonalt[CONFIG.s_col]
            - nonalt["signature"]
        )

        row.update({
            "family": family,
            "k": k,
            "m": m,
            "mean_s_minus_sigma": (
                float(delta.mean())
                if len(delta)
                else np.nan
            ),
            "delta_ge_4_prop": (
                float(delta.ge(4).mean())
                if len(delta)
                else np.nan
            ),
        })

        conditional_summary_rows.append(row)

conditional_membership_summary_100 = pd.DataFrame(
    conditional_summary_rows
)

display(
    conditional_membership_summary_100[
        [
            "family",
            "k",
            "m",
            "n",
            "median_sigma",
            "median_s",
            "sigma_ge_10_prop",
            "s_ge_10_prop",
            "s_ge_12_prop",
            "alternating_prop",
            "nonalternating_n",
            "nonalt_s_gt_sigma_prop",
            "mean_s_minus_sigma",
            "delta_ge_4_prop",
        ]
    ].sort_values(
        ["family", "k"]
    )
)

In [ ]:
conditional_diagnostics_100.to_csv(
    CONDITIONAL_DIR /
    "conditional_100bins_diagnostics.csv",
    index=False,
)

conditional_membership_summary_100.to_csv(
    CONDITIONAL_DIR /
    "conditional_100bins_membership_summary.csv",
    index=False,
)

ch.save_hard_sets_by_id(
    conditional_hard_sets_100,
    meta,
    CONDITIONAL_DIR /
    "conditional_100bins_hard_sets.csv",
    id_col=CONFIG.universe.id_col,
)

conditional_score_payload = {}

for invariant, result_i in (
    conditional_results_100.items()
):
    key = ch.safe_name(invariant)

    conditional_score_payload[
        f"{key}_conditional_score"
    ] = result_i["sse"]

    conditional_score_payload[
        f"{key}_norm_bin"
    ] = result_i["norm_bin"]

np.savez_compressed(
    CONDITIONAL_DIR /
    "conditional_100bins_scores.npz",
    **conditional_score_payload,
)

Sensibilidad a 50, 100 y 200 estratos

In [ ]:
from itertools import combinations

BIN_COUNTS = (50, 100, 200)

conditional_bin_results = {
    100: {
        "adjusted_results": conditional_results_100,
        "diagnostics": conditional_diagnostics_100,
        "hard_sets": conditional_hard_sets_100,
        "membership_sets": conditional_membership_sets,
        "summary": conditional_membership_summary_100,
    }
}


def summarize_conditional_families(
    hard_sets,
    meta,
):
    membership_sets = {}
    rows = []

    for family, specification in (
        CONDITIONAL_FAMILIES.items()
    ):
        names = specification["names"]
        m = len(names)

        for k in specification["thresholds"]:
            selected_set = ch.at_least_k_consensus(
                hard_sets=hard_sets,
                names=names,
                k=k,
                n_objects=len(meta),
            )

            membership_sets[
                (family, k)
            ] = selected_set

            row = ch.summarize_selected_set(
                name=f"{family}: >= {k}/{m}",
                selected=selected_set,
                meta=meta,
                s_col=CONFIG.s_col,
            )

            selected_df = meta.iloc[
                sorted(selected_set)
            ]

            nonalt = selected_df[
                selected_df[
                    "is_alternating"
                ].eq(0)
            ]

            delta = (
                nonalt[CONFIG.s_col]
                - nonalt["signature"]
            )

            row.update({
                "family": family,
                "k": k,
                "m": m,
                "mean_s_minus_sigma": (
                    float(delta.mean())
                    if len(delta)
                    else np.nan
                ),
                "delta_ge_4_prop": (
                    float(delta.ge(4).mean())
                    if len(delta)
                    else np.nan
                ),
            })

            rows.append(row)

    return membership_sets, pd.DataFrame(rows)

In [ ]:
for n_bins in (50, 200):

    print("\nNorm bins:", n_bins)

    adjusted_results, diagnostics = (
        ch.norm_adjusted_scores(
            fixed_results=primary["fixed_results"],
            norm_meta=meta_norm,
            norm_col_map=norm_col_map,
            method="conditional_percentile",
            n_bins=n_bins,
            seed=CONFIG.random_seed,
        )
    )

    hard_sets = (
        ch.build_hard_sets_from_fixed_results(
            fixed_results=adjusted_results,
            score_name="sse",
            tail_mass=CONFIG.pca.tail_mass,
            stable_ids=meta[
                CONFIG.universe.id_col
            ],
        )
    )

    membership_sets, summary = (
        summarize_conditional_families(
            hard_sets,
            meta,
        )
    )

    summary["norm_bins"] = n_bins
    diagnostics["norm_bins"] = n_bins

    conditional_bin_results[n_bins] = {
        "adjusted_results": adjusted_results,
        "diagnostics": diagnostics,
        "hard_sets": hard_sets,
        "membership_sets": membership_sets,
        "summary": summary,
    }

In [ ]:
conditional_bin_results[100][
    "summary"
] = (
    conditional_bin_results[100]["summary"]
    .assign(norm_bins=100)
)

conditional_bin_results[100][
    "diagnostics"
] = (
    conditional_bin_results[100]["diagnostics"]
    .assign(norm_bins=100)
)

In [ ]:
conditional_bin_summary = pd.concat(
    [
        conditional_bin_results[n_bins][
            "summary"
        ]
        for n_bins in BIN_COUNTS
    ],
    ignore_index=True,
)

conditional_bin_diagnostics = pd.concat(
    [
        conditional_bin_results[n_bins][
            "diagnostics"
        ]
        for n_bins in BIN_COUNTS
    ],
    ignore_index=True,
)

display(
    conditional_bin_summary[
        [
            "norm_bins",
            "family",
            "k",
            "m",
            "n",
            "median_sigma",
            "median_s",
            "sigma_ge_10_prop",
            "s_ge_10_prop",
            "s_ge_12_prop",
            "alternating_prop",
            "nonalt_s_gt_sigma_prop",
            "mean_s_minus_sigma",
        ]
    ].sort_values(
        ["family", "k", "norm_bins"]
    )
)

In [ ]:
conditional_bin_overlap_rows = []

for family, specification in (
    CONDITIONAL_FAMILIES.items()
):
    for k in specification["thresholds"]:

        for bins_a, bins_b in combinations(
            BIN_COUNTS,
            2,
        ):
            set_a = conditional_bin_results[
                bins_a
            ]["membership_sets"][(family, k)]

            set_b = conditional_bin_results[
                bins_b
            ]["membership_sets"][(family, k)]

            row = ch.compare_consensus_sets(
                set_a,
                set_b,
                name_a=f"{family} {k}, {bins_a} bins",
                name_b=f"{family} {k}, {bins_b} bins",
            )

            row.update({
                "family": family,
                "k": k,
                "m": len(
                    specification["names"]
                ),
                "bins_a": bins_a,
                "bins_b": bins_b,
            })

            conditional_bin_overlap_rows.append(
                row
            )

conditional_bin_overlap = pd.DataFrame(
    conditional_bin_overlap_rows
)

display(
    conditional_bin_overlap[
        [
            "family",
            "k",
            "bins_a",
            "bins_b",
            "size_a",
            "size_b",
            "overlap",
            "jaccard",
        ]
    ].sort_values(
        ["family", "k", "bins_a", "bins_b"]
    )
)

In [ ]:
conditional_bin_summary.to_csv(
    CONDITIONAL_DIR /
    "conditional_bin_sensitivity_summary.csv",
    index=False,
)

conditional_bin_diagnostics.to_csv(
    CONDITIONAL_DIR /
    "conditional_bin_sensitivity_diagnostics.csv",
    index=False,
)

conditional_bin_overlap.to_csv(
    CONDITIONAL_DIR /
    "conditional_bin_sensitivity_overlap.csv",
    index=False,
)

for n_bins in (50, 200):
    ch.save_hard_sets_by_id(
        conditional_bin_results[
            n_bins
        ]["hard_sets"],
        meta,
        CONDITIONAL_DIR /
        f"conditional_{n_bins}bins_hard_sets.csv",
        id_col=CONFIG.universe.id_col,
    )

## Null models condicionados

In [ ]:
NULL_DIR = ch.stage_directory(
    OUTPUT_DIR,
    9,
    "conditional_nulls",
)

norm_bins_100 = {
    invariant: np.asarray(
        result_i["norm_bin"],
        dtype=np.int32,
    )
    for invariant, result_i
    in conditional_results_100.items()
}

observed_conditional_masks = (
    ch.hard_sets_to_masks(
        conditional_hard_sets_100,
        n=len(meta),
    )
)

joint_strata_exact = (
    ch.build_joint_strata_codes(
        norm_bins_by_invariant=
            norm_bins_100,
        meta=meta,
        extra_cols=(
            "number_of_crossings",
            "is_alternating",
            "signature",
        ),
    )
)

In [ ]:
def audit_stratified_null(
    hard_masks,
    strata_codes,
):
    rows = []

    for invariant in hard_masks:

        audit_df = pd.DataFrame({
            "stratum":
                strata_codes[invariant],
            "hard":
                hard_masks[invariant].astype(int),
        })

        grouped = (
            audit_df
            .groupby("stratum")
            .agg(
                n=("hard", "size"),
                hard_n=("hard", "sum"),
            )
            .reset_index()
        )

        grouped["nonhard_n"] = (
            grouped["n"]
            - grouped["hard_n"]
        )

        grouped["fully_fixed"] = (
            grouped["hard_n"]
            == grouped["n"]
        )

        grouped["movable"] = (
            grouped["hard_n"].gt(0)
            & grouped["nonhard_n"].gt(0)
        )

        hard_total = int(
            grouped["hard_n"].sum()
        )

        hard_fixed = int(
            grouped.loc[
                grouped["fully_fixed"],
                "hard_n",
            ].sum()
        )

        hard_movable = int(
            grouped.loc[
                grouped["movable"],
                "hard_n",
            ].sum()
        )

        rows.append({
            "invariant": invariant,
            "n_strata": len(grouped),
            "singleton_strata": int(
                grouped["n"].eq(1).sum()
            ),
            "strata_containing_hard": int(
                grouped["hard_n"].gt(0).sum()
            ),
            "movable_hard_strata": int(
                grouped["movable"].sum()
            ),
            "hard_total": hard_total,
            "hard_in_fixed_strata":
                hard_fixed,
            "hard_in_movable_strata":
                hard_movable,
            "fraction_hard_fixed": (
                hard_fixed / hard_total
            ),
            "fraction_hard_movable": (
                hard_movable / hard_total
            ),
            "movable_candidate_pool": int(
                grouped.loc[
                    grouped["movable"],
                    "n",
                ].sum()
            ),
        })

    return pd.DataFrame(rows)


joint_strata_audit = audit_stratified_null(
    observed_conditional_masks,
    joint_strata_exact,
)

display(joint_strata_audit)

joint_strata_audit.to_csv(
    NULL_DIR /
    "exact_signature_strata_audit.csv",
    index=False,
)

In [ ]:
null_norm_only = (
    ch.run_stratified_membership_null(
        observed_hard_masks=
            observed_conditional_masks,
        strata_codes_by_invariant=
            norm_bins_100,
        families=
            CONDITIONAL_FAMILIES,
        meta=meta,
        n_reps=CONFIG.conditional_null_reps,
        seed=20260730,
        s_col=CONFIG.s_col,
    )
)

null_norm_only_summary = (
    ch.summarize_null_against_observed(
        observed=
            conditional_membership_summary_100,
        null=
            null_norm_only,
    )
)

null_norm_only_summary.insert(
    0,
    "null_model",
    "norm_bin",
)

In [ ]:
null_joint_exact = (
    ch.run_stratified_membership_null(
        observed_hard_masks=
            observed_conditional_masks,
        strata_codes_by_invariant=
            joint_strata_exact,
        families=
            CONDITIONAL_FAMILIES,
        meta=meta,
        n_reps=CONFIG.conditional_null_reps,
        seed=20260731,
        s_col=CONFIG.s_col,
    )
)

null_joint_exact_summary = (
    ch.summarize_null_against_observed(
        observed=
            conditional_membership_summary_100,
        null=
            null_joint_exact,
    )
)

null_joint_exact_summary.insert(
    0,
    "null_model",
    "norm_cross_alt_exact_signature",
)

In [ ]:
def add_null_fdr(table):
    out = table.copy()

    valid = out["empirical_p"].notna()
    out["q_value"] = np.nan

    _, qvals, _, _ = multipletests(
        out.loc[valid, "empirical_p"],
        method="fdr_bh",
    )

    out.loc[valid, "q_value"] = qvals

    out["fdr_significant"] = (
        out["q_value"] < 0.05
    )

    return out


null_norm_only_summary = add_null_fdr(
    null_norm_only_summary
)

null_joint_exact_summary = add_null_fdr(
    null_joint_exact_summary
)

conditional_null_summary = pd.concat(
    [
        null_norm_only_summary,
        null_joint_exact_summary,
    ],
    ignore_index=True,
)

focus_metrics = [
    "n",
    "s_ge_10_prop",
    "s_ge_12_prop",
    "nonalt_s_gt_sigma_prop",
    "mean_s_minus_sigma",
    "delta_ge_4_prop",
]

display(
    conditional_null_summary[
        conditional_null_summary[
            "metric"
        ].isin(focus_metrics)
    ][
        [
            "null_model",
            "family",
            "k",
            "m",
            "metric",
            "observed",
            "null_mean",
            "null_q95",
            "null_q99",
            "n_ge_observed",
            "empirical_p",
            "q_value",
            "fdr_significant",
        ]
    ].sort_values(
        [
            "null_model",
            "family",
            "k",
            "metric",
        ]
    )
)

In [ ]:
null_norm_only.to_csv(
    NULL_DIR / "norm_bin_null_raw.csv",
    index=False,
)

null_joint_exact.to_csv(
    NULL_DIR /
    "norm_cross_alt_exact_signature_null_raw.csv",
    index=False,
)

conditional_null_summary.to_csv(
    NULL_DIR /
    "conditional_null_summary_fdr.csv",
    index=False,
)

In [ ]:
# ------------------------------------------------------------
# Add effective number of valid null replicates
# ------------------------------------------------------------

null_raw_lookup = {
    "norm_bin": null_norm_only,
    "norm_cross_alt_exact_signature":
        null_joint_exact,
}

valid_count_rows = []

for null_model, raw_table in (
    null_raw_lookup.items()
):

    for (
        family,
        k,
    ), group in raw_table.groupby(
        ["family", "k"],
        sort=False,
    ):

        for metric in focus_metrics:

            if metric not in group.columns:
                continue

            n_valid = int(
                group[metric].notna().sum()
            )

            valid_count_rows.append({
                "null_model": null_model,
                "family": family,
                "k": int(k),
                "metric": metric,
                "n_valid_null": n_valid,
                "n_missing_null": (
                    len(group) - n_valid
                ),
            })

valid_counts = pd.DataFrame(
    valid_count_rows
)

conditional_null_summary = (
    conditional_null_summary
    .drop(
        columns=[
            "n_valid_null",
            "n_missing_null",
        ],
        errors="ignore",
    )
    .merge(
        valid_counts,
        on=[
            "null_model",
            "family",
            "k",
            "metric",
        ],
        how="left",
    )
)

In [ ]:
primary_null_rows = (
    (
        conditional_null_summary["family"]
        .eq("All 5")
        &
        conditional_null_summary["k"]
        .isin([3, 4])
    )
    |
    (
        conditional_null_summary["family"]
        .eq("No Khovanov")
        &
        conditional_null_summary["k"]
        .eq(3)
    )
    |
    (
        conditional_null_summary["family"]
        .eq("Polynomial only")
        &
        conditional_null_summary["k"]
        .eq(3)
    )
)

primary_conditional_null_results = (
    conditional_null_summary[
        primary_null_rows
        &
        conditional_null_summary[
            "metric"
        ].isin(focus_metrics)
    ][
        [
            "null_model",
            "family",
            "k",
            "m",
            "metric",
            "observed",
            "null_mean",
            "null_q95",
            "null_q99",
            "n_ge_observed",
            "n_valid_null",
            "n_missing_null",
            "empirical_p",
            "q_value",
            "fdr_significant",
        ]
    ]
    .sort_values(
        [
            "null_model",
            "family",
            "k",
            "metric",
        ]
    )
)

print(
    primary_conditional_null_results
    .to_string(index=False)
)

In [ ]:
conditional_null_summary.to_csv(
    NULL_DIR /
    "conditional_null_summary_fdr_with_valid_counts.csv",
    index=False,
)

primary_conditional_null_results.to_csv(
    NULL_DIR /
    "primary_conditional_null_results.csv",
    index=False,
)

Extensión preespecificada a 5,000 permutaciones

In [ ]:
# ------------------------------------------------------------
# Focused 5,000-replicate exact-structure gap null
# ------------------------------------------------------------

GAP_FAMILIES = {
    "All 5": {
        "names": INVARIANTS,
        "thresholds": (3,),
    },
    "No Khovanov": {
        "names": NO_KHOVANOV,
        "thresholds": (3,),
    },
    "Polynomial only": {
        "names": POLYNOMIAL_ONLY,
        "thresholds": (3,),
    },
}

GAP_METRICS = (
    "nonalt_s_gt_sigma_prop",
    "mean_s_minus_sigma",
    "delta_ge_4_prop",
)

observed_gap_primary = (
    conditional_membership_summary_100[
        (
            conditional_membership_summary_100[
                "family"
            ].eq("All 5")
            &
            conditional_membership_summary_100[
                "k"
            ].eq(3)
        )
        |
        (
            conditional_membership_summary_100[
                "family"
            ].eq("No Khovanov")
            &
            conditional_membership_summary_100[
                "k"
            ].eq(3)
        )
        |
        (
            conditional_membership_summary_100[
                "family"
            ].eq("Polynomial only")
            &
            conditional_membership_summary_100[
                "k"
            ].eq(3)
        )
    ]
    .copy()
)

null_gap_exact_5000 = (
    ch.run_stratified_membership_null(
        observed_hard_masks=
            observed_conditional_masks,
        strata_codes_by_invariant=
            joint_strata_exact,
        families=GAP_FAMILIES,
        meta=meta,
        n_reps=CONFIG.gap_null_reps,
        seed=20260731,
        s_col=CONFIG.s_col,
    )
)

gap_exact_5000_summary = (
    ch.summarize_null_against_observed(
        observed=observed_gap_primary,
        null=null_gap_exact_5000,
        metrics=GAP_METRICS,
    )
)

gap_exact_5000_summary.insert(
    0,
    "null_model",
    "norm_cross_alt_exact_signature_5000",
)

In [ ]:
gap_valid_rows = []

for (
    family,
    k,
), group in null_gap_exact_5000.groupby(
    ["family", "k"],
    sort=False,
):
    for metric in GAP_METRICS:

        n_valid = int(
            group[metric].notna().sum()
        )

        gap_valid_rows.append({
            "family": family,
            "k": int(k),
            "metric": metric,
            "n_valid_null": n_valid,
            "n_missing_null": (
                len(group) - n_valid
            ),
        })

gap_valid_counts = pd.DataFrame(
    gap_valid_rows
)

gap_exact_5000_summary = (
    gap_exact_5000_summary
    .merge(
        gap_valid_counts,
        on=["family", "k", "metric"],
        how="left",
    )
)

In [ ]:
valid = gap_exact_5000_summary[
    "empirical_p"
].notna()

gap_exact_5000_summary["q_value"] = np.nan

_, gap_qvalues, _, _ = multipletests(
    gap_exact_5000_summary.loc[
        valid,
        "empirical_p",
    ],
    method="fdr_bh",
)

gap_exact_5000_summary.loc[
    valid,
    "q_value",
] = gap_qvalues

gap_exact_5000_summary[
    "fdr_significant"
] = (
    gap_exact_5000_summary[
        "q_value"
    ] < 0.05
)

display(
    gap_exact_5000_summary[
        [
            "family",
            "k",
            "m",
            "metric",
            "observed",
            "null_mean",
            "null_q95",
            "null_q99",
            "n_ge_observed",
            "n_valid_null",
            "n_missing_null",
            "empirical_p",
            "q_value",
            "fdr_significant",
        ]
    ].sort_values(
        ["family", "metric"]
    )
)

In [ ]:
null_gap_exact_5000.to_csv(
    NULL_DIR /
    "exact_structure_gap_null_5000_raw.csv",
    index=False,
)

gap_exact_5000_summary.to_csv(
    NULL_DIR /
    "exact_structure_gap_null_5000_summary_fdr.csv",
    index=False,
)

## residualización continua

In [ ]:
# ------------------------------------------------------------
# Cross-fitted continuous norm residualization
# ------------------------------------------------------------

RESIDUAL_DIR = ch.stage_directory(
    OUTPUT_DIR,
    10,
    "crossfitted_norm_residual",
)

residual_results, residual_diagnostics = (
    ch.norm_adjusted_scores(
        fixed_results=primary["fixed_results"],
        norm_meta=meta_norm,
        norm_col_map=norm_col_map,
        method="crossfitted_residual",
        seed=CONFIG.random_seed,
    )
)

display(residual_diagnostics)

In [ ]:
residual_hard_sets = (
    ch.build_hard_sets_from_fixed_results(
        fixed_results=residual_results,
        score_name="sse",
        tail_mass=CONFIG.pca.tail_mass,
        stable_ids=meta[
            CONFIG.universe.id_col
        ],
    )
)

(
    residual_membership_sets,
    residual_membership_summary,
) = summarize_conditional_families(
    residual_hard_sets,
    meta,
)

residual_membership_summary[
    "adjustment_method"
] = "crossfitted_spline_residual"

display(
    residual_membership_summary[
        [
            "family",
            "k",
            "m",
            "n",
            "median_sigma",
            "median_s",
            "sigma_ge_10_prop",
            "s_ge_10_prop",
            "s_ge_12_prop",
            "alternating_prop",
            "nonalt_s_gt_sigma_prop",
            "mean_s_minus_sigma",
            "delta_ge_4_prop",
        ]
    ].sort_values(
        ["family", "k"]
    )
)

In [ ]:
residual_vs_binned_rows = []

for family, specification in (
    CONDITIONAL_FAMILIES.items()
):
    for k in specification["thresholds"]:

        binned_set = (
            conditional_membership_sets[
                (family, k)
            ]
        )

        residual_set = (
            residual_membership_sets[
                (family, k)
            ]
        )

        row = ch.compare_consensus_sets(
            binned_set,
            residual_set,
            name_a=(
                f"Binned 100: {family} "
                f">={k}/{len(specification['names'])}"
            ),
            name_b=(
                f"Residual: {family} "
                f">={k}/{len(specification['names'])}"
            ),
        )

        row.update({
            "family": family,
            "k": k,
            "m": len(
                specification["names"]
            ),
        })

        residual_vs_binned_rows.append(row)

residual_vs_binned_overlap = pd.DataFrame(
    residual_vs_binned_rows
)

display(
    residual_vs_binned_overlap[
        [
            "family",
            "k",
            "m",
            "size_a",
            "size_b",
            "overlap",
            "jaccard",
            "fraction_a_in_b",
            "fraction_b_in_a",
        ]
    ].sort_values(
        ["family", "k"]
    )
)

In [ ]:
residual_diagnostics.to_csv(
    RESIDUAL_DIR /
    "residual_diagnostics.csv",
    index=False,
)

residual_membership_summary.to_csv(
    RESIDUAL_DIR /
    "residual_membership_summary.csv",
    index=False,
)

residual_vs_binned_overlap.to_csv(
    RESIDUAL_DIR /
    "residual_vs_binned_overlap.csv",
    index=False,
)

ch.save_hard_sets_by_id(
    residual_hard_sets,
    meta,
    RESIDUAL_DIR /
    "residual_hard_sets.csv",
    id_col=CONFIG.universe.id_col,
)

np.savez_compressed(
    RESIDUAL_DIR /
    "residual_scores.npz",
    **{
        f"{ch.safe_name(invariant)}_residual":
            result_i["residual"]
        for invariant, result_i
        in residual_results.items()
    },
)

## score continuo C_k
	​


In [ ]:
# ------------------------------------------------------------
# Continuous C_k scores
# ------------------------------------------------------------

CK_DIR = ch.stage_directory(
    OUTPUT_DIR,
    11,
    "continuous_ck",
)

CK_FAMILIES = {
    "All 5": {
        "names": INVARIANTS,
        "k": 3,
    },
    "No Khovanov": {
        "names": NO_KHOVANOV,
        "k": 3,
    },
    "Polynomial only": {
        "names": POLYNOMIAL_ONLY,
        "k": 2,
    },
}

conditional_scores_by_view = {
    invariant: np.asarray(
        conditional_results_100[
            invariant
        ]["sse"],
        dtype=float,
    )
    for invariant in INVARIANTS
}

ck_scores = ch.build_ck_scores(
    scores_by_view=
        conditional_scores_by_view,
    families=CK_FAMILIES,
)

In [ ]:
from scipy.stats import spearmanr

ck_diagnostic_rows = []

for family, score in ck_scores.items():

    ck_diagnostic_rows.append({
        "family": family,
        "k": CK_FAMILIES[family]["k"],
        "spearman_ck_mean_log_norm": (
            spearmanr(
                score,
                meta_norm[
                    "mean_log_sq_norm"
                ],
            ).statistic
        ),
        "spearman_ck_max_log_norm": (
            spearmanr(
                score,
                meta_norm[
                    "max_log_sq_norm"
                ],
            ).statistic
        ),
    })

ck_diagnostics = pd.DataFrame(
    ck_diagnostic_rows
)

display(ck_diagnostics)

In [ ]:
ck_fraction_summaries = []

for fraction in (
    0.005,
    0.01,
    0.02,
    0.05,
):

    summary_i = (
        ch.continuous_association_summary(
            ck_scores=ck_scores,
            meta=meta,
            top_fraction=fraction,
            s_col=CONFIG.s_col,
            nonalternating_only=True,
            matching_cols=(
                "number_of_crossings",
                "signature",
            ),
        )
    )

    ck_fraction_summaries.append(
        summary_i
    )

ck_top_fraction_summary = pd.concat(
    ck_fraction_summaries,
    ignore_index=True,
)

display(
    ck_top_fraction_summary[
        [
            "family",
            "top_fraction",
            "n_selected",
            "spearman_ck_delta",
            "selected_mean_delta",
            "selected_median_delta",
            "matched_expected_mean_delta",
            "matched_mean_delta_effect",
            "delta_ge_2_prop",
            "delta_ge_4_prop",
        ]
    ].sort_values(
        ["family", "top_fraction"]
    )
)

In [ ]:
# ------------------------------------------------------------
# C_k decile dose-response
# ------------------------------------------------------------

meta_delta = ch.add_delta(
    meta,
    s_col=CONFIG.s_col,
    signature_col="signature",
    output_col="delta_s_minus_sigma",
)

ck_decile_rows = []

for family, score in ck_scores.items():

    eligible = (
        meta_delta[
            "is_alternating"
        ].eq(0)
        &
        np.isfinite(score)
        &
        meta_delta[
            "delta_s_minus_sigma"
        ].notna()
    )

    frame = pd.DataFrame({
        "ck_score": score[eligible],
        "delta": meta_delta.loc[
            eligible,
            "delta_s_minus_sigma",
        ].to_numpy(),
        "signature": meta_delta.loc[
            eligible,
            "signature",
        ].to_numpy(),
        "s": meta_delta.loc[
            eligible,
            CONFIG.s_col,
        ].to_numpy(),
    })

    frame["ck_decile"] = pd.qcut(
        frame["ck_score"],
        q=10,
        labels=False,
        duplicates="drop",
    )

    grouped = (
        frame
        .groupby("ck_decile")
        .agg(
            n=("delta", "size"),
            ck_mean=("ck_score", "mean"),
            ck_min=("ck_score", "min"),
            ck_max=("ck_score", "max"),
            mean_delta=("delta", "mean"),
            median_delta=("delta", "median"),
            delta_gt_0_prop=(
                "delta",
                lambda x: x.gt(0).mean(),
            ),
            delta_ge_2_prop=(
                "delta",
                lambda x: x.ge(2).mean(),
            ),
            delta_ge_4_prop=(
                "delta",
                lambda x: x.ge(4).mean(),
            ),
            mean_signature=(
                "signature", "mean"
            ),
            mean_s=("s", "mean"),
        )
        .reset_index()
    )

    grouped.insert(
        0,
        "family",
        family,
    )

    ck_decile_rows.append(grouped)

ck_decile_summary = pd.concat(
    ck_decile_rows,
    ignore_index=True,
)

display(
    ck_decile_summary[
        [
            "family",
            "ck_decile",
            "n",
            "ck_mean",
            "mean_delta",
            "delta_gt_0_prop",
            "delta_ge_2_prop",
            "delta_ge_4_prop",
        ]
    ]
)

In [ ]:
ck_diagnostics.to_csv(
    CK_DIR / "ck_norm_diagnostics.csv",
    index=False,
)

ck_top_fraction_summary.to_csv(
    CK_DIR / "ck_top_fraction_summary.csv",
    index=False,
)

ck_decile_summary.to_csv(
    CK_DIR / "ck_decile_dose_response.csv",
    index=False,
)

np.savez_compressed(
    CK_DIR / "ck_scores.npz",
    **{
        ch.safe_name(family): score
        for family, score
        in ck_scores.items()
    },
)

Ajuste continuo adicional por normas familiares

In [ ]:
# ------------------------------------------------------------
# Norm-aware continuous C_k sensitivity
# ------------------------------------------------------------

CK_AGGREGATE_NORM_BINS = 10

meta_ck_norm = meta.copy()

ck_norm_summary_rows = []
ck_partial_rows = []

for family, specification in (
    CK_FAMILIES.items()
):
    names = specification["names"]
    safe_family = ch.safe_name(family)

    family_log_norm_cols = [
        f"{ch.safe_name(name)}_log_sq_norm"
        for name in names
    ]

    family_mean_norm_col = (
        f"{safe_family}_family_mean_log_norm"
    )

    family_max_norm_col = (
        f"{safe_family}_family_max_log_norm"
    )

    mean_bin_col = (
        f"{safe_family}_mean_norm_bin"
    )

    max_bin_col = (
        f"{safe_family}_max_norm_bin"
    )

    meta_ck_norm[family_mean_norm_col] = (
        meta_norm[
            family_log_norm_cols
        ].mean(axis=1)
    )

    meta_ck_norm[family_max_norm_col] = (
        meta_norm[
            family_log_norm_cols
        ].max(axis=1)
    )

    meta_ck_norm[mean_bin_col] = pd.qcut(
        meta_ck_norm[
            family_mean_norm_col
        ],
        q=CK_AGGREGATE_NORM_BINS,
        labels=False,
        duplicates="drop",
    )

    meta_ck_norm[max_bin_col] = pd.qcut(
        meta_ck_norm[
            family_max_norm_col
        ],
        q=CK_AGGREGATE_NORM_BINS,
        labels=False,
        duplicates="drop",
    )

    matching_cols_ck = (
        "number_of_crossings",
        "signature",
        mean_bin_col,
        max_bin_col,
    )

    for fraction in (
        0.005,
        0.01,
        0.02,
        0.05,
    ):
        summary_i = (
            ch.continuous_association_summary(
                ck_scores={
                    family:
                        ck_scores[family]
                },
                meta=meta_ck_norm,
                top_fraction=fraction,
                s_col=CONFIG.s_col,
                nonalternating_only=True,
                matching_cols=
                    matching_cols_ck,
            )
        )

        summary_i[
            "aggregate_norm_bins"
        ] = CK_AGGREGATE_NORM_BINS

        ck_norm_summary_rows.append(
            summary_i
        )

    # Partial Spearman after centering delta
    # within the same exact structural/norm strata.
    work = ch.add_delta(
        meta_ck_norm,
        s_col=CONFIG.s_col,
        output_col="delta_s_minus_sigma",
    )

    eligible = (
        work["is_alternating"].eq(0)
        &
        np.isfinite(ck_scores[family])
    )

    partial_frame = work.loc[
        eligible,
        list(matching_cols_ck)
        + ["delta_s_minus_sigma"],
    ].copy()

    partial_frame["ck_score"] = (
        ck_scores[family][eligible]
    )

    partial_frame[
        "expected_delta_in_stratum"
    ] = (
        partial_frame
        .groupby(
            list(matching_cols_ck),
            dropna=False,
        )["delta_s_minus_sigma"]
        .transform("mean")
    )

    partial_frame[
        "delta_centered"
    ] = (
        partial_frame[
            "delta_s_minus_sigma"
        ]
        -
        partial_frame[
            "expected_delta_in_stratum"
        ]
    )

    ck_partial_rows.append({
        "family": family,
        "spearman_ck_centered_delta": (
            spearmanr(
                partial_frame["ck_score"],
                partial_frame[
                    "delta_centered"
                ],
            ).statistic
        ),
        "n_nonalternating": len(
            partial_frame
        ),
    })

In [ ]:
ck_norm_adjusted_summary = pd.concat(
    ck_norm_summary_rows,
    ignore_index=True,
)

ck_partial_association = pd.DataFrame(
    ck_partial_rows
)

ck_comparison = (
    ck_top_fraction_summary[
        [
            "family",
            "top_fraction",
            "selected_mean_delta",
            "matched_expected_mean_delta",
            "matched_mean_delta_effect",
        ]
    ]
    .rename(columns={
        "matched_expected_mean_delta":
            "structural_expected_delta",
        "matched_mean_delta_effect":
            "structural_effect",
    })
    .merge(
        ck_norm_adjusted_summary[
            [
                "family",
                "top_fraction",
                "matched_expected_mean_delta",
                "matched_mean_delta_effect",
            ]
        ].rename(columns={
            "matched_expected_mean_delta":
                "structural_norm_expected_delta",
            "matched_mean_delta_effect":
                "structural_norm_effect",
        }),
        on=["family", "top_fraction"],
        how="left",
    )
    .merge(
        ck_partial_association,
        on="family",
        how="left",
    )
)

display(
    ck_comparison.sort_values(
        ["family", "top_fraction"]
    )
)

In [ ]:
ck_norm_adjusted_summary.to_csv(
    CK_DIR /
    "ck_structural_norm_adjusted_summary.csv",
    index=False,
)

ck_partial_association.to_csv(
    CK_DIR /
    "ck_partial_association.csv",
    index=False,
)

ck_comparison.to_csv(
    CK_DIR /
    "ck_adjustment_comparison.csv",
    index=False,
)

In [ ]:
from scipy.stats import pearsonr

ck_fixed_effect_rows = []

for family in CK_FAMILIES:

    safe_family = ch.safe_name(family)

    matching_cols = [
        "number_of_crossings",
        "signature",
        f"{safe_family}_mean_norm_bin",
        f"{safe_family}_max_norm_bin",
    ]

    work = ch.add_delta(
        meta_ck_norm,
        s_col=CONFIG.s_col,
        output_col="delta_s_minus_sigma",
    )

    eligible = (
        work["is_alternating"].eq(0)
        &
        np.isfinite(ck_scores[family])
    )

    frame = work.loc[
        eligible,
        matching_cols
        + ["delta_s_minus_sigma"],
    ].copy()

    frame["ck_score"] = (
        ck_scores[family][eligible]
    )

    # Spearman = Pearson correlation of ranks.
    frame["ck_rank"] = (
        frame["ck_score"]
        .rank(method="average")
    )

    frame["delta_rank"] = (
        frame["delta_s_minus_sigma"]
        .rank(method="average")
    )

    group = frame.groupby(
        matching_cols,
        dropna=False,
    )

    frame["ck_rank_centered"] = (
        frame["ck_rank"]
        - group["ck_rank"].transform("mean")
    )

    frame["delta_rank_centered"] = (
        frame["delta_rank"]
        - group["delta_rank"].transform("mean")
    )

    usable = (
        frame["ck_rank_centered"].notna()
        &
        frame["delta_rank_centered"].notna()
    )

    rho_fixed, p_fixed = pearsonr(
        frame.loc[
            usable,
            "ck_rank_centered",
        ],
        frame.loc[
            usable,
            "delta_rank_centered",
        ],
    )

    ck_fixed_effect_rows.append({
        "family": family,
        "fixed_effect_partial_spearman":
            rho_fixed,
        "p_value": p_fixed,
        "n": int(usable.sum()),
    })

ck_fixed_effect_association = pd.DataFrame(
    ck_fixed_effect_rows
)

display(ck_fixed_effect_association)

ck_fixed_effect_association.to_csv(
    CK_DIR /
    "ck_fixed_effect_partial_spearman.csv",
    index=False,
)

# Atlas del hard regime

In [ ]:
# ------------------------------------------------------------
# Master hard-regime atlas
# ------------------------------------------------------------

ATLAS_DIR = ch.stage_directory(
    OUTPUT_DIR,
    12,
    "hard_regime_atlas",
)

atlas = ch.add_delta(
    meta,
    s_col=CONFIG.s_col,
    output_col="delta_s_minus_sigma",
)


def hard_membership_count(
    hard_sets,
    names=None,
):
    names = (
        tuple(hard_sets.keys())
        if names is None
        else tuple(names)
    )

    counts = np.zeros(
        len(meta),
        dtype=np.uint8,
    )

    for name in names:
        indices = np.fromiter(
            hard_sets[name],
            dtype=int,
        )
        counts[indices] += 1

    return counts


atlas["raw_pca_membership"] = (
    hard_membership_count(
        primary["hard_sets"]
    )
)

atlas["conditional_membership"] = (
    hard_membership_count(
        conditional_hard_sets_100
    )
)

atlas["conditional_no_kh_membership"] = (
    hard_membership_count(
        conditional_hard_sets_100,
        NO_KHOVANOV,
    )
)

atlas["conditional_poly_membership"] = (
    hard_membership_count(
        conditional_hard_sets_100,
        POLYNOMIAL_ONLY,
    )
)

atlas["residual_membership"] = (
    hard_membership_count(
        residual_hard_sets
    )
)

In [ ]:
atlas["in_frozen_test"] = (
    atlas.index.isin(test_idx)
)

atlas["pca_heldout_consensus"] = (
    atlas.index.isin(
        pca_holdout_consensus
    )
)

atlas["ae_seed_membership"] = 0

atlas.loc[
    ae_frequency.index,
    "ae_seed_membership",
] = ae_frequency.to_numpy()

atlas["ae_majority_consensus"] = (
    atlas["ae_seed_membership"] >= 3
)

atlas["ae_strict_core"] = (
    atlas["ae_seed_membership"] == 5
)

In [ ]:
for invariant in INVARIANTS:

    key = ch.safe_name(invariant)

    atlas[
        f"{key}_raw_pca_sse"
    ] = np.asarray(
        primary["fixed_results"][
            invariant
        ]["sse"]
    )

    atlas[
        f"{key}_conditional_percentile"
    ] = np.asarray(
        conditional_results_100[
            invariant
        ]["sse"]
    )

    atlas[
        f"{key}_raw_hard"
    ] = atlas.index.isin(
        primary["hard_sets"][
            invariant
        ]
    )

    atlas[
        f"{key}_conditional_hard"
    ] = atlas.index.isin(
        conditional_hard_sets_100[
            invariant
        ]
    )

    atlas[
        f"{key}_log_sq_norm"
    ] = meta_norm[
        f"{key}_log_sq_norm"
    ].to_numpy()

atlas["mean_log_sq_norm"] = (
    meta_norm["mean_log_sq_norm"]
    .to_numpy()
)

atlas["max_log_sq_norm"] = (
    meta_norm["max_log_sq_norm"]
    .to_numpy()
)

for family, score in ck_scores.items():
    atlas[
        f"ck_{ch.safe_name(family)}"
    ] = score

In [ ]:
raw_absolute_hard = (
    atlas["raw_pca_membership"] == 5
)

conditional_hard = (
    atlas["conditional_membership"] >= 3
)

atlas["hard_regime"] = np.select(
    [
        raw_absolute_hard
        & conditional_hard,

        raw_absolute_hard
        & ~conditional_hard,

        ~raw_absolute_hard
        & conditional_hard,
    ],
    [
        "shared_raw_and_conditional",
        "amplitude_hard_only",
        "conditional_hard_only",
    ],
    default="neither",
)

universal_core_test = set(
    atlas.index[
        atlas["pca_heldout_consensus"]
        &
        atlas["ae_majority_consensus"]
        &
        conditional_hard
    ]
)

print(
    "Raw absolute hard:",
    int(raw_absolute_hard.sum()),
)

print(
    "Conditional >=3/5:",
    int(conditional_hard.sum()),
)

print(
    "Shared raw + conditional:",
    int(
        (
            raw_absolute_hard
            & conditional_hard
        ).sum()
    ),
)

print(
    "PCA + AE + conditional test core:",
    len(universal_core_test),
)

In [ ]:
def atlas_group_summary(
    label,
    frame,
):
    nonalt = frame[
        frame["is_alternating"].eq(0)
    ]

    return {
        "hard_regime": label,
        "n": len(frame),
        "prop_crossing_15": (
            frame[
                "number_of_crossings"
            ].eq(15).mean()
        ),
        "prop_alternating": (
            frame["is_alternating"].mean()
        ),
        "median_signature": (
            frame["signature"].median()
        ),
        "median_s": (
            frame[CONFIG.s_col].median()
        ),
        "mean_nonalt_delta": (
            nonalt[
                "delta_s_minus_sigma"
            ].mean()
            if len(nonalt)
            else np.nan
        ),
        "nonalt_delta_positive_prop": (
            nonalt[
                "delta_s_minus_sigma"
            ].gt(0).mean()
            if len(nonalt)
            else np.nan
        ),
        "nonalt_delta_ge_4_prop": (
            nonalt[
                "delta_s_minus_sigma"
            ].ge(4).mean()
            if len(nonalt)
            else np.nan
        ),
        "mean_log_sq_norm": (
            frame[
                "mean_log_sq_norm"
            ].mean()
        ),
        "mean_raw_membership": (
            frame[
                "raw_pca_membership"
            ].mean()
        ),
        "mean_conditional_membership": (
            frame[
                "conditional_membership"
            ].mean()
        ),
    }


atlas_regime_summary = pd.DataFrame([
    atlas_group_summary(
        label,
        group,
    )
    for label, group in atlas.groupby(
        "hard_regime",
        sort=False,
    )
])

display(atlas_regime_summary)

In [ ]:
universal_core_df = (
    atlas.loc[
        sorted(universal_core_test)
    ][
        [
            "knot_id_base",
            "knot_id",
            "number_of_crossings",
            "is_alternating",
            "signature",
            CONFIG.s_col,
            "delta_s_minus_sigma",
            "raw_pca_membership",
            "conditional_membership",
            "conditional_no_kh_membership",
            "conditional_poly_membership",
            "ae_seed_membership",
            "mean_log_sq_norm",
            "max_log_sq_norm",
            "ck_All_5",
            "ck_No_Khovanov",
            "ck_Polynomial_only",
        ]
    ]
    .sort_values(
        [
            "conditional_membership",
            "ae_seed_membership",
            "delta_s_minus_sigma",
        ],
        ascending=False,
    )
)

display(universal_core_df)

In [ ]:
atlas_regime_summary.to_csv(
    ATLAS_DIR /
    "hard_regime_summary.csv",
    index=False,
)

universal_core_df.to_csv(
    ATLAS_DIR /
    "universal_core_knots.csv",
    index=False,
)

atlas.to_parquet(
    ATLAS_DIR /
    "hard_regime_atlas.parquet",
    index=False,
)

## ATLAS export

In [ ]:
# ============================================================
# STEP 13 — FREEZE HARD-REGIME CANDIDATES
# ============================================================

FINAL_CANDIDATES_DIR = ch.stage_directory(
    OUTPUT_DIR,
    13,
    "hard_regime_candidates",
)

# ------------------------------------------------------------
# 1. Detect useful columns safely
# ------------------------------------------------------------

identity_candidates = [
    "knot_id_base",
    "knot_id_clean",
    "knot_id",
]

property_candidates = [
    "number_of_crossings",
    "is_alternating",
    "signature",
    "s_invariant_qc",
    "s_minus_sigma_qc",
    "mean_log_sq_norm",
    "max_log_sq_norm",
    "raw_pca_membership",
    "conditional_membership",
    "conditional_no_khovanov_membership",
    "conditional_polynomial_membership",
    "ae_seed_membership",
    "is_pca_holdout_consensus",
    "is_ae_majority_consensus",
    "hard_regime",
]

ck_cols = [
    col for col in atlas.columns
    if col.lower().startswith("ck_")
]

candidate_cols = [
    col
    for col in (
        identity_candidates
        + property_candidates
        + ck_cols
    )
    if col in atlas.columns
]

print("Candidate columns:")
print(candidate_cols)

# ------------------------------------------------------------
# 2. Define final candidate groups
# ------------------------------------------------------------

raw_absolute_df = atlas.loc[
    atlas["raw_pca_membership"] == 5,
    candidate_cols,
].copy()

conditional_hard_df = atlas.loc[
    atlas["conditional_membership"] >= 3,
    candidate_cols,
].copy()

shared_hard_df = atlas.loc[
    atlas["hard_regime"] == "shared_raw_and_conditional",
    candidate_cols,
].copy()

amplitude_only_df = atlas.loc[
    atlas["hard_regime"] == "amplitude_hard_only",
    candidate_cols,
].copy()

conditional_only_df = atlas.loc[
    atlas["hard_regime"] == "conditional_hard_only",
    candidate_cols,
].copy()

universal_core_export = universal_core_df.loc[
    :,
    [
        col for col in candidate_cols
        if col in universal_core_df.columns
    ],
].copy()

# ------------------------------------------------------------
# 3. Sort by strongest available conditional score
# ------------------------------------------------------------

preferred_rank_cols = [
    col for col in ck_cols
    if "all" in col.lower()
]

if preferred_rank_cols:
    rank_col = preferred_rank_cols[0]

elif ck_cols:
    rank_col = ck_cols[0]

else:
    rank_col = "conditional_membership"

print("Ranking candidates by:", rank_col)

for frame in [
    conditional_hard_df,
    shared_hard_df,
    conditional_only_df,
    universal_core_export,
]:
    if rank_col in frame.columns:
        frame.sort_values(
            rank_col,
            ascending=False,
            inplace=True,
        )

# ------------------------------------------------------------
# 4. Save
# ------------------------------------------------------------

atlas.to_parquet(
    FINAL_CANDIDATES_DIR / "complete_hard_regime_atlas.parquet",
    index=False,
)

raw_absolute_df.to_csv(
    FINAL_CANDIDATES_DIR / "raw_absolute_hard_292.csv",
    index=False,
)

conditional_hard_df.to_csv(
    FINAL_CANDIDATES_DIR / "conditional_hard_413.csv",
    index=False,
)

shared_hard_df.to_csv(
    FINAL_CANDIDATES_DIR / "shared_raw_conditional_46.csv",
    index=False,
)

amplitude_only_df.to_csv(
    FINAL_CANDIDATES_DIR / "amplitude_hard_only_246.csv",
    index=False,
)

conditional_only_df.to_csv(
    FINAL_CANDIDATES_DIR / "conditional_hard_only_367.csv",
    index=False,
)

universal_core_export.to_csv(
    FINAL_CANDIDATES_DIR / "universal_pca_ae_conditional_core_6.csv",
    index=False,
)

# ------------------------------------------------------------
# 5. Final census verification
# ------------------------------------------------------------

candidate_census = pd.DataFrame(
    [
        {
            "candidate_set": "Raw absolute hard",
            "n": len(raw_absolute_df),
        },
        {
            "candidate_set": "Conditional hard >=3/5",
            "n": len(conditional_hard_df),
        },
        {
            "candidate_set": "Shared raw + conditional",
            "n": len(shared_hard_df),
        },
        {
            "candidate_set": "Amplitude hard only",
            "n": len(amplitude_only_df),
        },
        {
            "candidate_set": "Conditional hard only",
            "n": len(conditional_only_df),
        },
        {
            "candidate_set": "PCA + AE + conditional core",
            "n": len(universal_core_export),
        },
    ]
)

candidate_census.to_csv(
    FINAL_CANDIDATES_DIR / "candidate_census.csv",
    index=False,
)

display(candidate_census)

print("\nUniversal six-knot core:")
display(universal_core_export)

print("\nTop 20 shared raw + conditional:")
display(shared_hard_df.head(20))

print("\nTop 20 conditional-only:")
display(conditional_only_df.head(20))

print("\nSaved to:")
print(FINAL_CANDIDATES_DIR)

In [ ]:
# ============================================================
# STEP 13B — ADD EXPLICIT s - sigma GAP
# ============================================================

atlas["s_minus_sigma_qc"] = (
    atlas["s_invariant_qc"]
    - atlas["signature"]
)

# Rebuild the main candidate tables with the new column
candidate_cols_with_delta = [
    col for col in (
        [
            "knot_id_base",
            "knot_id_clean",
            "knot_id",
            "number_of_crossings",
            "is_alternating",
            "signature",
            "s_invariant_qc",
            "s_minus_sigma_qc",
            "mean_log_sq_norm",
            "max_log_sq_norm",
            "raw_pca_membership",
            "conditional_membership",
            "ae_seed_membership",
            "hard_regime",
        ]
        + ck_cols
    )
    if col in atlas.columns
]

raw_absolute_df = atlas.loc[
    atlas["raw_pca_membership"] == 5,
    candidate_cols_with_delta,
].copy()

conditional_hard_df = atlas.loc[
    atlas["conditional_membership"] >= 3,
    candidate_cols_with_delta,
].copy()

shared_hard_df = atlas.loc[
    atlas["hard_regime"] == "shared_raw_and_conditional",
    candidate_cols_with_delta,
].copy()

amplitude_only_df = atlas.loc[
    atlas["hard_regime"] == "amplitude_hard_only",
    candidate_cols_with_delta,
].copy()

conditional_only_df = atlas.loc[
    atlas["hard_regime"] == "conditional_hard_only",
    candidate_cols_with_delta,
].copy()

universal_indices = universal_core_df.index

universal_core_export = atlas.loc[
    universal_indices,
    candidate_cols_with_delta,
].copy()

# Sort
for frame in [
    raw_absolute_df,
    conditional_hard_df,
    shared_hard_df,
    amplitude_only_df,
    conditional_only_df,
    universal_core_export,
]:
    frame.sort_values(
        "ck_All_5",
        ascending=False,
        inplace=True,
    )

# Overwrite corrected exports
raw_absolute_df.to_csv(
    FINAL_CANDIDATES_DIR / "raw_absolute_hard_292.csv",
    index=False,
)

conditional_hard_df.to_csv(
    FINAL_CANDIDATES_DIR / "conditional_hard_413.csv",
    index=False,
)

shared_hard_df.to_csv(
    FINAL_CANDIDATES_DIR / "shared_raw_conditional_46.csv",
    index=False,
)

amplitude_only_df.to_csv(
    FINAL_CANDIDATES_DIR / "amplitude_hard_only_246.csv",
    index=False,
)

conditional_only_df.to_csv(
    FINAL_CANDIDATES_DIR / "conditional_hard_only_367.csv",
    index=False,
)

universal_core_export.to_csv(
    FINAL_CANDIDATES_DIR / "universal_pca_ae_conditional_core_6.csv",
    index=False,
)

atlas.to_parquet(
    FINAL_CANDIDATES_DIR / "complete_hard_regime_atlas.parquet",
    index=False,
)

display(universal_core_export)

In [ ]:
# ============================================================
# STEP 14A — DOWNLOAD AND INSPECT KNOTINFO DATABASE
# ============================================================

from pathlib import Path
import requests
import subprocess
import sys

TOPOLOGY_DIR = ch.stage_directory(
    OUTPUT_DIR,
    14,
    "topological_characterization",
)

KNOTINFO_URL = (
    "https://knotinfo.org/"
    "knotinfo_data_complete.xls"
)

KNOTINFO_FILE = (
    TOPOLOGY_DIR /
    "knotinfo_data_complete.xls"
)

# xlrd is needed for the legacy .xls format
try:
    import xlrd
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "xlrd",
        ]
    )

# Download only if it is not already saved
if not KNOTINFO_FILE.exists():

    print("Downloading KnotInfo database...")

    with requests.get(
        KNOTINFO_URL,
        stream=True,
        timeout=180,
    ) as response:

        response.raise_for_status()

        with open(KNOTINFO_FILE, "wb") as output_file:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    output_file.write(chunk)

else:
    print("Using cached KnotInfo file.")

print("KnotInfo file:")
print(KNOTINFO_FILE)

print(
    "Size MB:",
    KNOTINFO_FILE.stat().st_size / 1024**2,
)

# Inspect workbook structure
knotinfo_excel = pd.ExcelFile(
    KNOTINFO_FILE,
    engine="xlrd",
)

print("\nSheet names:")
print(knotinfo_excel.sheet_names)

# Read just a few rows first
for sheet_name in knotinfo_excel.sheet_names:

    preview = pd.read_excel(
        KNOTINFO_FILE,
        sheet_name=sheet_name,
        nrows=5,
        engine="xlrd",
    )

    print("\n" + "=" * 80)
    print("SHEET:", sheet_name)
    print("Shape of preview:", preview.shape)

    print("\nColumns:")
    for i, col in enumerate(preview.columns):
        print(i, repr(col))

    display(preview.head())

In [ ]:
# ============================================================
# STEP 14B — LOAD SLIM KNOTINFO TABLE AND AUDIT IDENTIFIERS
# ============================================================

import re

KNOTINFO_SLIM_FILE = (
    TOPOLOGY_DIR /
    "knotinfo_topological_annotations.parquet"
)

# ------------------------------------------------------------
# 1. Topological variables of interest
# ------------------------------------------------------------

knotinfo_cols = [
    # Identification
    "name",
    "category",
    "alternating",
    "dt_name",
    "classical_conway_name",
    "conway_notation",

    # Important knot families
    "two_bridge_notation",
    "montesinos_notation",
    "pretzel_notation",
    "geometric_type",

    # Structural properties
    "fibered",
    "adequate",
    "quasi_alternating",
    "almost_alternating",
    "positive_braid",
    "positive",
    "strongly_quasipositive",
    "quasipositive",
    "l_space",
    "ribbon",
    "small_large",

    # Classical numeric invariants
    "crossing_number",
    "three_genus",
    "smooth_four_genus",
    "crosscap_number",
    "bridge_index",
    "braid_index",
    "braid_length",
    "unknotting_number",
    "tunnel_number",
    "nakanishi_index",
    "determinant",
    "turaev_genus",
    "arf_invariant",

    # Concordance invariants for verification
    "signature",
    "rasmussen_invariant",
    "ozsvath_szabo_tau_invariant",

    # Geometry and symmetry
    "volume",
    "symmetry_type",
    "full_symmetry_group",
]

# ------------------------------------------------------------
# 2. Read once, then cache as parquet
# ------------------------------------------------------------

if KNOTINFO_SLIM_FILE.exists():

    print("Loading cached slim KnotInfo table.")

    knotinfo_slim = pd.read_parquet(
        KNOTINFO_SLIM_FILE
    )

else:

    print("Reading selected KnotInfo columns...")

    knotinfo_slim = pd.read_excel(
        KNOTINFO_FILE,
        sheet_name="sample_dat",
        usecols=knotinfo_cols,
        dtype=str,
        engine="xlrd",
    )

    # The first data row contains descriptive labels:
    # Name, Category, Alternating, etc.
    knotinfo_slim["crossing_number_numeric"] = (
        pd.to_numeric(
            knotinfo_slim["crossing_number"],
            errors="coerce",
        )
    )

    knotinfo_slim = knotinfo_slim.loc[
        knotinfo_slim[
            "crossing_number_numeric"
        ].notna()
    ].copy()

    knotinfo_slim.to_parquet(
        KNOTINFO_SLIM_FILE,
        index=False,
    )

print("Slim KnotInfo shape:", knotinfo_slim.shape)

# ------------------------------------------------------------
# 3. Normalize DT identifiers
#
# 15n_159774 -> 15n159774
# 15n159774  -> 15n159774
# ------------------------------------------------------------

def normalize_dt_identifier(value):

    if pd.isna(value):
        return np.nan

    value = str(value).lower().strip()

    # Remove underscores, braces, spaces and punctuation
    value = re.sub(
        r"[^0-9an]",
        "",
        value,
    )

    return value


knotinfo_slim["dt_key"] = (
    knotinfo_slim["dt_name"]
    .map(normalize_dt_identifier)
)

atlas["knot_key"] = (
    atlas["knot_id_base"]
    .map(normalize_dt_identifier)
)

# ------------------------------------------------------------
# 4. Check duplicates before merging
# ------------------------------------------------------------

knotinfo_duplicate_keys = (
    knotinfo_slim.loc[
        knotinfo_slim["dt_key"].duplicated(
            keep=False
        )
        & knotinfo_slim["dt_key"].notna()
    ]
    .sort_values("dt_key")
)

print(
    "Duplicated KnotInfo DT keys:",
    knotinfo_duplicate_keys[
        "dt_key"
    ].nunique(),
)

if len(knotinfo_duplicate_keys):
    display(
        knotinfo_duplicate_keys[
            [
                "name",
                "dt_name",
                "dt_key",
                "crossing_number",
            ]
        ].head(30)
    )

# ------------------------------------------------------------
# 5. Merge complete atlas with KnotInfo
# ------------------------------------------------------------

atlas_knotinfo = atlas.merge(
    knotinfo_slim,
    how="left",
    left_on="knot_key",
    right_on="dt_key",
    suffixes=(
        "_internal",
        "_knotinfo",
    ),
    indicator=True,
    validate="one_to_one",
)

n_matched = int(
    (atlas_knotinfo["_merge"] == "both").sum()
)

n_unmatched = int(
    (atlas_knotinfo["_merge"] == "left_only").sum()
)

print("\nAtlas rows:", len(atlas_knotinfo))
print("Matched to KnotInfo:", n_matched)
print("Unmatched:", n_unmatched)
print("Match fraction:", n_matched / len(atlas_knotinfo))

# ------------------------------------------------------------
# 6. Inspect unmatched identifiers
# ------------------------------------------------------------

unmatched_ids = atlas_knotinfo.loc[
    atlas_knotinfo["_merge"] == "left_only",
    [
        "knot_id_base",
        "knot_id_clean",
        "knot_key",
        "number_of_crossings",
    ],
].copy()

print("\nUnmatched preview:")
display(unmatched_ids.head(30))

# ------------------------------------------------------------
# 7. Crossing and alternating consistency audit
# ------------------------------------------------------------

atlas_knotinfo["crossing_number_knotinfo"] = (
    pd.to_numeric(
        atlas_knotinfo[
            "crossing_number"
        ],
        errors="coerce",
    )
)

crossing_disagreement = atlas_knotinfo.loc[
    atlas_knotinfo["_merge"].eq("both")
    & (
        atlas_knotinfo[
            "number_of_crossings"
        ]
        !=
        atlas_knotinfo[
            "crossing_number_knotinfo"
        ]
    )
]

print(
    "\nCrossing-number disagreements:",
    len(crossing_disagreement),
)

alternating_map = {
    "Y": True,
    "N": False,
    "Yes": True,
    "No": False,
    "1": True,
    "0": False,
}

atlas_knotinfo["alternating_knotinfo_bool"] = (
    atlas_knotinfo["alternating"]
    .map(alternating_map)
)

alternating_disagreement = atlas_knotinfo.loc[
    atlas_knotinfo[
        "alternating_knotinfo_bool"
    ].notna()
    & (
        atlas_knotinfo[
            "alternating_knotinfo_bool"
        ].astype(bool)
        !=
        atlas_knotinfo[
            "is_alternating"
        ].astype(bool)
    )
]

print(
    "Alternating-status disagreements:",
    len(alternating_disagreement),
)

# ------------------------------------------------------------
# 8. Extract the universal six-knot topology
# ------------------------------------------------------------

universal_keys = set(
    universal_core_export[
        "knot_id_base"
    ].map(normalize_dt_identifier)
)

universal_core_topology = (
    atlas_knotinfo.loc[
        atlas_knotinfo[
            "knot_key"
        ].isin(universal_keys),
        [
            "knot_id_base",
            "dt_name",
            "name",
            "number_of_crossings",
            "is_alternating",
            "signature_internal",
            "s_invariant_qc",
            "s_minus_sigma_qc",
            "raw_pca_membership",
            "conditional_membership",
            "ae_seed_membership",
            "ck_All_5",
            "geometric_type",
            "fibered",
            "two_bridge_notation",
            "montesinos_notation",
            "pretzel_notation",
            "adequate",
            "quasi_alternating",
            "almost_alternating",
            "positive",
            "positive_braid",
            "strongly_quasipositive",
            "quasipositive",
            "three_genus",
            "smooth_four_genus",
            "bridge_index",
            "braid_index",
            "unknotting_number",
            "turaev_genus",
            "volume",
            "symmetry_type",
        ],
    ]
    .sort_values(
        "ck_All_5",
        ascending=False,
    )
)

display(universal_core_topology)

# ------------------------------------------------------------
# 9. Save merged atlas
# ------------------------------------------------------------

atlas_knotinfo.to_parquet(
    TOPOLOGY_DIR /
    "complete_atlas_with_knotinfo.parquet",
    index=False,
)

universal_core_topology.to_csv(
    TOPOLOGY_DIR /
    "universal_core_topological_annotations.csv",
    index=False,
)

print("\nSaved merged topology atlas.")

Caracterizar el núcleo universal con SnapPy

In [ ]:
# ============================================================
# STEP 14C — SNAPPY CHARACTERIZATION OF UNIVERSAL CORE
# ============================================================

import sys
import subprocess
import importlib
import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Install SnapPy and the expanded 15-crossing census
# ------------------------------------------------------------

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "snappy",
        "snappy_15_knots",
    ]
)

importlib.invalidate_caches()

import snappy

print("SnapPy version:", snappy.__version__)

# This must equal the complete 15-crossing census
n_15_census = len(
    snappy.HTLinkExteriors(
        crossings=15
    )
)

print(
    "15-crossing knots in SnapPy census:",
    n_15_census,
)

assert n_15_census == 253_293, (
    "The extended 15-crossing database "
    "was not loaded correctly."
)

# ------------------------------------------------------------
# 2. Convert our identifiers to SnapPy names
#
# 15n159774 -> K15n159774
# 15a58615  -> K15a58615
# ------------------------------------------------------------

def to_snappy_name(knot_id):

    knot_id = str(knot_id).strip()

    compact = re.sub(
        r"[^0-9aAnN]",
        "",
        knot_id,
    )

    if re.fullmatch(
        r"\d+[aAnN]\d+",
        compact,
    ):
        return "K" + compact.lower()

    # Legacy identifiers such as 03_1
    match = re.fullmatch(
        r"0*(\d+)_0*(\d+)",
        knot_id,
    )

    if match:
        crossing = int(match.group(1))
        rank = int(match.group(2))
        return f"{crossing}_{rank}"

    return knot_id


# ------------------------------------------------------------
# 3. Safe SnapPy extraction
# ------------------------------------------------------------

def characterize_snappy_knot(
    knot_id,
    verify=True,
):

    snappy_name = to_snappy_name(
        knot_id
    )

    row = {
        "knot_id_base": knot_id,
        "snappy_name": snappy_name,
        "snappy_loaded": False,
        "solution_type": np.nan,
        "hyperbolicity_verified": np.nan,
        "volume": np.nan,
        "num_tetrahedra": np.nan,
        "is_two_bridge": np.nan,
        "two_bridge_fraction": np.nan,
        "symmetry_group": np.nan,
        "symmetry_order": np.nan,
        "symmetry_full_group": np.nan,
        "amphicheiral": np.nan,
        "invertible": np.nan,
        "isometry_signature": np.nan,
        "error": np.nan,
    }

    try:

        manifold = snappy.Manifold(
            snappy_name
        )

        row["snappy_loaded"] = True

        row["solution_type"] = str(
            manifold.solution_type()
        )

        row["volume"] = float(
            manifold.volume()
        )

        row["num_tetrahedra"] = int(
            manifold.num_tetrahedra()
        )

        # Two-bridge recognition
        two_bridge = (
            manifold.is_two_bridge()
        )

        row["is_two_bridge"] = bool(
            two_bridge
        )

        if two_bridge:
            row["two_bridge_fraction"] = str(
                two_bridge
            )

        # Hyperbolic symmetry group
        try:

            symmetry = (
                manifold.symmetry_group()
            )

            row["symmetry_group"] = str(
                symmetry
            )

            row["symmetry_order"] = int(
                symmetry.order()
            )

            row["symmetry_full_group"] = bool(
                symmetry.is_full_group()
            )

            row["amphicheiral"] = bool(
                symmetry.is_amphicheiral()
            )

            row["invertible"] = bool(
                symmetry.is_invertible_knot()
            )

        except Exception:
            pass

        # Complete invariant of the hyperbolic link complement
        try:

            row["isometry_signature"] = (
                manifold.isometry_signature(
                    of_link=True
                )
            )

        except Exception:
            pass

        # Rigorous verification if supported in this environment
        if verify:

            try:

                verified, _ = (
                    manifold.verify_hyperbolicity()
                )

                row[
                    "hyperbolicity_verified"
                ] = bool(verified)

            except Exception:

                # Some pip environments do not include
                # the interval-arithmetic backend.
                row[
                    "hyperbolicity_verified"
                ] = np.nan

    except Exception as error:

        row["error"] = repr(error)

    return row


# ------------------------------------------------------------
# 4. Characterize the universal six-knot core
# ------------------------------------------------------------

snappy_core_rows = []

for knot_id in universal_core_export[
    "knot_id_base"
]:

    print("Processing:", knot_id)

    snappy_core_rows.append(
        characterize_snappy_knot(
            knot_id,
            verify=True,
        )
    )

snappy_core = pd.DataFrame(
    snappy_core_rows
)

universal_core_snappy = (
    universal_core_export.merge(
        snappy_core,
        on="knot_id_base",
        how="left",
        validate="one_to_one",
    )
)

# ------------------------------------------------------------
# 5. Display central geometric information
# ------------------------------------------------------------

core_display_cols = [
    "knot_id_base",
    "is_alternating",
    "signature",
    "s_invariant_qc",
    "s_minus_sigma_qc",
    "conditional_membership",
    "ae_seed_membership",
    "ck_All_5",
    "solution_type",
    "hyperbolicity_verified",
    "volume",
    "num_tetrahedra",
    "is_two_bridge",
    "two_bridge_fraction",
    "symmetry_group",
    "symmetry_order",
    "amphicheiral",
    "invertible",
    "snappy_loaded",
    "error",
]

display(
    universal_core_snappy[
        core_display_cols
    ]
)

# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------

universal_core_snappy.to_csv(
    TOPOLOGY_DIR /
    "universal_core_snappy_characterization.csv",
    index=False,
)

print("\nLoaded successfully:")
print(
    universal_core_snappy[
        "snappy_loaded"
    ].value_counts(
        dropna=False
    )
)

In [ ]:
# ============================================================
# STEP 15 — SNAPPY ATLAS FOR RAW ∪ CONDITIONAL HARD KNOTS
# ============================================================

SNAPPY_CANDIDATE_FILE = (
    TOPOLOGY_DIR /
    "snappy_raw_conditional_union_659.parquet"
)

SNAPPY_CANDIDATE_SUMMARY_FILE = (
    TOPOLOGY_DIR /
    "snappy_hard_regime_summary.csv"
)

# ------------------------------------------------------------
# 1. Candidate union
# ------------------------------------------------------------

candidate_union = atlas.loc[
    (atlas["raw_pca_membership"] == 5)
    |
    (atlas["conditional_membership"] >= 3)
].copy()

candidate_union = (
    candidate_union
    .drop_duplicates("knot_id_base")
    .copy()
)

print(
    "Candidate union size:",
    len(candidate_union),
)

assert len(candidate_union) == 659

# ------------------------------------------------------------
# 2. Fast geometric characterization
# ------------------------------------------------------------

def characterize_snappy_fast(knot_id):

    snappy_name = to_snappy_name(
        knot_id
    )

    row = {
        "knot_id_base": knot_id,
        "snappy_name": snappy_name,
        "snappy_loaded": False,
        "solution_type": np.nan,
        "numerically_hyperbolic": np.nan,
        "volume": np.nan,
        "num_tetrahedra": np.nan,
        "is_two_bridge": np.nan,
        "two_bridge_fraction": np.nan,
        "symmetry_group": np.nan,
        "symmetry_order": np.nan,
        "symmetry_full_group": np.nan,
        "amphicheiral": np.nan,
        "invertible": np.nan,
        "error": np.nan,
    }

    try:

        manifold = snappy.Manifold(
            snappy_name
        )

        row["snappy_loaded"] = True

        solution_type = str(
            manifold.solution_type()
        )

        row["solution_type"] = (
            solution_type
        )

        row["numerically_hyperbolic"] = (
            "positively oriented"
            in solution_type.lower()
        )

        try:
            row["volume"] = float(
                manifold.volume()
            )
        except Exception:
            pass

        row["num_tetrahedra"] = int(
            manifold.num_tetrahedra()
        )

        # Two-bridge recognition
        try:

            two_bridge = (
                manifold.is_two_bridge()
            )

            row["is_two_bridge"] = bool(
                two_bridge
            )

            if two_bridge:
                row[
                    "two_bridge_fraction"
                ] = str(two_bridge)

        except Exception:
            pass

        # Hyperbolic symmetry information
        try:

            symmetry = (
                manifold.symmetry_group()
            )

            row["symmetry_group"] = str(
                symmetry
            )

            row["symmetry_order"] = int(
                symmetry.order()
            )

            row["symmetry_full_group"] = bool(
                symmetry.is_full_group()
            )

            row["amphicheiral"] = bool(
                symmetry.is_amphicheiral()
            )

            row["invertible"] = bool(
                symmetry.is_invertible_knot()
            )

        except Exception:
            pass

    except Exception as error:

        row["error"] = repr(error)

    return row


# ------------------------------------------------------------
# 3. Resume from checkpoint if available
# ------------------------------------------------------------

if SNAPPY_CANDIDATE_FILE.exists():

    snappy_candidate_results = (
        pd.read_parquet(
            SNAPPY_CANDIDATE_FILE
        )
    )

    print(
        "Recovered previous SnapPy results:",
        len(snappy_candidate_results),
    )

else:

    snappy_candidate_results = (
        pd.DataFrame()
    )

completed_ids = set()

if len(snappy_candidate_results):

    completed_ids = set(
        snappy_candidate_results[
            "knot_id_base"
        ].astype(str)
    )

remaining_ids = [
    knot_id
    for knot_id
    in candidate_union[
        "knot_id_base"
    ].astype(str)
    if knot_id not in completed_ids
]

print(
    "Remaining knots:",
    len(remaining_ids),
)

# ------------------------------------------------------------
# 4. Process with regular checkpoints
# ------------------------------------------------------------

new_rows = []

for position, knot_id in enumerate(
    remaining_ids,
    start=1,
):

    new_rows.append(
        characterize_snappy_fast(
            knot_id
        )
    )

    if (
        position % 50 == 0
        or
        position == len(remaining_ids)
    ):

        new_df = pd.DataFrame(
            new_rows
        )

        if len(snappy_candidate_results):

            snappy_candidate_results = (
                pd.concat(
                    [
                        snappy_candidate_results,
                        new_df,
                    ],
                    ignore_index=True,
                )
            )

        else:

            snappy_candidate_results = (
                new_df.copy()
            )

        snappy_candidate_results = (
            snappy_candidate_results
            .drop_duplicates(
                "knot_id_base",
                keep="last",
            )
        )

        snappy_candidate_results.to_parquet(
            SNAPPY_CANDIDATE_FILE,
            index=False,
        )

        print(
            f"Checkpoint: "
            f"{len(snappy_candidate_results)} / "
            f"{len(candidate_union)}"
        )

        new_rows = []

# ------------------------------------------------------------
# 5. Merge geometry with hard-regime atlas
# ------------------------------------------------------------

candidate_geometry = (
    candidate_union.merge(
        snappy_candidate_results,
        on="knot_id_base",
        how="left",
        validate="one_to_one",
    )
)

print("\nSnapPy load status:")
print(
    candidate_geometry[
        "snappy_loaded"
    ].value_counts(
        dropna=False
    )
)

print("\nErrors:")
display(
    candidate_geometry.loc[
        candidate_geometry[
            "snappy_loaded"
        ].ne(True),
        [
            "knot_id_base",
            "number_of_crossings",
            "hard_regime",
            "error",
        ],
    ]
)

# ------------------------------------------------------------
# 6. Geometric summary by regime
# ------------------------------------------------------------

snappy_regime_summary = (
    candidate_geometry
    .groupby(
        "hard_regime",
        observed=True,
    )
    .agg(
        n=(
            "knot_id_base",
            "size",
        ),
        snappy_coverage=(
            "snappy_loaded",
            "mean",
        ),
        numerical_hyperbolic_prop=(
            "numerically_hyperbolic",
            "mean",
        ),
        volume_mean=(
            "volume",
            "mean",
        ),
        volume_median=(
            "volume",
            "median",
        ),
        volume_q25=(
            "volume",
            lambda x: x.quantile(0.25),
        ),
        volume_q75=(
            "volume",
            lambda x: x.quantile(0.75),
        ),
        tetrahedra_median=(
            "num_tetrahedra",
            "median",
        ),
        two_bridge_prop=(
            "is_two_bridge",
            "mean",
        ),
        symmetry_order_median=(
            "symmetry_order",
            "median",
        ),
        amphicheiral_prop=(
            "amphicheiral",
            "mean",
        ),
        invertible_prop=(
            "invertible",
            "mean",
        ),
    )
    .reset_index()
)

display(snappy_regime_summary)

# ------------------------------------------------------------
# 7. Universal-core comparison row
# ------------------------------------------------------------

universal_core_summary = pd.DataFrame(
    [
        {
            "hard_regime": (
                "universal_PCA_AE_conditional_core"
            ),
            "n": len(
                universal_core_snappy
            ),
            "snappy_coverage": (
                universal_core_snappy[
                    "snappy_loaded"
                ].mean()
            ),
            "numerical_hyperbolic_prop": (
                universal_core_snappy[
                    "solution_type"
                ]
                .str.lower()
                .str.contains(
                    "positively oriented",
                    na=False,
                )
                .mean()
            ),
            "volume_mean": (
                universal_core_snappy[
                    "volume"
                ].mean()
            ),
            "volume_median": (
                universal_core_snappy[
                    "volume"
                ].median()
            ),
            "volume_q25": (
                universal_core_snappy[
                    "volume"
                ].quantile(0.25)
            ),
            "volume_q75": (
                universal_core_snappy[
                    "volume"
                ].quantile(0.75)
            ),
            "tetrahedra_median": (
                universal_core_snappy[
                    "num_tetrahedra"
                ].median()
            ),
            "two_bridge_prop": (
                universal_core_snappy[
                    "is_two_bridge"
                ].mean()
            ),
            "symmetry_order_median": (
                universal_core_snappy[
                    "symmetry_order"
                ].median()
            ),
            "amphicheiral_prop": (
                universal_core_snappy[
                    "amphicheiral"
                ].mean()
            ),
            "invertible_prop": (
                universal_core_snappy[
                    "invertible"
                ].mean()
            ),
        }
    ]
)

snappy_regime_summary_with_core = (
    pd.concat(
        [
            snappy_regime_summary,
            universal_core_summary,
        ],
        ignore_index=True,
    )
)

display(
    snappy_regime_summary_with_core
)

# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------

candidate_geometry.to_parquet(
    TOPOLOGY_DIR /
    "hard_regime_candidate_geometry.parquet",
    index=False,
)

snappy_regime_summary_with_core.to_csv(
    SNAPPY_CANDIDATE_SUMMARY_FILE,
    index=False,
)

print("\nSaved candidate geometry.")

In [ ]:
# ============================================================
# STEP 15B — FIX ZERO-PADDED HT IDENTIFIERS
# ============================================================

def to_snappy_name(knot_id):

    knot_id = str(knot_id).strip()

    compact = re.sub(
        r"[^0-9aAnN]",
        "",
        knot_id,
    ).lower()

    # Hoste–Thistlethwaite identifiers:
    #
    # 14a02298  -> K14a2298
    # 15n096562 -> K15n96562
    # 15n159774 -> K15n159774
    ht_match = re.fullmatch(
        r"0*(\d+)([an])0*(\d+)",
        compact,
    )

    if ht_match:

        crossing = int(
            ht_match.group(1)
        )

        alternating_code = (
            ht_match.group(2)
        )

        rank = int(
            ht_match.group(3)
        )

        return (
            f"K{crossing}"
            f"{alternating_code}"
            f"{rank}"
        )

    # Legacy Rolfsen-style identifiers:
    #
    # 03_1  -> 3_1
    # 08_01 -> 8_1
    legacy_match = re.fullmatch(
        r"0*(\d+)_0*(\d+)",
        knot_id,
    )

    if legacy_match:

        crossing = int(
            legacy_match.group(1)
        )

        rank = int(
            legacy_match.group(2)
        )

        return f"{crossing}_{rank}"

    return knot_id


# Verify the correction
name_audit = pd.DataFrame(
    {
        "internal_name": [
            "14a02298",
            "14n01604",
            "15n096562",
            "15n159774",
            "15a58615",
        ]
    }
)

name_audit["snappy_name"] = (
    name_audit["internal_name"]
    .map(to_snappy_name)
)

display(name_audit)

expected_names = [
    "K14a2298",
    "K14n1604",
    "K15n96562",
    "K15n159774",
    "K15a58615",
]

assert (
    name_audit["snappy_name"].tolist()
    ==
    expected_names
)

# ------------------------------------------------------------
# Retry only failed knots
# ------------------------------------------------------------

failed_ids = (
    snappy_candidate_results.loc[
        snappy_candidate_results[
            "snappy_loaded"
        ].ne(True),
        "knot_id_base",
    ]
    .astype(str)
    .tolist()
)

print(
    "Failed knots to retry:",
    len(failed_ids),
)

retry_rows = []

for position, knot_id in enumerate(
    failed_ids,
    start=1,
):

    retry_rows.append(
        characterize_snappy_fast(
            knot_id
        )
    )

    if (
        position % 50 == 0
        or
        position == len(failed_ids)
    ):

        print(
            f"Retried: "
            f"{position} / {len(failed_ids)}"
        )

retry_df = pd.DataFrame(
    retry_rows
)

# Remove old failed records and replace them
snappy_candidate_results = (
    pd.concat(
        [
            snappy_candidate_results.loc[
                ~snappy_candidate_results[
                    "knot_id_base"
                ]
                .astype(str)
                .isin(failed_ids)
            ],
            retry_df,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        "knot_id_base",
        keep="last",
    )
)

snappy_candidate_results.to_parquet(
    SNAPPY_CANDIDATE_FILE,
    index=False,
)

# ------------------------------------------------------------
# Rebuild merged candidate geometry
# ------------------------------------------------------------

candidate_geometry = (
    candidate_union.merge(
        snappy_candidate_results,
        on="knot_id_base",
        how="left",
        validate="one_to_one",
    )
)

print("\nFinal SnapPy status:")
print(
    candidate_geometry[
        "snappy_loaded"
    ].value_counts(
        dropna=False
    )
)

remaining_errors = (
    candidate_geometry.loc[
        candidate_geometry[
            "snappy_loaded"
        ].ne(True),
        [
            "knot_id_base",
            "snappy_name",
            "number_of_crossings",
            "hard_regime",
            "error",
        ],
    ]
)

print(
    "\nRemaining errors:",
    len(remaining_errors),
)

display(
    remaining_errors.head(30)
)

# ------------------------------------------------------------
# Recompute unbiased regime summary
# ------------------------------------------------------------

snappy_regime_summary = (
    candidate_geometry
    .groupby(
        "hard_regime",
        observed=True,
    )
    .agg(
        n=(
            "knot_id_base",
            "size",
        ),
        snappy_coverage=(
            "snappy_loaded",
            "mean",
        ),
        numerical_hyperbolic_prop=(
            "numerically_hyperbolic",
            "mean",
        ),
        volume_mean=(
            "volume",
            "mean",
        ),
        volume_median=(
            "volume",
            "median",
        ),
        volume_q25=(
            "volume",
            lambda x: x.quantile(0.25),
        ),
        volume_q75=(
            "volume",
            lambda x: x.quantile(0.75),
        ),
        tetrahedra_median=(
            "num_tetrahedra",
            "median",
        ),
        two_bridge_prop=(
            "is_two_bridge",
            "mean",
        ),
        symmetry_order_median=(
            "symmetry_order",
            "median",
        ),
        amphicheiral_prop=(
            "amphicheiral",
            "mean",
        ),
        invertible_prop=(
            "invertible",
            "mean",
        ),
    )
    .reset_index()
)

snappy_regime_summary_with_core = (
    pd.concat(
        [
            snappy_regime_summary,
            universal_core_summary,
        ],
        ignore_index=True,
    )
)

display(
    snappy_regime_summary_with_core
)

# ------------------------------------------------------------
# Save corrected outputs
# ------------------------------------------------------------

candidate_geometry.to_parquet(
    TOPOLOGY_DIR /
    "hard_regime_candidate_geometry.parquet",
    index=False,
)

snappy_regime_summary_with_core.to_csv(
    TOPOLOGY_DIR /
    "snappy_hard_regime_summary.csv",
    index=False,
)

print("\nCorrected geometry saved.")

Matching geométrico: sensibilidad de caliper

In [ ]:
# ============================================================
# STEP 16A — GEOMETRIC MATCHING DESIGN
# ============================================================

from pathlib import Path

GEOMETRY_MATCH_DIR = ch.stage_directory(
    OUTPUT_DIR,
    16,
    "geometric_matched_controls",
)

# ------------------------------------------------------------
# 1. Inspect the two non-positive initial triangulations
# ------------------------------------------------------------

nonpositive_geometry = candidate_geometry.loc[
    candidate_geometry[
        "numerically_hyperbolic"
    ].ne(True),
    [
        "knot_id_base",
        "number_of_crossings",
        "is_alternating",
        "signature",
        "s_invariant_qc",
        "s_minus_sigma_qc",
        "hard_regime",
        "solution_type",
        "volume",
        "num_tetrahedra",
        "is_two_bridge",
    ],
].copy()

print(
    "Non-positive initial triangulations:",
    len(nonpositive_geometry),
)

display(nonpositive_geometry)

# ------------------------------------------------------------
# 2. Define mutually exclusive regimes
# ------------------------------------------------------------

geometry_match_base = atlas.copy()

geometry_match_base[
    "is_amplitude_only"
] = (
    geometry_match_base[
        "hard_regime"
    ]
    ==
    "amplitude_hard_only"
)

geometry_match_base[
    "is_conditional_only"
] = (
    geometry_match_base[
        "hard_regime"
    ]
    ==
    "conditional_hard_only"
)

geometry_match_base[
    "is_shared_hard"
] = (
    geometry_match_base[
        "hard_regime"
    ]
    ==
    "shared_raw_and_conditional"
)

regime_specs = {
    "amplitude_only": (
        "is_amplitude_only"
    ),
    "conditional_only": (
        "is_conditional_only"
    ),
    "shared": (
        "is_shared_hard"
    ),
}

exact_cols = [
    "number_of_crossings",
    "is_alternating",
    "signature_bin",
]

norm_cols = [
    "mean_log_sq_norm",
    "max_log_sq_norm",
]

for col in exact_cols + norm_cols:

    assert col in geometry_match_base.columns, (
        f"Missing matching column: {col}"
    )

# ------------------------------------------------------------
# 3. Obtain the five nearest controls once
#
# We use caliper=None here and subsequently filter the
# resulting distances. This avoids repeating the expensive
# nearest-neighbor search for every caliper.
# ------------------------------------------------------------

geometry_match_pairs = {}
geometry_unmatched_exact = {}

for regime_name, selected_col in (
    regime_specs.items()
):

    print(
        "\nMatching regime:",
        regime_name,
    )

    (
        matched_selected,
        matched_controls,
        matched_pairs,
        unmatched,
    ) = ch.nearest_norm_matched_controls(
        df=geometry_match_base,
        selected_col=selected_col,
        exact_cols=exact_cols,
        norm_cols=norm_cols,
        ratio=5,
        caliper=None,
        replace=True,
        seed=42,
    )

    geometry_match_pairs[
        regime_name
    ] = matched_pairs.copy()

    geometry_unmatched_exact[
        regime_name
    ] = unmatched.copy()

    matched_pairs.to_csv(
        GEOMETRY_MATCH_DIR /
        f"{regime_name}_nearest5_pairs.csv",
        index=False,
    )

    unmatched.to_csv(
        GEOMETRY_MATCH_DIR /
        f"{regime_name}_exact_unmatched.csv",
        index=False,
    )

    print(
        "Selected:",
        int(
            geometry_match_base[
                selected_col
            ].sum()
        ),
    )

    print(
        "Exact-stratum unmatched:",
        len(unmatched),
    )

    print(
        "Nearest pairs:",
        len(matched_pairs),
    )

# ------------------------------------------------------------
# 4. Balance function
# ------------------------------------------------------------

def standardized_mean_difference_local(
    selected_values,
    control_values,
):

    selected_values = np.asarray(
        selected_values,
        dtype=float,
    )

    control_values = np.asarray(
        control_values,
        dtype=float,
    )

    selected_values = selected_values[
        np.isfinite(selected_values)
    ]

    control_values = control_values[
        np.isfinite(control_values)
    ]

    if (
        len(selected_values) == 0
        or
        len(control_values) == 0
    ):
        return np.nan

    pooled_sd = np.sqrt(
        (
            np.var(selected_values)
            +
            np.var(control_values)
        )
        /
        2
    )

    if pooled_sd == 0:
        return 0.0

    return float(
        (
            np.mean(selected_values)
            -
            np.mean(control_values)
        )
        /
        pooled_sd
    )


# ------------------------------------------------------------
# 5. Derive caliper sensitivity from stored distances
# ------------------------------------------------------------

calipers = [
    0.20,
    0.35,
    0.50,
    0.75,
    None,
]

sensitivity_rows = []

for regime_name, selected_col in (
    regime_specs.items()
):

    pairs_all = (
        geometry_match_pairs[
            regime_name
        ]
    )

    n_selected_total = int(
        geometry_match_base[
            selected_col
        ].sum()
    )

    for caliper in calipers:

        if caliper is None:

            pairs = (
                pairs_all.copy()
            )

        else:

            pairs = pairs_all.loc[
                pairs_all[
                    "match_distance"
                ]
                <=
                caliper
            ].copy()

        selected_indices = (
            pairs[
                "selected_index"
            ]
            .drop_duplicates()
            .to_numpy()
        )

        control_indices = (
            pairs[
                "control_index"
            ]
            .to_numpy()
        )

        selected_matched = (
            geometry_match_base.loc[
                selected_indices
            ]
        )

        controls_matched = (
            geometry_match_base.loc[
                control_indices
            ]
        )

        norm_smds = {}

        for norm_col in norm_cols:

            norm_smds[norm_col] = (
                standardized_mean_difference_local(
                    selected_matched[
                        norm_col
                    ],
                    controls_matched[
                        norm_col
                    ],
                )
            )

        sensitivity_rows.append(
            {
                "regime": regime_name,
                "caliper": (
                    "None"
                    if caliper is None
                    else caliper
                ),
                "n_selected_total": (
                    n_selected_total
                ),
                "n_selected_matched": (
                    len(selected_indices)
                ),
                "match_coverage": (
                    len(selected_indices)
                    /
                    n_selected_total
                ),
                "n_pairs": len(pairs),
                "n_unique_controls": (
                    pairs[
                        "control_index"
                    ].nunique()
                ),
                "mean_match_distance": (
                    pairs[
                        "match_distance"
                    ].mean()
                ),
                "median_match_distance": (
                    pairs[
                        "match_distance"
                    ].median()
                ),
                "mean_norm_smd": (
                    norm_smds[
                        "mean_log_sq_norm"
                    ]
                ),
                "max_norm_smd": (
                    norm_smds[
                        "max_log_sq_norm"
                    ]
                ),
                "max_abs_norm_smd": (
                    np.nanmax(
                        np.abs(
                            list(
                                norm_smds.values()
                            )
                        )
                    )
                ),
            }
        )

geometry_matching_sensitivity = (
    pd.DataFrame(
        sensitivity_rows
    )
)

display(
    geometry_matching_sensitivity
)

geometry_matching_sensitivity.to_csv(
    GEOMETRY_MATCH_DIR /
    "geometry_matching_caliper_sensitivity.csv",
    index=False,
)

print("\nSaved matching sensitivity.")

Matching limpio contra background

In [ ]:
# Recorded run fails the final balance gate (shared group SMD = 0.288980).
# This assertion is intentionally retained; these comparisons are exploratory.
# ============================================================
# STEP 16B — CLEAN GEOMETRIC MATCHING
#
# Controls are restricted to hard_regime == "neither".
# ============================================================

chosen_calipers = {
    "amplitude_only": 0.50,
    "conditional_only": 0.50,
    "shared": 0.35,
}

calipers = [
    0.20,
    0.35,
    0.50,
    0.75,
    None,
]

clean_geometry_match_pairs = {}
clean_geometry_unmatched = {}
clean_sensitivity_rows = []

# ------------------------------------------------------------
# 1. Match each regime only against "neither"
# ------------------------------------------------------------

for regime_name, selected_col in (
    regime_specs.items()
):

    selected_mask = (
        geometry_match_base[
            selected_col
        ]
    )

    background_mask = (
        geometry_match_base[
            "hard_regime"
        ]
        ==
        "neither"
    )

    clean_match_df = (
        geometry_match_base.loc[
            selected_mask
            |
            background_mask
        ]
        .copy()
    )

    print(
        "\nClean matching regime:",
        regime_name,
    )

    (
        matched_selected,
        matched_controls,
        matched_pairs,
        unmatched,
    ) = ch.nearest_norm_matched_controls(
        df=clean_match_df,
        selected_col=selected_col,
        exact_cols=exact_cols,
        norm_cols=norm_cols,
        ratio=5,
        caliper=None,
        replace=True,
        seed=42,
    )

    clean_geometry_match_pairs[
        regime_name
    ] = matched_pairs.copy()

    clean_geometry_unmatched[
        regime_name
    ] = unmatched.copy()

    # Ensure that every control is from neither
    control_regimes = (
        geometry_match_base.loc[
            matched_pairs[
                "control_index"
            ].to_numpy(),
            "hard_regime",
        ]
    )

    assert (
        control_regimes
        ==
        "neither"
    ).all()

    print(
        "Selected:",
        int(selected_mask.sum()),
    )

    print(
        "Exact-stratum unmatched:",
        len(unmatched),
    )

    print(
        "Nearest pairs:",
        len(matched_pairs),
    )

    # Save complete nearest-5 pairs
    matched_pairs.to_csv(
        GEOMETRY_MATCH_DIR /
        f"{regime_name}_clean_nearest5_pairs.csv",
        index=False,
    )

    unmatched.to_csv(
        GEOMETRY_MATCH_DIR /
        f"{regime_name}_clean_exact_unmatched.csv",
        index=False,
    )

    # --------------------------------------------------------
    # 2. Caliper sensitivity
    # --------------------------------------------------------

    n_selected_total = int(
        selected_mask.sum()
    )

    for caliper in calipers:

        if caliper is None:

            pairs = (
                matched_pairs.copy()
            )

        else:

            pairs = matched_pairs.loc[
                matched_pairs[
                    "match_distance"
                ]
                <=
                caliper
            ].copy()

        selected_indices = (
            pairs[
                "selected_index"
            ]
            .drop_duplicates()
            .to_numpy()
        )

        control_indices = (
            pairs[
                "control_index"
            ]
            .to_numpy()
        )

        selected_matched = (
            geometry_match_base.loc[
                selected_indices
            ]
        )

        controls_matched = (
            geometry_match_base.loc[
                control_indices
            ]
        )

        norm_smds = {}

        for norm_col in norm_cols:

            norm_smds[norm_col] = (
                standardized_mean_difference_local(
                    selected_matched[
                        norm_col
                    ],
                    controls_matched[
                        norm_col
                    ],
                )
            )

        clean_sensitivity_rows.append(
            {
                "regime": regime_name,
                "caliper": (
                    "None"
                    if caliper is None
                    else caliper
                ),
                "n_selected_total": (
                    n_selected_total
                ),
                "n_selected_matched": (
                    len(selected_indices)
                ),
                "match_coverage": (
                    len(selected_indices)
                    /
                    n_selected_total
                ),
                "n_pairs": len(pairs),
                "n_unique_controls": (
                    pairs[
                        "control_index"
                    ].nunique()
                ),
                "mean_match_distance": (
                    pairs[
                        "match_distance"
                    ].mean()
                ),
                "median_match_distance": (
                    pairs[
                        "match_distance"
                    ].median()
                ),
                "mean_norm_smd": (
                    norm_smds[
                        "mean_log_sq_norm"
                    ]
                ),
                "max_norm_smd": (
                    norm_smds[
                        "max_log_sq_norm"
                    ]
                ),
                "max_abs_norm_smd": (
                    np.nanmax(
                        np.abs(
                            list(
                                norm_smds.values()
                            )
                        )
                    )
                ),
            }
        )

clean_geometry_matching_sensitivity = (
    pd.DataFrame(
        clean_sensitivity_rows
    )
)

display(
    clean_geometry_matching_sensitivity
)

# ------------------------------------------------------------
# 3. Freeze chosen clean pairs
# ------------------------------------------------------------

geometry_chosen_pairs = {}

chosen_rows = []

for regime_name, chosen_caliper in (
    chosen_calipers.items()
):

    pairs_all = (
        clean_geometry_match_pairs[
            regime_name
        ]
    )

    chosen_pairs = pairs_all.loc[
        pairs_all[
            "match_distance"
        ]
        <=
        chosen_caliper
    ].copy()

    geometry_chosen_pairs[
        regime_name
    ] = chosen_pairs

    chosen_pairs.to_csv(
        GEOMETRY_MATCH_DIR /
        (
            f"{regime_name}_"
            f"chosen_clean_pairs.csv"
        ),
        index=False,
    )

    sensitivity_row = (
        clean_geometry_matching_sensitivity.loc[
            (
                clean_geometry_matching_sensitivity[
                    "regime"
                ]
                ==
                regime_name
            )
            &
            (
                clean_geometry_matching_sensitivity[
                    "caliper"
                ].astype(str)
                ==
                str(chosen_caliper)
            )
        ]
        .copy()
    )

    chosen_rows.append(
        sensitivity_row
    )

chosen_geometry_design = (
    pd.concat(
        chosen_rows,
        ignore_index=True,
    )
)

print("\nChosen clean matching designs:")
display(chosen_geometry_design)

# ------------------------------------------------------------
# 4. Support assertions
# ------------------------------------------------------------

chosen_design_indexed = (
    chosen_geometry_design
    .set_index("regime")
)

assert (
    chosen_design_indexed.loc[
        "amplitude_only",
        "match_coverage",
    ]
    >=
    0.75
)

assert (
    chosen_design_indexed.loc[
        "conditional_only",
        "match_coverage",
    ]
    >=
    0.90
)

assert (
    chosen_design_indexed.loc[
        "shared",
        "match_coverage",
    ]
    >=
    0.45
)

assert (
    chosen_geometry_design[
        "max_abs_norm_smd"
    ].max()
    <
    0.10
)

# ------------------------------------------------------------
# 5. Save
# ------------------------------------------------------------

clean_geometry_matching_sensitivity.to_csv(
    GEOMETRY_MATCH_DIR /
    "clean_geometry_matching_caliper_sensitivity.csv",
    index=False,
)

chosen_geometry_design.to_csv(
    GEOMETRY_MATCH_DIR /
    "chosen_clean_geometry_matching_design.csv",
    index=False,
)

print("\nClean matching design saved.")

In [ ]:
# ============================================================
# STEP 16C — UNIQUE ONE-TO-ONE GEOMETRIC MATCHING
# ============================================================

import networkx as nx

# ------------------------------------------------------------
# 1. Shared is explicitly exploratory
# ------------------------------------------------------------

shared_support_limitation = (
    clean_geometry_matching_sensitivity.loc[
        clean_geometry_matching_sensitivity[
            "regime"
        ]
        ==
        "shared"
    ]
    .copy()
)

shared_support_limitation[
    "inferential_status"
] = (
    "insufficient common support; descriptive only"
)

shared_support_limitation.to_csv(
    GEOMETRY_MATCH_DIR /
    "shared_common_support_limitation.csv",
    index=False,
)

# ------------------------------------------------------------
# 2. Valid inferential regimes
# ------------------------------------------------------------

inferential_calipers = {
    "amplitude_only": 0.50,
    "conditional_only": 0.50,
}

inferential_pairs_with_replacement = {}

for regime_name, caliper in (
    inferential_calipers.items()
):

    pairs = (
        clean_geometry_match_pairs[
            regime_name
        ]
        .loc[
            lambda df:
            df["match_distance"]
            <=
            caliper
        ]
        .copy()
    )

    inferential_pairs_with_replacement[
        regime_name
    ] = pairs

# ------------------------------------------------------------
# 3. Maximum-cardinality minimum-distance bipartite matching
#
# Each selected knot and each control can appear only once.
# ------------------------------------------------------------

def maximum_unique_matching(
    pairs,
):

    graph = nx.Graph()

    for row in pairs.itertuples(
        index=False
    ):

        selected_node = (
            "selected",
            int(row.selected_index),
        )

        control_node = (
            "control",
            int(row.control_index),
        )

        # Maximize cardinality first and then prefer
        # smaller matching distance.
        weight = (
            1_000_000.0
            -
            float(row.match_distance)
        )

        graph.add_edge(
            selected_node,
            control_node,
            weight=weight,
            match_distance=float(
                row.match_distance
            ),
        )

    matching = nx.algorithms.matching.max_weight_matching(
        graph,
        maxcardinality=True,
        weight="weight",
    )

    records = []

    for node_a, node_b in matching:

        if node_a[0] == "selected":
            selected_node = node_a
            control_node = node_b
        else:
            selected_node = node_b
            control_node = node_a

        edge_data = graph.get_edge_data(
            selected_node,
            control_node,
        )

        records.append(
            {
                "selected_index": (
                    selected_node[1]
                ),
                "control_index": (
                    control_node[1]
                ),
                "match_group_id": str(
                    selected_node[1]
                ),
                "match_distance": (
                    edge_data[
                        "match_distance"
                    ]
                ),
            }
        )

    return (
        pd.DataFrame(records)
        .sort_values(
            "selected_index"
        )
        .reset_index(
            drop=True
        )
    )


geometry_unique_pairs = {}
unique_design_rows = []

for regime_name, pairs in (
    inferential_pairs_with_replacement.items()
):

    unique_pairs = (
        maximum_unique_matching(
            pairs
        )
    )

    geometry_unique_pairs[
        regime_name
    ] = unique_pairs

    selected_indices = (
        unique_pairs[
            "selected_index"
        ].to_numpy()
    )

    control_indices = (
        unique_pairs[
            "control_index"
        ].to_numpy()
    )

    selected_df = (
        geometry_match_base.loc[
            selected_indices
        ]
    )

    control_df = (
        geometry_match_base.loc[
            control_indices
        ]
    )

    # Verify uniqueness
    assert (
        unique_pairs[
            "selected_index"
        ].is_unique
    )

    assert (
        unique_pairs[
            "control_index"
        ].is_unique
    )

    # Verify pure background controls
    assert (
        control_df[
            "hard_regime"
        ]
        ==
        "neither"
    ).all()

    norm_smds = {
        norm_col:
        standardized_mean_difference_local(
            selected_df[norm_col],
            control_df[norm_col],
        )
        for norm_col in norm_cols
    }

    selected_col = (
        regime_specs[
            regime_name
        ]
    )

    n_total = int(
        geometry_match_base[
            selected_col
        ].sum()
    )

    unique_design_rows.append(
        {
            "regime": regime_name,
            "caliper": (
                inferential_calipers[
                    regime_name
                ]
            ),
            "n_selected_total": n_total,
            "n_unique_pairs": (
                len(unique_pairs)
            ),
            "coverage": (
                len(unique_pairs)
                /
                n_total
            ),
            "mean_match_distance": (
                unique_pairs[
                    "match_distance"
                ].mean()
            ),
            "median_match_distance": (
                unique_pairs[
                    "match_distance"
                ].median()
            ),
            "mean_norm_smd": (
                norm_smds[
                    "mean_log_sq_norm"
                ]
            ),
            "max_norm_smd": (
                norm_smds[
                    "max_log_sq_norm"
                ]
            ),
            "max_abs_norm_smd": (
                max(
                    abs(value)
                    for value
                    in norm_smds.values()
                )
            ),
        }
    )

    unique_pairs.to_csv(
        GEOMETRY_MATCH_DIR /
        (
            f"{regime_name}_"
            "unique_one_to_one_pairs.csv"
        ),
        index=False,
    )

unique_geometry_design = (
    pd.DataFrame(
        unique_design_rows
    )
)

display(unique_geometry_design)

# ------------------------------------------------------------
# 4. Save all design decisions
# ------------------------------------------------------------

clean_geometry_matching_sensitivity.to_csv(
    GEOMETRY_MATCH_DIR /
    "clean_geometry_matching_caliper_sensitivity.csv",
    index=False,
)

unique_geometry_design.to_csv(
    GEOMETRY_MATCH_DIR /
    "unique_geometry_matching_design.csv",
    index=False,
)

matching_decisions = pd.DataFrame(
    [
        {
            "regime": "amplitude_only",
            "status": "inferential",
            "design": (
                "exact structural + nearest norm; "
                "caliper 0.50; unique 1:1"
            ),
        },
        {
            "regime": "conditional_only",
            "status": "inferential",
            "design": (
                "exact structural + nearest norm; "
                "caliper 0.50; unique 1:1"
            ),
        },
        {
            "regime": "shared",
            "status": "descriptive_only",
            "design": (
                "insufficient common support "
                "after norm adjustment"
            ),
        },
        {
            "regime": "universal_core",
            "status": "case_study",
            "design": (
                "six robust PCA + AE + "
                "conditional examples"
            ),
        },
    ]
)

matching_decisions.to_csv(
    GEOMETRY_MATCH_DIR /
    "geometry_inference_decisions.csv",
    index=False,
)

display(matching_decisions)

print("\nInferential design frozen.")

In [ ]:
# ============================================================
# STEP 16D — FINAL MATCHED GEOMETRIC INFERENCE
# ============================================================

from scipy.stats import wilcoxon, binomtest

SNAPPY_CONTROL_FILE = (
    GEOMETRY_MATCH_DIR /
    "snappy_unique_matched_controls.parquet"
)

# ------------------------------------------------------------
# 1. Collect all unique matched controls
# ------------------------------------------------------------

all_control_indices = sorted(
    set().union(
        *[
            set(
                pairs[
                    "control_index"
                ].astype(int)
            )
            for pairs
            in geometry_unique_pairs.values()
        ]
    )
)

control_metadata = (
    geometry_match_base.loc[
        all_control_indices,
        [
            "knot_id_base",
            "hard_regime",
            "number_of_crossings",
            "is_alternating",
            "signature",
            "s_invariant_qc",
            "s_minus_sigma_qc",
            "mean_log_sq_norm",
            "max_log_sq_norm",
        ],
    ]
    .copy()
)

assert (
    control_metadata[
        "hard_regime"
    ]
    ==
    "neither"
).all()

print(
    "Unique controls to process:",
    len(control_metadata),
)

# ------------------------------------------------------------
# 2. Resume SnapPy control processing
# ------------------------------------------------------------

if SNAPPY_CONTROL_FILE.exists():

    snappy_control_results = (
        pd.read_parquet(
            SNAPPY_CONTROL_FILE
        )
    )

    print(
        "Recovered control results:",
        len(snappy_control_results),
    )

else:

    snappy_control_results = (
        pd.DataFrame()
    )

completed_control_ids = set()

if len(snappy_control_results):

    completed_control_ids = set(
        snappy_control_results[
            "knot_id_base"
        ].astype(str)
    )

remaining_control_ids = [
    knot_id
    for knot_id
    in control_metadata[
        "knot_id_base"
    ].astype(str)
    if knot_id
    not in completed_control_ids
]

print(
    "Remaining controls:",
    len(remaining_control_ids),
)

new_control_rows = []

for position, knot_id in enumerate(
    remaining_control_ids,
    start=1,
):

    new_control_rows.append(
        characterize_snappy_fast(
            knot_id
        )
    )

    if (
        position % 50 == 0
        or
        position
        ==
        len(remaining_control_ids)
    ):

        new_df = pd.DataFrame(
            new_control_rows
        )

        if len(snappy_control_results):

            snappy_control_results = (
                pd.concat(
                    [
                        snappy_control_results,
                        new_df,
                    ],
                    ignore_index=True,
                )
            )

        else:

            snappy_control_results = (
                new_df.copy()
            )

        snappy_control_results = (
            snappy_control_results
            .drop_duplicates(
                "knot_id_base",
                keep="last",
            )
        )

        snappy_control_results.to_parquet(
            SNAPPY_CONTROL_FILE,
            index=False,
        )

        print(
            f"Control checkpoint: "
            f"{len(snappy_control_results)} / "
            f"{len(control_metadata)}"
        )

        new_control_rows = []

print("\nControl load status:")
print(
    snappy_control_results[
        "snappy_loaded"
    ].value_counts(
        dropna=False
    )
)

# ------------------------------------------------------------
# 3. Combine candidate and control geometry
# ------------------------------------------------------------

geometry_lookup = (
    pd.concat(
        [
            snappy_candidate_results,
            snappy_control_results,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        "knot_id_base",
        keep="first",
    )
    .set_index(
        "knot_id_base"
    )
)

# Hyperbolic volume is retained only when the initial
# triangulation has all tetrahedra positively oriented.
geometry_lookup[
    "hyperbolic_volume"
] = geometry_lookup[
    "volume"
].where(
    geometry_lookup[
        "numerically_hyperbolic"
    ].eq(True)
)

# Numeric versions of boolean outcomes
for boolean_col in [
    "numerically_hyperbolic",
    "is_two_bridge",
    "amphicheiral",
    "invertible",
]:

    geometry_lookup[
        boolean_col + "_numeric"
    ] = (
        geometry_lookup[
            boolean_col
        ]
        .map(
            {
                True: 1.0,
                False: 0.0,
            }
        )
    )

# ------------------------------------------------------------
# 4. Inspect non-positive control triangulations
# ------------------------------------------------------------

nonpositive_controls = (
    snappy_control_results.loc[
        snappy_control_results[
            "numerically_hyperbolic"
        ].ne(True),
        [
            "knot_id_base",
            "snappy_name",
            "solution_type",
            "volume",
            "num_tetrahedra",
            "is_two_bridge",
            "error",
        ],
    ]
    .copy()
)

print(
    "\nNon-positive control triangulations:",
    len(nonpositive_controls),
)

display(nonpositive_controls)

# ------------------------------------------------------------
# 5. Benjamini-Hochberg implementation
# ------------------------------------------------------------

def benjamini_hochberg(
    p_values,
):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    n_tests = len(p_values)

    order = np.argsort(
        p_values
    )

    ranked_p = p_values[
        order
    ]

    adjusted = (
        ranked_p
        *
        n_tests
        /
        np.arange(
            1,
            n_tests + 1,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.clip(
        adjusted,
        0,
        1,
    )

    output = np.empty(
        n_tests,
        dtype=float,
    )

    output[order] = adjusted

    return output


# ------------------------------------------------------------
# 6. Outcome specifications
# ------------------------------------------------------------

continuous_outcomes = {
    "hyperbolic_volume": (
        "Hyperbolic volume"
    ),
    "num_tetrahedra": (
        "Census triangulation tetrahedra"
    ),
    "symmetry_order": (
        "Symmetry-group order"
    ),
}

binary_outcomes = {
    "numerically_hyperbolic_numeric": (
        "Numerically positive hyperbolic solution"
    ),
    "is_two_bridge_numeric": (
        "SnapPy-recognized two-bridge"
    ),
    "invertible_numeric": (
        "Invertible"
    ),
    "amphicheiral_numeric": (
        "Amphicheiral"
    ),
}

# ------------------------------------------------------------
# 7. Paired tests
# ------------------------------------------------------------

paired_geometry_rows = []
paired_geometry_tables = {}

index_to_knot = (
    geometry_match_base[
        "knot_id_base"
    ]
)

for regime_name, pairs in (
    geometry_unique_pairs.items()
):

    pair_table = pairs.copy()

    pair_table[
        "selected_knot_id"
    ] = (
        pair_table[
            "selected_index"
        ]
        .map(index_to_knot)
    )

    pair_table[
        "control_knot_id"
    ] = (
        pair_table[
            "control_index"
        ]
        .map(index_to_knot)
    )

    # Add all outcomes to the pair-level table
    all_outcomes = (
        list(
            continuous_outcomes.keys()
        )
        +
        list(
            binary_outcomes.keys()
        )
    )

    for outcome_col in all_outcomes:

        pair_table[
            "selected_" + outcome_col
        ] = (
            geometry_lookup[
                outcome_col
            ]
            .reindex(
                pair_table[
                    "selected_knot_id"
                ]
            )
            .to_numpy()
        )

        pair_table[
            "control_" + outcome_col
        ] = (
            geometry_lookup[
                outcome_col
            ]
            .reindex(
                pair_table[
                    "control_knot_id"
                ]
            )
            .to_numpy()
        )

    paired_geometry_tables[
        regime_name
    ] = pair_table.copy()

    pair_table.to_csv(
        GEOMETRY_MATCH_DIR /
        f"{regime_name}_final_geometry_pairs.csv",
        index=False,
    )

    # Continuous outcomes: paired Wilcoxon
    for outcome_col, outcome_label in (
        continuous_outcomes.items()
    ):

        selected_values = pd.to_numeric(
            pair_table[
                "selected_" + outcome_col
            ],
            errors="coerce",
        ).to_numpy()

        control_values = pd.to_numeric(
            pair_table[
                "control_" + outcome_col
            ],
            errors="coerce",
        ).to_numpy()

        valid = (
            np.isfinite(selected_values)
            &
            np.isfinite(control_values)
        )

        selected_valid = (
            selected_values[valid]
        )

        control_valid = (
            control_values[valid]
        )

        differences = (
            selected_valid
            -
            control_valid
        )

        if (
            len(differences) == 0
            or
            np.allclose(
                differences,
                0,
            )
        ):

            p_value = 1.0

        else:

            p_value = float(
                wilcoxon(
                    selected_valid,
                    control_valid,
                    alternative="two-sided",
                    zero_method="wilcox",
                ).pvalue
            )

        difference_sd = np.std(
            differences,
            ddof=1,
        )

        paired_smd = (
            np.mean(differences)
            /
            difference_sd
            if difference_sd > 0
            else np.nan
        )

        paired_geometry_rows.append(
            {
                "regime": regime_name,
                "outcome": outcome_label,
                "outcome_type": "continuous",
                "n_valid_pairs": (
                    len(differences)
                ),
                "selected_mean": (
                    np.mean(
                        selected_valid
                    )
                ),
                "control_mean": (
                    np.mean(
                        control_valid
                    )
                ),
                "mean_paired_difference": (
                    np.mean(
                        differences
                    )
                ),
                "median_paired_difference": (
                    np.median(
                        differences
                    )
                ),
                "paired_smd": paired_smd,
                "selected_prop": np.nan,
                "control_prop": np.nan,
                "risk_difference": np.nan,
                "selected_yes_control_no": np.nan,
                "selected_no_control_yes": np.nan,
                "p_value": p_value,
                "test": "paired Wilcoxon",
            }
        )

    # Binary outcomes: exact McNemar/binomial test
    for outcome_col, outcome_label in (
        binary_outcomes.items()
    ):

        selected_values = pd.to_numeric(
            pair_table[
                "selected_" + outcome_col
            ],
            errors="coerce",
        ).to_numpy()

        control_values = pd.to_numeric(
            pair_table[
                "control_" + outcome_col
            ],
            errors="coerce",
        ).to_numpy()

        valid = (
            np.isfinite(selected_values)
            &
            np.isfinite(control_values)
        )

        selected_valid = (
            selected_values[valid]
            .astype(int)
        )

        control_valid = (
            control_values[valid]
            .astype(int)
        )

        selected_yes_control_no = int(
            (
                (selected_valid == 1)
                &
                (control_valid == 0)
            ).sum()
        )

        selected_no_control_yes = int(
            (
                (selected_valid == 0)
                &
                (control_valid == 1)
            ).sum()
        )

        n_discordant = (
            selected_yes_control_no
            +
            selected_no_control_yes
        )

        if n_discordant == 0:

            p_value = 1.0

        else:

            p_value = float(
                binomtest(
                    k=min(
                        selected_yes_control_no,
                        selected_no_control_yes,
                    ),
                    n=n_discordant,
                    p=0.5,
                    alternative="two-sided",
                ).pvalue
            )

        selected_prop = (
            np.mean(selected_valid)
            if len(selected_valid)
            else np.nan
        )

        control_prop = (
            np.mean(control_valid)
            if len(control_valid)
            else np.nan
        )

        paired_geometry_rows.append(
            {
                "regime": regime_name,
                "outcome": outcome_label,
                "outcome_type": "binary",
                "n_valid_pairs": (
                    len(selected_valid)
                ),
                "selected_mean": np.nan,
                "control_mean": np.nan,
                "mean_paired_difference": np.nan,
                "median_paired_difference": np.nan,
                "paired_smd": np.nan,
                "selected_prop": selected_prop,
                "control_prop": control_prop,
                "risk_difference": (
                    selected_prop
                    -
                    control_prop
                ),
                "selected_yes_control_no": (
                    selected_yes_control_no
                ),
                "selected_no_control_yes": (
                    selected_no_control_yes
                ),
                "p_value": p_value,
                "test": (
                    "exact paired McNemar"
                ),
            }
        )

# ------------------------------------------------------------
# 8. FDR correction
# ------------------------------------------------------------

paired_geometry_results = (
    pd.DataFrame(
        paired_geometry_rows
    )
)

paired_geometry_results[
    "q_value"
] = benjamini_hochberg(
    paired_geometry_results[
        "p_value"
    ]
)

paired_geometry_results[
    "fdr_significant"
] = (
    paired_geometry_results[
        "q_value"
    ]
    <
    0.05
)

paired_geometry_results = (
    paired_geometry_results
    .sort_values(
        [
            "regime",
            "q_value",
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    paired_geometry_results
)

# ------------------------------------------------------------
# 9. Update final inference decisions
# ------------------------------------------------------------

final_geometry_decisions = pd.DataFrame(
    [
        {
            "regime": "conditional_only",
            "status": "primary_matched_inference",
            "coverage": 357 / 367,
            "interpretation": (
                "Excellent unique-control support "
                "and norm balance."
            ),
        },
        {
            "regime": "amplitude_only",
            "status": "matched_subset_sensitivity",
            "coverage": 109 / 246,
            "interpretation": (
                "Valid for common-support subset; "
                "not generalizable to full regime."
            ),
        },
        {
            "regime": "shared",
            "status": "descriptive_only",
            "coverage": np.nan,
            "interpretation": (
                "Insufficient common support "
                "after amplitude adjustment."
            ),
        },
        {
            "regime": "universal_core",
            "status": "case_study",
            "coverage": np.nan,
            "interpretation": (
                "Six highly robust examples; "
                "not a population-level test."
            ),
        },
    ]
)

display(
    final_geometry_decisions
)

# ------------------------------------------------------------
# 10. Save final experimental outputs
# ------------------------------------------------------------

paired_geometry_results.to_csv(
    GEOMETRY_MATCH_DIR /
    "final_paired_geometry_results.csv",
    index=False,
)

final_geometry_decisions.to_csv(
    GEOMETRY_MATCH_DIR /
    "final_geometry_inference_decisions.csv",
    index=False,
)

nonpositive_controls.to_csv(
    GEOMETRY_MATCH_DIR /
    "nonpositive_control_triangulations.csv",
    index=False,
)

print()
print("=" * 80)
print("EXPERIMENTAL ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 17 — FREEZE FINAL PAPER RUN
# ============================================================

from pathlib import Path
import json
import sys
import platform
import importlib.metadata

FINAL_PAPER_DIR = ch.stage_directory(
    OUTPUT_DIR,
    17,
    "final_paper_outputs",
)

RUN_ROOT = FINAL_CANDIDATES_DIR.parent

# ------------------------------------------------------------
# 1. Central quantitative claims
# ------------------------------------------------------------

central_results = pd.DataFrame(
    [
        {
            "result": "Raw PCA strict consensus",
            "value": 292,
            "interpretation": (
                "Top 1% raw SSE in all five representations."
            ),
        },
        {
            "result": "Conditional >=3/5 regime",
            "value": 413,
            "interpretation": (
                "Norm-conditioned multiview hard regime."
            ),
        },
        {
            "result": "Raw and conditional overlap",
            "value": 46,
            "interpretation": (
                "Small extreme intersection of both regimes."
            ),
        },
        {
            "result": "PCA + AE + conditional universal core",
            "value": 6,
            "interpretation": (
                "Highly robust case-study knots."
            ),
        },
        {
            "result": "Held-out PCA consensus",
            "value": 39,
            "interpretation": (
                "Exact replication of full-data consensus "
                "restricted to the frozen test set."
            ),
        },
        {
            "result": "AE majority consensus >=3/5",
            "value": 25,
            "interpretation": (
                "Held-out nonlinear reconstruction regime."
            ),
        },
        {
            "result": "All-5 conditional nonalternating s>sigma",
            "value": 0.769231,
            "interpretation": (
                "Exact-signature null mean 0.350; "
                "focused-null q=0.000236."
            ),
        },
        {
            "result": "No-Khovanov conditional nonalternating s>sigma",
            "value": 0.868263,
            "interpretation": (
                "Exact-signature null mean 0.343; "
                "focused-null q=0.000236."
            ),
        },
        {
            "result": "All-5 fixed-effect partial Spearman",
            "value": 0.057234,
            "interpretation": (
                "Weak continuous bulk association; "
                "effect concentrated in extreme tail."
            ),
        },
        {
            "result": "Conditional matched volume difference",
            "value": -0.778665,
            "interpretation": (
                "357 unique pairs; q approximately 5e-6."
            ),
        },
        {
            "result": "Conditional matched symmetry-order difference",
            "value": 0.188732,
            "interpretation": (
                "Modestly greater symmetry; q=0.0153."
            ),
        },
        {
            "result": "Amplitude matched volume difference",
            "value": 0.149246,
            "interpretation": (
                "No evidence of an independent difference; "
                "matched-subset sensitivity only."
            ),
        },
    ]
)

central_results.to_csv(
    FINAL_PAPER_DIR /
    "central_results.csv",
    index=False,
)

display(central_results)

# ------------------------------------------------------------
# 2. Freeze important in-memory result tables
# ------------------------------------------------------------

tables_to_freeze = {
    "hard_regime_summary":
        atlas_regime_summary,

    "fixed_effect_continuous_association":
        ck_fixed_effect_association,

    "universal_core":
        universal_core_export,

    "universal_core_snappy":
        universal_core_snappy,

    "snappy_regime_summary":
        snappy_regime_summary_with_core,

    "unique_geometry_matching_design":
        unique_geometry_design,

    "final_paired_geometry_results":
        paired_geometry_results,

    "final_geometry_inference_decisions":
        final_geometry_decisions,

    "clean_geometry_matching_sensitivity":
        clean_geometry_matching_sensitivity,
}

for table_name, table in (
    tables_to_freeze.items()
):

    if not isinstance(
        table,
        pd.DataFrame,
    ):
        raise TypeError(
            f"{table_name} is not a DataFrame."
        )

    table.to_csv(
        FINAL_PAPER_DIR /
        f"{table_name}.csv",
        index=False,
    )

# Large final atlas
atlas.to_parquet(
    FINAL_PAPER_DIR /
    "final_hard_regime_atlas.parquet",
    index=False,
)

candidate_geometry.to_parquet(
    FINAL_PAPER_DIR /
    "final_candidate_geometry.parquet",
    index=False,
)

# ------------------------------------------------------------
# 3. Record methodological decisions
# ------------------------------------------------------------

methodological_decisions = pd.DataFrame(
    [
        {
            "topic": "Primary absolute hardness",
            "decision": (
                "Raw PCA SSE, EVR 99%, top 1%, "
                "strict five-view intersection."
            ),
            "status": "primary descriptive regime",
        },
        {
            "topic": "Structural hardness",
            "decision": (
                "Conditional-percentile scores with "
                "100 norm bins; membership >=3/5."
            ),
            "status": "primary conditional regime",
        },
        {
            "topic": "Conditional inference",
            "decision": (
                "Exact crossing, alternation and signature "
                "stratified null; 5000 focused repetitions."
            ),
            "status": "confirmatory",
        },
        {
            "topic": "Continuous sensitivity",
            "decision": (
                "Fixed-effect partial rank association "
                "and extreme-tail matched effects."
            ),
            "status": "supporting",
        },
        {
            "topic": "Held-out PCA",
            "decision": (
                "Frozen 70/15/15 split; PCA fit on train only."
            ),
            "status": "confirmatory",
        },
        {
            "topic": "Held-out autoencoder",
            "decision": (
                "Five seeds; majority >=3/5 and strict core."
            ),
            "status": "model-class sensitivity",
        },
        {
            "topic": "Conditional-only geometry",
            "decision": (
                "Unique one-to-one controls, exact structural "
                "matching and norm caliper 0.50."
            ),
            "status": "primary matched inference",
        },
        {
            "topic": "Amplitude-only geometry",
            "decision": (
                "Unique one-to-one matched subset; "
                "44.3% common-support coverage."
            ),
            "status": "limited sensitivity",
        },
        {
            "topic": "Shared geometry",
            "decision": (
                "No adequate common support after "
                "norm adjustment."
            ),
            "status": "descriptive only",
        },
        {
            "topic": "Universal six-knot core",
            "decision": (
                "Raw PCA + held-out PCA + AE majority "
                "+ conditional membership."
            ),
            "status": "case study",
        },
        {
            "topic": "SnapPy tetrahedra",
            "decision": (
                "Number in census triangulation is not "
                "treated as a minimal triangulation invariant."
            ),
            "status": "supplementary only",
        },
    ]
)

methodological_decisions.to_csv(
    FINAL_PAPER_DIR /
    "methodological_decisions.csv",
    index=False,
)

# ------------------------------------------------------------
# 4. Software versions
# ------------------------------------------------------------

def package_version(
    package_name,
):

    try:
        return importlib.metadata.version(
            package_name
        )
    except Exception:
        return "not available"


software_versions = pd.DataFrame(
    [
        {
            "software": "Python",
            "version": sys.version.split()[0],
        },
        {
            "software": "Platform",
            "version": platform.platform(),
        },
        {
            "software": "numpy",
            "version": package_version("numpy"),
        },
        {
            "software": "pandas",
            "version": package_version("pandas"),
        },
        {
            "software": "scipy",
            "version": package_version("scipy"),
        },
        {
            "software": "scikit-learn",
            "version": package_version(
                "scikit-learn"
            ),
        },
        {
            "software": "TensorFlow",
            "version": package_version(
                "tensorflow"
            ),
        },
        {
            "software": "SnapPy",
            "version": package_version(
                "snappy"
            ),
        },
        {
            "software": "snappy_15_knots",
            "version": package_version(
                "snappy_15_knots"
            ),
        },
        {
            "software": "consensus_hardness",
            "version": getattr(
                ch,
                "__version__",
                "local source version",
            ),
        },
    ]
)

software_versions.to_csv(
    FINAL_PAPER_DIR /
    "software_versions.csv",
    index=False,
)

display(software_versions)

# ------------------------------------------------------------
# 5. Machine-readable final manifest
# ------------------------------------------------------------

final_manifest = {
    "run_name": "corrected_run_20260819",
    "analysis_status": (
        "experimental analysis complete"
    ),
    "universe": {
        "n_knots": 313230,
        "crossing_min": 3,
        "crossing_max": 15,
        "identity_excluded": True,
        "duplicate_ids": 0,
        "s_qc_corrections": 1,
    },
    "seeds": {
        "primary": 42,
        "autoencoders": [0, 1, 2, 3, 4],
    },
    "primary_configuration": {
        "pca_evr": 0.99,
        "tail_mass": 0.01,
        "raw_consensus_requirement": "5 of 5",
        "conditional_norm_bins": 100,
        "conditional_membership_requirement": (
            "at least 3 of 5"
        ),
    },
    "principal_set_sizes": {
        "raw_absolute": 292,
        "conditional_3_of_5": 413,
        "shared": 46,
        "amplitude_only": 246,
        "conditional_only": 367,
        "heldout_pca": 39,
        "ae_majority": 25,
        "ae_strict_core": 13,
        "universal_pca_ae_conditional": 6,
    },
    "geometry_inference": {
        "conditional_unique_pairs": 357,
        "conditional_coverage": (
            357 / 367
        ),
        "amplitude_unique_pairs": 109,
        "amplitude_coverage": (
            109 / 246
        ),
        "shared_status": (
            "descriptive only; "
            "insufficient common support"
        ),
    },
}

with open(
    FINAL_PAPER_DIR /
    "final_run_manifest.json",
    "w",
    encoding="utf-8",
) as manifest_file:

    json.dump(
        final_manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
    )

# ------------------------------------------------------------
# 6. Create result-file registry
# ------------------------------------------------------------

registry_rows = []

for path in sorted(
    RUN_ROOT.rglob("*")
):

    if (
        path.is_file()
        and
        path.suffix.lower()
        in {
            ".csv",
            ".parquet",
            ".json",
            ".md",
        }
    ):

        registry_rows.append(
            {
                "relative_path": str(
                    path.relative_to(
                        RUN_ROOT
                    )
                ),
                "suffix": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
            }
        )

result_registry = pd.DataFrame(
    registry_rows
)

result_registry.to_csv(
    FINAL_PAPER_DIR /
    "result_file_registry.csv",
    index=False,
)

print(
    "Registered result files:",
    len(result_registry),
)

# ------------------------------------------------------------
# 7. Human-readable README
# ------------------------------------------------------------

final_readme = """# Final consensus-hardness paper run

Status: experimental analysis complete.

## Main conclusion

Multiview reconstruction hardness separates into two partially
independent regimes.

1. Raw SSE hardness predominantly captures large invariant amplitude,
15-crossing knots, and alternating structure.

2. Norm-conditioned hardness identifies a predominantly nonalternating
regime with an anomalously large Rasmussen-signature gap.

The nonalternating gap survives exact matching on crossing number,
alternation, and signature. It is reproduced across PCA and
autoencoder reconstruction and under held-out evaluation.

After structural and norm matching, conditional-hard knots have
slightly lower hyperbolic volume and modestly greater symmetry.
Therefore, conditional reconstruction hardness is not equivalent
to greater hyperbolic geometric complexity.

## Inferential hierarchy

- Confirmatory:
  held-out PCA, exact conditional nulls, conditional-only geometry.

- Supporting:
  held-out autoencoders, continuous tail effects, bin sensitivity.

- Descriptive:
  raw amplitude regime and shared-regime geometry.

- Case studies:
  six-knot PCA + AE + conditional universal core.

## Important limitations

- Raw SSE is strongly affected by invariant amplitude.
- Strict conditional intersections are small and bin-sensitive.
- AE membership varies across random seeds.
- Shared hard knots lack sufficient matched common support.
- SnapPy hyperbolicity was numerical rather than interval-certified
  in the Colab environment.
- Census triangulation size is not assumed minimal.
"""

(
    FINAL_PAPER_DIR /
    "README_FINAL_RUN.md"
).write_text(
    final_readme,
    encoding="utf-8",
)

print()
print("=" * 80)
print("FINAL RUN FROZEN")
print("=" * 80)
print(FINAL_PAPER_DIR)

# Figures

In [ ]:
# ============================================================
# STEP 18A — FIGURE 1: TWO HARDNESS REGIMES
# ============================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

FIGURE_DIR = ch.stage_directory(
    OUTPUT_DIR,
    18,
    "paper_figures",
)

FIGURE_SOURCE_DIR = (
    FIGURE_DIR /
    "source_data"
)

FIGURE_SOURCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 1. Publication style
# ------------------------------------------------------------

mpl.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 400,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

regime_order = [
    "neither",
    "amplitude_hard_only",
    "conditional_hard_only",
    "shared_raw_and_conditional",
]

regime_labels = {
    "neither": "Background",
    "amplitude_hard_only": "Amplitude\nonly",
    "conditional_hard_only": "Conditional\nonly",
    "shared_raw_and_conditional": "Shared",
}

regime_colors = {
    "neither": "#B8BDC6",
    "amplitude_hard_only": "#D95F02",
    "conditional_hard_only": "#1B9E77",
    "shared_raw_and_conditional": "#7570B3",
}

# ------------------------------------------------------------
# 2. Build figure source summary
# ------------------------------------------------------------

figure1_rows = []

for regime in regime_order:

    group = atlas.loc[
        atlas["hard_regime"] == regime
    ].copy()

    nonalternating = group.loc[
        ~group["is_alternating"].astype(bool)
    ]

    figure1_rows.append(
        {
            "hard_regime": regime,
            "label": regime_labels[regime],
            "n": len(group),
            "prop_crossing_15": (
                group[
                    "number_of_crossings"
                ]
                .eq(15)
                .mean()
            ),
            "prop_alternating": (
                group[
                    "is_alternating"
                ]
                .astype(bool)
                .mean()
            ),
            "median_signature": (
                group[
                    "signature"
                ].median()
            ),
            "median_s": (
                group[
                    "s_invariant_qc"
                ].median()
            ),
            "median_mean_log_norm": (
                group[
                    "mean_log_sq_norm"
                ].median()
            ),
            "nonalternating_n": (
                len(nonalternating)
            ),
            "nonalt_delta_positive_prop": (
                nonalternating[
                    "s_minus_sigma_qc"
                ]
                .gt(0)
                .mean()
                if len(nonalternating)
                else np.nan
            ),
            "nonalt_delta_ge_4_prop": (
                nonalternating[
                    "s_minus_sigma_qc"
                ]
                .ge(4)
                .mean()
                if len(nonalternating)
                else np.nan
            ),
            "nonalt_mean_delta": (
                nonalternating[
                    "s_minus_sigma_qc"
                ].mean()
                if len(nonalternating)
                else np.nan
            ),
        }
    )

figure1_source = pd.DataFrame(
    figure1_rows
)

figure1_source.to_csv(
    FIGURE_SOURCE_DIR /
    "figure1_two_regimes_summary.csv",
    index=False,
)

display(figure1_source)

# ------------------------------------------------------------
# 3. Build Figure 1
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11.5, 8.0),
)

ax_a, ax_b, ax_c, ax_d = (
    axes.flatten()
)

# ------------------------------------------------------------
# Panel A — Hard-regime census
# ------------------------------------------------------------

hard_only_order = [
    "amplitude_hard_only",
    "shared_raw_and_conditional",
    "conditional_hard_only",
]

hard_counts = [
    int(
        figure1_source.loc[
            figure1_source[
                "hard_regime"
            ].eq(regime),
            "n",
        ].iloc[0]
    )
    for regime in hard_only_order
]

bars = ax_a.bar(
    np.arange(3),
    hard_counts,
    color=[
        regime_colors[regime]
        for regime
        in hard_only_order
    ],
    width=0.67,
)

ax_a.set_xticks(
    np.arange(3)
)

ax_a.set_xticklabels(
    [
        "Amplitude only",
        "Shared",
        "Conditional only",
    ]
)

ax_a.set_ylabel("Number of knots")

ax_a.set_title(
    "A   Two partially distinct hard regimes",
    loc="left",
    fontweight="bold",
)

for bar, value in zip(
    bars,
    hard_counts,
):

    ax_a.text(
        bar.get_x()
        +
        bar.get_width() / 2,
        bar.get_height() + 8,
        f"{value}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax_a.text(
    0.02,
    0.94,
    "Raw total = 292\nConditional total = 413\nOverlap = 46",
    transform=ax_a.transAxes,
    ha="left",
    va="top",
    fontsize=9,
)

ax_a.set_ylim(
    0,
    max(hard_counts) * 1.22,
)

# ------------------------------------------------------------
# Panel B — Norm/amplitude distributions
# ------------------------------------------------------------

box_data = [
    atlas.loc[
        atlas["hard_regime"].eq(regime),
        "mean_log_sq_norm",
    ]
    .dropna()
    .to_numpy()
    for regime
    in regime_order
]

boxplot = ax_b.boxplot(
    box_data,
    labels=[
        regime_labels[regime]
        for regime
        in regime_order
    ],
    patch_artist=True,
    showfliers=False,
    whis=(5, 95),
    widths=0.62,
    medianprops={
        "color": "black",
        "linewidth": 1.3,
    },
)

for patch, regime in zip(
    boxplot["boxes"],
    regime_order,
):

    patch.set_facecolor(
        regime_colors[regime]
    )

    patch.set_alpha(0.82)

ax_b.set_ylabel(
    "Mean log squared norm"
)

ax_b.set_title(
    "B   Raw hardness is strongly amplitude-associated",
    loc="left",
    fontweight="bold",
)

# ------------------------------------------------------------
# Panel C — Structural composition
# ------------------------------------------------------------

x = np.arange(
    len(regime_order)
)

width = 0.34

alternating_values = (
    figure1_source
    .set_index("hard_regime")
    .loc[
        regime_order,
        "prop_alternating",
    ]
    .to_numpy()
)

crossing15_values = (
    figure1_source
    .set_index("hard_regime")
    .loc[
        regime_order,
        "prop_crossing_15",
    ]
    .to_numpy()
)

ax_c.bar(
    x - width / 2,
    alternating_values,
    width,
    label="Alternating",
    color="#4C78A8",
)

ax_c.bar(
    x + width / 2,
    crossing15_values,
    width,
    label="15 crossings",
    color="#F2B134",
)

ax_c.set_xticks(x)

ax_c.set_xticklabels(
    [
        regime_labels[regime]
        for regime
        in regime_order
    ]
)

ax_c.set_ylim(0, 1.08)
ax_c.set_ylabel("Proportion")
ax_c.legend(
    frameon=False,
    loc="upper left",
)

ax_c.set_title(
    "C   Structural composition differs sharply",
    loc="left",
    fontweight="bold",
)

# ------------------------------------------------------------
# Panel D — Nonalternating Rasmussen–signature gap
# ------------------------------------------------------------

delta_positive = (
    figure1_source
    .set_index("hard_regime")
    .loc[
        regime_order,
        "nonalt_delta_positive_prop",
    ]
    .to_numpy()
)

delta_ge_4 = (
    figure1_source
    .set_index("hard_regime")
    .loc[
        regime_order,
        "nonalt_delta_ge_4_prop",
    ]
    .to_numpy()
)

ax_d.bar(
    x - width / 2,
    delta_positive,
    width,
    label=r"$s-\sigma>0$",
    color="#2A9D8F",
)

ax_d.bar(
    x + width / 2,
    delta_ge_4,
    width,
    label=r"$s-\sigma\geq4$",
    color="#8E5EA2",
)

ax_d.set_xticks(x)

ax_d.set_xticklabels(
    [
        regime_labels[regime]
        for regime
        in regime_order
    ]
)

ax_d.set_ylim(0, 1.08)
ax_d.set_ylabel(
    "Proportion among nonalternating knots"
)

ax_d.legend(
    frameon=False,
    loc="upper left",
)

ax_d.set_title(
    "D   Conditional hardness isolates the concordance gap",
    loc="left",
    fontweight="bold",
)

# ------------------------------------------------------------
# 4. Final layout and save
# ------------------------------------------------------------

fig.suptitle(
    "Absolute and norm-conditioned multiview hardness identify distinct knot regimes",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)

fig.tight_layout(
    h_pad=2.3,
    w_pad=1.7,
)

figure1_png = (
    FIGURE_DIR /
    "figure1_two_hardness_regimes.png"
)

figure1_pdf = (
    FIGURE_DIR /
    "figure1_two_hardness_regimes.pdf"
)

fig.savefig(
    figure1_png,
    bbox_inches="tight",
)

fig.savefig(
    figure1_pdf,
    bbox_inches="tight",
)

plt.show()

print("Saved:")
print(figure1_png)
print(figure1_pdf)

## correcta imagen

In [ ]:
# More neutral and statistically faithful Panel D title
ax_d.set_title(
    "D   Nonalternating concordance gaps concentrate in hard regimes",
    loc="left",
    fontweight="bold",
)

ax_d.set_ylim(0, 1.18)

nonalternating_counts = (
    figure1_source
    .set_index("hard_regime")
    .loc[
        regime_order,
        "nonalternating_n",
    ]
    .astype(int)
    .to_numpy()
)

for position, (
    n_nonalt,
    positive_prop,
    large_gap_prop,
) in enumerate(
    zip(
        nonalternating_counts,
        delta_positive,
        delta_ge_4,
    )
):

    height = max(
        positive_prop,
        large_gap_prop,
    )

    ax_d.text(
        position,
        height + 0.035,
        f"non-alt n={n_nonalt:,}",
        ha="center",
        va="bottom",
        fontsize=8,
        color="#444444",
    )

ax_d.legend(
    frameon=False,
    loc="center left",
    bbox_to_anchor=(0.01, 0.70),
)

fig.savefig(
    figure1_png,
    bbox_inches="tight",
)

fig.savefig(
    figure1_pdf,
    bbox_inches="tight",
)

display(fig)

In [ ]:
# ============================================================
# STEP 18B — FIGURE 2: EXACT CONDITIONAL NULLS
# ============================================================

from matplotlib.lines import Line2D

# ------------------------------------------------------------
# 1. Frozen source data from the 5000-repetition focused null
# ------------------------------------------------------------

focused_gap_null_summary = pd.DataFrame(
    [
        # All five views
        {
            "family": "All 5",
            "metric": "nonalt_s_gt_sigma_prop",
            "observed": 0.769231,
            "null_mean": 0.350172,
            "null_q95": 0.417476,
            "null_q99": 0.447368,
            "n_ge_observed": 0,
            "n_valid_null": 5000,
            "n_missing_null": 0,
            "empirical_p": 0.000200,
            "q_value": 0.000236,
        },
        {
            "family": "All 5",
            "metric": "mean_s_minus_sigma",
            "observed": 1.733728,
            "null_mean": 0.747633,
            "null_q95": 0.888889,
            "null_q99": 0.948469,
            "n_ge_observed": 0,
            "n_valid_null": 5000,
            "n_missing_null": 0,
            "empirical_p": 0.000200,
            "q_value": 0.000236,
        },
        {
            "family": "All 5",
            "metric": "delta_ge_4_prop",
            "observed": 0.097633,
            "null_mean": 0.024309,
            "null_q95": 0.038462,
            "null_q99": 0.045455,
            "n_ge_observed": 0,
            "n_valid_null": 5000,
            "n_missing_null": 0,
            "empirical_p": 0.000200,
            "q_value": 0.000236,
        },

        # Without Khovanov
        {
            "family": "No Khovanov",
            "metric": "nonalt_s_gt_sigma_prop",
            "observed": 0.868263,
            "null_mean": 0.342729,
            "null_q95": 0.436364,
            "null_q99": 0.480769,
            "n_ge_observed": 0,
            "n_valid_null": 5000,
            "n_missing_null": 0,
            "empirical_p": 0.000200,
            "q_value": 0.000236,
        },
        {
            "family": "No Khovanov",
            "metric": "mean_s_minus_sigma",
            "observed": 2.059880,
            "null_mean": 0.742885,
            "null_q95": 0.941176,
            "null_q99": 1.041685,
            "n_ge_observed": 0,
            "n_valid_null": 5000,
            "n_missing_null": 0,
            "empirical_p": 0.000200,
            "q_value": 0.000236,
        },
        {
            "family": "No Khovanov",
            "metric": "delta_ge_4_prop",
            "observed": 0.161677,
            "null_mean": 0.028983,
            "null_q95": 0.051770,
            "null_q99": 0.065217,
            "n_ge_observed": 0,
            "n_valid_null": 5000,
            "n_missing_null": 0,
            "empirical_p": 0.000200,
            "q_value": 0.000236,
        },

        # Polynomial-only
        {
            "family": "Polynomial only",
            "metric": "nonalt_s_gt_sigma_prop",
            "observed": 1.000000,
            "null_mean": 0.111260,
            "null_q95": 0.500000,
            "null_q99": 1.000000,
            "n_ge_observed": 93,
            "n_valid_null": 4768,
            "n_missing_null": 232,
            "empirical_p": 0.019711,
            "q_value": 0.019711,
        },
        {
            "family": "Polynomial only",
            "metric": "mean_s_minus_sigma",
            "observed": 2.750000,
            "null_mean": 0.222437,
            "null_q95": 1.000000,
            "null_q99": 2.000000,
            "n_ge_observed": 0,
            "n_valid_null": 4768,
            "n_missing_null": 232,
            "empirical_p": 0.000210,
            "q_value": 0.000236,
        },
        {
            "family": "Polynomial only",
            "metric": "delta_ge_4_prop",
            "observed": 0.375000,
            "null_mean": 0.000105,
            "null_q95": 0.000000,
            "null_q99": 0.000000,
            "n_ge_observed": 0,
            "n_valid_null": 4768,
            "n_missing_null": 232,
            "empirical_p": 0.000210,
            "q_value": 0.000236,
        },
    ]
)

focused_gap_null_summary.to_csv(
    FIGURE_SOURCE_DIR /
    "figure2_exact_conditional_nulls.csv",
    index=False,
)

# ------------------------------------------------------------
# 2. Plot definitions
# ------------------------------------------------------------

family_order = [
    "All 5",
    "No Khovanov",
    "Polynomial only",
]

family_labels = {
    "All 5": "All five\nviews",
    "No Khovanov": "Without\nKhovanov",
    "Polynomial only": "Polynomial\nonly",
}

family_colors = {
    "All 5": "#1B9E77",
    "No Khovanov": "#4C78A8",
    "Polynomial only": "#D95F02",
}

metric_specs = [
    {
        "metric": "nonalt_s_gt_sigma_prop",
        "title": r"A   Positive gap: $s-\sigma>0$",
        "ylabel": (
            "Proportion among\n"
            "nonalternating knots"
        ),
    },
    {
        "metric": "mean_s_minus_sigma",
        "title": r"B   Mean concordance gap",
        "ylabel": r"Mean $s-\sigma$",
    },
    {
        "metric": "delta_ge_4_prop",
        "title": r"C   Large gap: $s-\sigma\geq4$",
        "ylabel": (
            "Proportion among\n"
            "nonalternating knots"
        ),
    },
]

# ------------------------------------------------------------
# 3. Draw exact-null comparison
# ------------------------------------------------------------

fig2, axes2 = plt.subplots(
    1,
    3,
    figsize=(12.2, 4.2),
)

for axis, spec in zip(
    axes2,
    metric_specs,
):

    metric_df = (
        focused_gap_null_summary.loc[
            focused_gap_null_summary[
                "metric"
            ].eq(spec["metric"])
        ]
        .set_index("family")
        .loc[family_order]
        .reset_index()
    )

    x = np.arange(
        len(family_order)
    )

    null_x = x - 0.08
    observed_x = x + 0.08

    # Null 95th–99th percentile segment
    for position, row in zip(
        null_x,
        metric_df.itertuples(
            index=False
        ),
    ):

        axis.plot(
            [
                position,
                position,
            ],
            [
                row.null_q95,
                row.null_q99,
            ],
            color="#777777",
            linewidth=5,
            solid_capstyle="round",
            alpha=0.75,
            zorder=2,
        )

    # Null means
    axis.scatter(
        null_x,
        metric_df["null_mean"],
        marker="o",
        s=45,
        facecolor="white",
        edgecolor="#333333",
        linewidth=1.3,
        zorder=3,
    )

    # Observed values
    for position, family, observed in zip(
        observed_x,
        family_order,
        metric_df["observed"],
    ):

        axis.scatter(
            position,
            observed,
            marker="D",
            s=68,
            color=family_colors[family],
            edgecolor="white",
            linewidth=0.8,
            zorder=4,
        )

    maximum_value = max(
        metric_df["observed"].max(),
        metric_df["null_q99"].max(),
    )

    annotation_offset = (
        0.055 * maximum_value
        if maximum_value > 0
        else 0.03
    )

    for position, row in zip(
        observed_x,
        metric_df.itertuples(
            index=False
        ),
    ):

        q_text = (
            f"q={row.q_value:.1e}"
            if row.q_value < 0.001
            else
            f"q={row.q_value:.3f}"
        )

        axis.text(
            position,
            row.observed
            +
            annotation_offset,
            q_text,
            ha="center",
            va="bottom",
            fontsize=8,
            color="#333333",
        )

    axis.set_xticks(x)

    axis.set_xticklabels(
        [
            family_labels[family]
            for family
            in family_order
        ]
    )

    axis.set_ylabel(
        spec["ylabel"]
    )

    axis.set_title(
        spec["title"],
        loc="left",
        fontweight="bold",
    )

    axis.set_ylim(
        0,
        maximum_value * 1.22,
    )

    axis.grid(
        axis="y",
        color="#DDDDDD",
        linewidth=0.7,
        alpha=0.7,
    )

# ------------------------------------------------------------
# 4. Shared legend and explanatory title
# ------------------------------------------------------------

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor="white",
        markeredgecolor="#333333",
        markersize=7,
        label="Null mean",
    ),
    Line2D(
        [0],
        [0],
        color="#777777",
        linewidth=5,
        alpha=0.75,
        label="Null 95th–99th percentiles",
    ),
    Line2D(
        [0],
        [0],
        marker="D",
        color="none",
        markerfacecolor="#1B9E77",
        markeredgecolor="white",
        markersize=8,
        label="Observed",
    ),
]

fig2.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.03),
    ncol=3,
    frameon=False,
)

fig2.suptitle(
    "Exact stratified null models confirm an exceptional nonalternating Rasmussen–signature gap",
    fontsize=13,
    fontweight="bold",
    y=1.14,
)

fig2.text(
    0.5,
    -0.02,
    (
        "Null permutations preserve norm bin, crossing number, "
        "alternation status, and exact signature; 5,000 repetitions."
    ),
    ha="center",
    va="top",
    fontsize=9,
    color="#444444",
)

fig2.tight_layout(
    w_pad=2.0,
)

# ------------------------------------------------------------
# 5. Save
# ------------------------------------------------------------

figure2_png = (
    FIGURE_DIR /
    "figure2_exact_conditional_nulls.png"
)

figure2_pdf = (
    FIGURE_DIR /
    "figure2_exact_conditional_nulls.pdf"
)

fig2.savefig(
    figure2_png,
    bbox_inches="tight",
)

fig2.savefig(
    figure2_pdf,
    bbox_inches="tight",
)

plt.show()

print("Saved:")
print(figure2_png)
print(figure2_pdf)

In [ ]:
# ============================================================
# STEP 18C — FIGURE 3: HELD-OUT AND MODEL-CLASS ROBUSTNESS
# ============================================================

# ------------------------------------------------------------
# 1. Frozen source data
# ------------------------------------------------------------

ae_seed_sizes = pd.DataFrame(
    {
        "seed": [0, 1, 2, 3, 4],
        "consensus_size": [28, 38, 41, 16, 20],
        "pca_overlap": [19, 19, 26, 9, 10],
    }
)

ae_seed_sizes[
    "fraction_ae_in_pca"
] = (
    ae_seed_sizes[
        "pca_overlap"
    ]
    /
    ae_seed_sizes[
        "consensus_size"
    ]
)

ae_jaccard_values = np.array(
    [
        [
            1.000000,
            0.650000,
            0.533333,
            0.517241,
            0.454545,
        ],
        [
            0.650000,
            1.000000,
            0.580000,
            0.421053,
            0.380952,
        ],
        [
            0.533333,
            0.580000,
            1.000000,
            0.357143,
            0.355556,
        ],
        [
            0.517241,
            0.421053,
            0.357143,
            1.000000,
            0.565217,
        ],
        [
            0.454545,
            0.380952,
            0.355556,
            0.565217,
            1.000000,
        ],
    ]
)

ae_membership_tiers = pd.DataFrame(
    {
        "minimum_seed_membership": [
            "≥1/5",
            "≥2/5",
            "≥3/5",
            "≥4/5",
            "5/5",
        ],
        "n_knots": [
            56,
            32,
            25,
            17,
            13,
        ],
    }
)

pca_ae_overlap_summary = pd.DataFrame(
    {
        "analysis": [
            "Seed 0",
            "Seed 1",
            "Seed 2",
            "Seed 3",
            "Seed 4",
            "AE ≥3/5",
            "AE ≥4/5",
            "AE 5/5",
        ],
        "ae_size": [
            28,
            38,
            41,
            16,
            20,
            25,
            17,
            13,
        ],
        "overlap_with_pca": [
            19,
            19,
            26,
            9,
            10,
            18,
            11,
            8,
        ],
    }
)

pca_ae_overlap_summary[
    "fraction_ae_in_pca"
] = (
    pca_ae_overlap_summary[
        "overlap_with_pca"
    ]
    /
    pca_ae_overlap_summary[
        "ae_size"
    ]
)

# Long-format Jaccard source data
jaccard_source_rows = []

for seed_a in range(5):
    for seed_b in range(5):

        jaccard_source_rows.append(
            {
                "seed_a": seed_a,
                "seed_b": seed_b,
                "jaccard": (
                    ae_jaccard_values[
                        seed_a,
                        seed_b,
                    ]
                ),
            }
        )

ae_jaccard_source = pd.DataFrame(
    jaccard_source_rows
)

# Save source data
ae_seed_sizes.to_csv(
    FIGURE_SOURCE_DIR /
    "figure3_ae_seed_sizes.csv",
    index=False,
)

ae_membership_tiers.to_csv(
    FIGURE_SOURCE_DIR /
    "figure3_ae_membership_tiers.csv",
    index=False,
)

pca_ae_overlap_summary.to_csv(
    FIGURE_SOURCE_DIR /
    "figure3_pca_ae_overlap.csv",
    index=False,
)

ae_jaccard_source.to_csv(
    FIGURE_SOURCE_DIR /
    "figure3_ae_jaccard_matrix.csv",
    index=False,
)

# ------------------------------------------------------------
# 2. Figure canvas
# ------------------------------------------------------------

fig3, axes3 = plt.subplots(
    2,
    2,
    figsize=(11.5, 8.2),
)

ax_a, ax_b, ax_c, ax_d = (
    axes3.flatten()
)

# ------------------------------------------------------------
# Panel A — Exact held-out PCA replication
# ------------------------------------------------------------

pca_labels = [
    "Full-data PCA\nrestricted to test",
    "Train-fitted\nheld-out PCA",
]

pca_sizes = [
    39,
    39,
]

pca_bars = ax_a.bar(
    [0, 1],
    pca_sizes,
    color=[
        "#4C78A8",
        "#1B9E77",
    ],
    width=0.58,
)

ax_a.set_xticks(
    [0, 1]
)

ax_a.set_xticklabels(
    pca_labels
)

ax_a.set_ylabel(
    "Consensus size"
)

ax_a.set_ylim(
    0,
    52,
)

for bar, value in zip(
    pca_bars,
    pca_sizes,
):

    ax_a.text(
        bar.get_x()
        +
        bar.get_width() / 2,
        value + 1.0,
        str(value),
        ha="center",
        va="bottom",
        fontweight="bold",
    )

# Overlap bracket
ax_a.plot(
    [0, 0, 1, 1],
    [45, 47, 47, 45],
    color="#333333",
    linewidth=1.1,
)

ax_a.text(
    0.5,
    48.2,
    "Overlap = 39 · Jaccard = 1.00",
    ha="center",
    va="bottom",
    fontsize=9,
)

ax_a.set_title(
    "A   Target-free PCA reproduces the test consensus exactly",
    loc="left",
    fontweight="bold",
)

# ------------------------------------------------------------
# Panel B — AE seed Jaccard matrix
# ------------------------------------------------------------

heatmap = ax_b.imshow(
    ae_jaccard_values,
    cmap="Blues",
    vmin=0.30,
    vmax=1.00,
    aspect="equal",
)

ax_b.set_xticks(
    np.arange(5)
)

ax_b.set_yticks(
    np.arange(5)
)

ax_b.set_xticklabels(
    [
        f"Seed {seed}"
        for seed in range(5)
    ],
    rotation=35,
    ha="right",
)

ax_b.set_yticklabels(
    [
        f"Seed {seed}"
        for seed in range(5)
    ]
)

for row in range(5):
    for col in range(5):

        value = (
            ae_jaccard_values[
                row,
                col,
            ]
        )

        ax_b.text(
            col,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8,
            color=(
                "white"
                if value >= 0.65
                else "#222222"
            ),
        )

colorbar = fig3.colorbar(
    heatmap,
    ax=ax_b,
    fraction=0.046,
    pad=0.04,
)

colorbar.set_label(
    "Jaccard similarity"
)

ax_b.set_title(
    "B   AE consensus varies moderately across seeds",
    loc="left",
    fontweight="bold",
)

# ------------------------------------------------------------
# Panel C — AE seed-membership stability
# ------------------------------------------------------------

tier_x = np.arange(
    len(ae_membership_tiers)
)

tier_colors = [
    "#A6CEE3",
    "#7FB3D5",
    "#4C78A8",
    "#6A51A3",
    "#54278F",
]

tier_bars = ax_c.bar(
    tier_x,
    ae_membership_tiers[
        "n_knots"
    ],
    color=tier_colors,
    width=0.66,
)

ax_c.set_xticks(
    tier_x
)

ax_c.set_xticklabels(
    ae_membership_tiers[
        "minimum_seed_membership"
    ]
)

ax_c.set_xlabel(
    "Minimum AE seed membership"
)

ax_c.set_ylabel(
    "Number of test knots"
)

ax_c.set_ylim(
    0,
    64,
)

for bar, value in zip(
    tier_bars,
    ae_membership_tiers[
        "n_knots"
    ],
):

    ax_c.text(
        bar.get_x()
        +
        bar.get_width() / 2,
        value + 1.2,
        str(value),
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax_c.axvspan(
    1.55,
    4.45,
    color="#1B9E77",
    alpha=0.06,
)

ax_c.text(
    2.0,
    58.5,
    "Majority consensus",
    ha="center",
    va="center",
    fontsize=9,
    color="#166B55",
)

ax_c.set_title(
    "C   A stable nonlinear core persists across seeds",
    loc="left",
    fontweight="bold",
)

# ------------------------------------------------------------
# Panel D — Fraction of each AE set recovered by PCA
# ------------------------------------------------------------

overlap_x = np.arange(
    len(
        pca_ae_overlap_summary
    )
)

overlap_colors = [
    "#A6CEE3",
    "#A6CEE3",
    "#A6CEE3",
    "#A6CEE3",
    "#A6CEE3",
    "#1B9E77",
    "#6A51A3",
    "#54278F",
]

overlap_bars = ax_d.bar(
    overlap_x,
    pca_ae_overlap_summary[
        "fraction_ae_in_pca"
    ],
    color=overlap_colors,
    width=0.68,
)

ax_d.set_xticks(
    overlap_x
)

ax_d.set_xticklabels(
    pca_ae_overlap_summary[
        "analysis"
    ],
    rotation=35,
    ha="right",
)

ax_d.set_ylim(
    0,
    0.86,
)

ax_d.set_ylabel(
    "Fraction of AE set recovered by PCA"
)

for bar, fraction in zip(
    overlap_bars,
    pca_ae_overlap_summary[
        "fraction_ae_in_pca"
    ],
):

    ax_d.text(
        bar.get_x()
        +
        bar.get_width() / 2,
        fraction + 0.025,
        f"{fraction:.2f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

ax_d.axhline(
    0.5,
    color="#555555",
    linestyle="--",
    linewidth=0.9,
    alpha=0.7,
)

ax_d.set_title(
    "D   PCA recovers most of the stable AE majority",
    loc="left",
    fontweight="bold",
)

# ------------------------------------------------------------
# 3. Final layout and save
# ------------------------------------------------------------

fig3.suptitle(
    "Held-out evaluation and nonlinear reconstruction support the hard-regime signal",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)

fig3.tight_layout(
    h_pad=2.5,
    w_pad=2.0,
)

figure3_png = (
    FIGURE_DIR /
    "figure3_heldout_ae_robustness.png"
)

figure3_pdf = (
    FIGURE_DIR /
    "figure3_heldout_ae_robustness.pdf"
)

fig3.savefig(
    figure3_png,
    bbox_inches="tight",
)

fig3.savefig(
    figure3_pdf,
    bbox_inches="tight",
)

plt.show()

print("Saved:")
print(figure3_png)
print(figure3_pdf)

In [ ]:
# ============================================================
# STEP 18D — FIGURE 4: GEOMETRIC CHARACTERIZATION
# ============================================================

# ------------------------------------------------------------
# 1. Descriptive hyperbolic-volume source data
# ------------------------------------------------------------

descriptive_regime_order = [
    "amplitude_hard_only",
    "conditional_hard_only",
    "shared_raw_and_conditional",
]

descriptive_labels = {
    "amplitude_hard_only": "Amplitude\nonly",
    "conditional_hard_only": "Conditional\nonly",
    "shared_raw_and_conditional": "Shared",
    "universal_core": "Universal\ncore",
}

descriptive_colors = {
    "amplitude_hard_only": "#D95F02",
    "conditional_hard_only": "#1B9E77",
    "shared_raw_and_conditional": "#7570B3",
    "universal_core": "#E7298A",
}

volume_source_frames = []

for regime in descriptive_regime_order:

    frame = candidate_geometry.loc[
        candidate_geometry[
            "hard_regime"
        ].eq(regime)
        &
        candidate_geometry[
            "numerically_hyperbolic"
        ].eq(True),
        [
            "knot_id_base",
            "hard_regime",
            "volume",
        ],
    ].copy()

    frame["plot_regime"] = regime

    volume_source_frames.append(
        frame
    )

core_volume_source = (
    universal_core_snappy.loc[
        universal_core_snappy[
            "solution_type"
        ]
        .str.lower()
        .str.contains(
            "positively oriented",
            na=False,
        ),
        [
            "knot_id_base",
            "volume",
        ],
    ]
    .copy()
)

core_volume_source[
    "hard_regime"
] = "universal_core"

core_volume_source[
    "plot_regime"
] = "universal_core"

volume_source_frames.append(
    core_volume_source
)

figure4_volume_source = pd.concat(
    volume_source_frames,
    ignore_index=True,
)

figure4_volume_source.to_csv(
    FIGURE_SOURCE_DIR /
    "figure4_descriptive_volumes.csv",
    index=False,
)

# ------------------------------------------------------------
# 2. Pair-level matched differences
# ------------------------------------------------------------

continuous_columns = [
    "hyperbolic_volume",
    "num_tetrahedra",
    "symmetry_order",
]

paired_difference_rows = []

for regime_name in [
    "amplitude_only",
    "conditional_only",
]:

    pair_table = (
        paired_geometry_tables[
            regime_name
        ]
    )

    for outcome in continuous_columns:

        selected_values = pd.to_numeric(
            pair_table[
                "selected_" + outcome
            ],
            errors="coerce",
        )

        control_values = pd.to_numeric(
            pair_table[
                "control_" + outcome
            ],
            errors="coerce",
        )

        valid = (
            selected_values.notna()
            &
            control_values.notna()
        )

        for selected_index, difference in zip(
            pair_table.loc[
                valid,
                "selected_index",
            ],
            (
                selected_values[valid]
                -
                control_values[valid]
            ),
        ):

            paired_difference_rows.append(
                {
                    "regime": regime_name,
                    "selected_index": (
                        selected_index
                    ),
                    "outcome": outcome,
                    "paired_difference": (
                        difference
                    ),
                }
            )

figure4_paired_differences = pd.DataFrame(
    paired_difference_rows
)

figure4_paired_differences.to_csv(
    FIGURE_SOURCE_DIR /
    "figure4_paired_geometry_differences.csv",
    index=False,
)

# ------------------------------------------------------------
# 3. Descriptive geometry proportions
# ------------------------------------------------------------

geometry_proportion_source = (
    snappy_regime_summary_with_core[
        [
            "hard_regime",
            "n",
            "two_bridge_prop",
            "invertible_prop",
            "amphicheiral_prop",
            "symmetry_order_median",
        ]
    ]
    .copy()
)

geometry_proportion_source.to_csv(
    FIGURE_SOURCE_DIR /
    "figure4_geometry_proportions.csv",
    index=False,
)

# ------------------------------------------------------------
# 4. Matched standardized effects
# ------------------------------------------------------------

effect_source = (
    paired_geometry_results.loc[
        paired_geometry_results[
            "outcome_type"
        ].eq("continuous"),
        [
            "regime",
            "outcome",
            "n_valid_pairs",
            "paired_smd",
            "p_value",
            "q_value",
            "fdr_significant",
        ],
    ]
    .copy()
)

effect_source.to_csv(
    FIGURE_SOURCE_DIR /
    "figure4_matched_standardized_effects.csv",
    index=False,
)

# ------------------------------------------------------------
# 5. Figure canvas
# ------------------------------------------------------------

fig4, axes4 = plt.subplots(
    2,
    2,
    figsize=(11.8, 8.2),
)

ax_a, ax_b, ax_c, ax_d = (
    axes4.flatten()
)

# ------------------------------------------------------------
# Panel A — Descriptive volume distributions
# ------------------------------------------------------------

plot_volume_order = [
    "amplitude_hard_only",
    "conditional_hard_only",
    "shared_raw_and_conditional",
    "universal_core",
]

volume_arrays = [
    figure4_volume_source.loc[
        figure4_volume_source[
            "plot_regime"
        ].eq(regime),
        "volume",
    ]
    .dropna()
    .to_numpy()
    for regime
    in plot_volume_order
]

volume_boxplot = ax_a.boxplot(
    volume_arrays,
    labels=[
        descriptive_labels[regime]
        for regime
        in plot_volume_order
    ],
    patch_artist=True,
    showfliers=False,
    whis=(5, 95),
    widths=0.62,
    medianprops={
        "color": "black",
        "linewidth": 1.3,
    },
)

for patch, regime in zip(
    volume_boxplot["boxes"],
    plot_volume_order,
):

    patch.set_facecolor(
        descriptive_colors[regime]
    )

    patch.set_alpha(0.82)

# Show all six universal-core values
core_values = volume_arrays[-1]

rng = np.random.default_rng(42)

ax_a.scatter(
    4
    +
    rng.uniform(
        -0.08,
        0.08,
        size=len(core_values),
    ),
    core_values,
    color="#9E0142",
    edgecolor="white",
    linewidth=0.6,
    s=34,
    zorder=4,
)

ax_a.set_ylabel(
    "Hyperbolic volume"
)

ax_a.set_title(
    "A   Candidate regimes occupy different geometric ranges",
    loc="left",
    fontweight="bold",
)

ax_a.text(
    0.02,
    0.03,
    "Descriptive comparison",
    transform=ax_a.transAxes,
    fontsize=8,
    color="#666666",
)

# ------------------------------------------------------------
# Panel B — Paired hyperbolic-volume differences
# ------------------------------------------------------------

volume_difference_order = [
    "amplitude_only",
    "conditional_only",
]

volume_differences = [
    figure4_paired_differences.loc[
        figure4_paired_differences[
            "regime"
        ].eq(regime)
        &
        figure4_paired_differences[
            "outcome"
        ].eq("hyperbolic_volume"),
        "paired_difference",
    ]
    .dropna()
    .to_numpy()
    for regime
    in volume_difference_order
]

paired_colors = [
    "#D95F02",
    "#1B9E77",
]

paired_boxplot = ax_b.boxplot(
    volume_differences,
    labels=[
        "Amplitude only\nmatched subset",
        "Conditional only",
    ],
    patch_artist=True,
    showfliers=False,
    widths=0.55,
    medianprops={
        "color": "black",
        "linewidth": 1.3,
    },
)

for patch, color in zip(
    paired_boxplot["boxes"],
    paired_colors,
):

    patch.set_facecolor(color)
    patch.set_alpha(0.75)

for position, (
    values,
    color,
) in enumerate(
    zip(
        volume_differences,
        paired_colors,
    ),
    start=1,
):

    jitter = rng.uniform(
        -0.13,
        0.13,
        size=len(values),
    )

    ax_b.scatter(
        position + jitter,
        values,
        color=color,
        alpha=0.25,
        s=12,
        edgecolor="none",
        zorder=1,
    )

    ax_b.scatter(
        position,
        np.mean(values),
        marker="D",
        color="white",
        edgecolor=color,
        linewidth=1.5,
        s=52,
        zorder=5,
    )

ax_b.axhline(
    0,
    color="#333333",
    linestyle="--",
    linewidth=1.0,
)

volume_q_values = {}

for regime in volume_difference_order:

    volume_q_values[regime] = float(
        paired_geometry_results.loc[
            paired_geometry_results[
                "regime"
            ].eq(regime)
            &
            paired_geometry_results[
                "outcome"
            ].eq(
                "Hyperbolic volume"
            ),
            "q_value",
        ].iloc[0]
    )

ax_b.text(
    1,
    ax_b.get_ylim()[1] * 0.91,
    (
        f"n={len(volume_differences[0])}\n"
        f"q={volume_q_values['amplitude_only']:.2f}"
    ),
    ha="center",
    va="top",
    fontsize=8,
)

ax_b.text(
    2,
    ax_b.get_ylim()[1] * 0.91,
    (
        f"n={len(volume_differences[1])}\n"
        f"q={volume_q_values['conditional_only']:.1e}"
    ),
    ha="center",
    va="top",
    fontsize=8,
)

ax_b.set_ylabel(
    "Selected − matched-control volume"
)

ax_b.set_title(
    "B   Conditional-hard knots have lower matched volume",
    loc="left",
    fontweight="bold",
)

# ------------------------------------------------------------
# Panel C — Descriptive topology proportions
# ------------------------------------------------------------

summary_index = (
    snappy_regime_summary_with_core
    .set_index("hard_regime")
)

summary_names = [
    "amplitude_hard_only",
    "conditional_hard_only",
    "shared_raw_and_conditional",
    "universal_PCA_AE_conditional_core",
]

summary_labels = [
    "Amplitude\nonly",
    "Conditional\nonly",
    "Shared",
    "Universal\ncore",
]

two_bridge_values = (
    summary_index.loc[
        summary_names,
        "two_bridge_prop",
    ]
    .to_numpy()
)

invertible_values = (
    summary_index.loc[
        summary_names,
        "invertible_prop",
    ]
    .to_numpy()
)

x = np.arange(
    len(summary_names)
)

width = 0.34

ax_c.bar(
    x - width / 2,
    two_bridge_values,
    width,
    color="#4C78A8",
    label="Two-bridge recognized",
)

ax_c.bar(
    x + width / 2,
    invertible_values,
    width,
    color="#E45756",
    label="Invertible",
)

ax_c.set_xticks(x)
ax_c.set_xticklabels(
    summary_labels
)

ax_c.set_ylim(
    0,
    1.08,
)

ax_c.set_ylabel(
    "Proportion"
)

ax_c.legend(
    frameon=False,
    loc="upper left",
)

# Panel C: “two-bridge” no es estrictamente una simetría
ax_c.set_title(
    "C   Topological and symmetry profiles differ across regimes",
    loc="left",
    fontweight="bold",
)

ax_c.text(
    0.02,
    0.03,
    "Descriptive comparison",
    transform=ax_c.transAxes,
    fontsize=8,
    color="#666666",
)

# ------------------------------------------------------------
# Panel D — Matched standardized continuous effects
# ------------------------------------------------------------

effect_outcome_order = [
    "Hyperbolic volume",
    "Census triangulation tetrahedra",
    "Symmetry-group order",
]

effect_labels = {
    "Hyperbolic volume": "Hyperbolic volume",
    "Census triangulation tetrahedra": (
        "Census tetrahedra"
    ),
    "Symmetry-group order": (
        "Symmetry-group order"
    ),
}

effect_y = np.arange(
    len(effect_outcome_order)
)[::-1]

effect_regime_specs = [
    {
        "regime": "amplitude_only",
        "label": "Amplitude-only subset",
        "color": "#D95F02",
        "offset": 0.11,
    },
    {
        "regime": "conditional_only",
        "label": "Conditional-only",
        "color": "#1B9E77",
        "offset": -0.11,
    },
]

ax_d.axvline(
    0,
    color="#333333",
    linewidth=1.0,
    linestyle="--",
)

for regime_spec in effect_regime_specs:

    regime_df = (
        effect_source.loc[
            effect_source[
                "regime"
            ].eq(
                regime_spec["regime"]
            )
        ]
        .set_index("outcome")
        .loc[
            effect_outcome_order
        ]
        .reset_index()
    )

    y_positions = (
        effect_y
        +
        regime_spec["offset"]
    )

    for y_position, row in zip(
        y_positions,
        regime_df.itertuples(
            index=False
        ),
    ):

        significant = bool(
            row.fdr_significant
        )

        ax_d.scatter(
            row.paired_smd,
            y_position,
            marker="o",
            s=74,
            facecolor=(
                regime_spec["color"]
                if significant
                else "white"
            ),
            edgecolor=regime_spec["color"],
            linewidth=1.7,
            zorder=4,
        )

        q_text = (
            f"q={row.q_value:.1e}"
            if row.q_value < 0.001
            else
            f"q={row.q_value:.3f}"
        )

        horizontal_alignment = (
            "left"
            if row.paired_smd >= 0
            else
            "right"
        )

        text_offset = (
            0.018
            if row.paired_smd >= 0
            else
            -0.018
        )

        ax_d.text(
            row.paired_smd
            +
            text_offset,
            y_position,
            q_text,
            ha=horizontal_alignment,
            va="center",
            fontsize=7.5,
            color="#444444",
        )

ax_d.set_yticks(
    effect_y
)

ax_d.set_yticklabels(
    [
        effect_labels[outcome]
        for outcome
        in effect_outcome_order
    ]
)

ax_d.set_xlabel(
    "Standardized paired difference"
)

ax_d.set_xlim(
    -0.42,
    0.30,
)

ax_d.set_title(
    "D   Matched effects are specific to conditional hardness",
    loc="left",
    fontweight="bold",
)

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor="white",
        markeredgecolor="#D95F02",
        markeredgewidth=1.7,
        markersize=7,
        label="Amplitude-only subset",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor="#1B9E77",
        markeredgecolor="#1B9E77",
        markersize=7,
        label="Conditional-only",
    ),
]

# Panel D: mover la leyenda para que no tape el punto ni q=0.015
ax_d.legend(
    loc="center right",
    bbox_to_anchor=(0.99, 0.58),
    frameon=False,
)

# ------------------------------------------------------------
# 6. Final layout and save
# ------------------------------------------------------------

fig4.suptitle(
    "Conditional multiview hardness marks lower-volume, modestly more symmetric knot complements",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)

fig4.text(
    0.5,
    -0.015,
    (
        "Hyperbolic structures are numerical SnapPy solutions. "
        "Matched inference uses unique controls exact on crossings, "
        "alternation and signature, with norm caliper 0.50."
    ),
    ha="center",
    va="top",
    fontsize=8.5,
    color="#444444",
)

fig4.tight_layout(
    h_pad=2.5,
    w_pad=2.0,
)

figure4_png = (
    FIGURE_DIR /
    "figure4_geometric_characterization.png"
)

figure4_pdf = (
    FIGURE_DIR /
    "figure4_geometric_characterization.pdf"
)

fig4.savefig(
    figure4_png,
    bbox_inches="tight",
)

fig4.savefig(
    figure4_pdf,
    bbox_inches="tight",
)

plt.show()

print("Saved:")
print(figure4_png)
print(figure4_pdf)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# Directories
# ------------------------------------------------------------------
RUN_DIR = Path(
    (__import__("os").environ.get('KNOT_OUTPUT_DIR', '/content/drive/MyDrive/Colab Notebooks/data_invariants/Invariants/processed_consensus_hardness/corrected_run_20260819') + '')
)

TABLE_DIR = RUN_DIR / "19_paper_tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TOTAL_KNOTS = 313_230


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def formatted_copy(df, formats=None):
    """Format only the paper-facing copy; preserve numeric source CSV."""
    out = df.copy()

    if formats:
        for column, formatter in formats.items():
            if column in out.columns:
                out[column] = out[column].map(
                    lambda x: formatter(x) if pd.notna(x) else ""
                )

    return out


def save_paper_table(
    df,
    stem,
    caption,
    label,
    formats=None,
    column_format=None,
):
    # Numeric/auditable source
    csv_path = TABLE_DIR / f"{stem}_source.csv"
    df.to_csv(csv_path, index=False)

    # Paper-formatted version
    paper_df = formatted_copy(df, formats)

    display_path = TABLE_DIR / f"{stem}_display.csv"
    paper_df.to_csv(display_path, index=False)

    latex = paper_df.to_latex(
        index=False,
        escape=True,
        caption=caption,
        label=label,
        position="t",
        column_format=column_format,
    )

    tex_path = TABLE_DIR / f"{stem}.tex"
    tex_path.write_text(latex, encoding="utf-8")

    print("\n" + "=" * 90)
    print(stem)
    print("=" * 90)
    display(paper_df)

    return {
        "source_csv": csv_path,
        "display_csv": display_path,
        "latex": tex_path,
    }


# ==================================================================
# TABLE 1 — Representation and PCA design
# ==================================================================
table1 = pd.DataFrame(
    [
        ["Alexander", 17, 4, 0.995566],
        ["Jones", 51, 10, 0.990385],
        ["HOMFLY-PT", 152, 32, 0.990028],
        ["Theta", 841, 10, 0.992110],
        ["Khovanov", 373, 77, 0.990368],
    ],
    columns=[
        "Representation",
        "Input dimension",
        "Components k99",
        "Cumulative explained variance",
    ],
)

table1["Compression ratio"] = (
    table1["Input dimension"] / table1["Components k99"]
)

files_table1 = save_paper_table(
    table1,
    stem="table1_representation_pca_design",
    caption=(
        "Dimensions and PCA compression of the five knot-invariant "
        "representations. The retained dimension k99 is the smallest "
        "number of components explaining at least 99 percent of the variance."
    ),
    label="tab:pca-design",
    formats={
        "Cumulative explained variance": lambda x: f"{x:.4f}",
        "Compression ratio": lambda x: f"{x:.2f}",
    },
    column_format="lrrrr",
)


# ==================================================================
# TABLE 2 — Hard-regime summary
# ==================================================================
table2 = pd.DataFrame(
    [
        [
            "Background", 312571, 0.808501, 0.355999,
            2, 2, 2.469207, 201296,
            0.061338, 0.000139, 0.029429,
        ],
        [
            "Amplitude only", 246, 0.971545, 0.723577,
            10, 10, 7.643325, 68,
            0.970588, 0.058824, 2.058824,
        ],
        [
            "Conditional only", 367, 0.806540, 0.128065,
            8, 10, 3.567514, 320,
            0.756250, 0.078125, 1.668750,
        ],
        [
            "Shared", 46, 0.956522, 0.608696,
            10, 12, 6.825840, 18,
            1.000000, 0.444444, 2.888889,
        ],
    ],
    columns=[
        "Regime",
        "n",
        "Crossing 15 proportion",
        "Alternating proportion",
        "Median signature",
        "Median s",
        "Median mean log norm",
        "Nonalternating n",
        "Positive gap proportion",
        "Large gap proportion",
        "Mean gap",
    ],
)

table2.insert(2, "Atlas proportion", table2["n"] / TOTAL_KNOTS)

files_table2 = save_paper_table(
    table2,
    stem="table2_hard_regime_summary",
    caption=(
        "Structural, amplitude, and concordance characteristics of the "
        "background, amplitude-only, conditional-only, and shared regimes. "
        "Gap statistics are computed among nonalternating knots, with "
        "gap defined as s minus signature."
    ),
    label="tab:regime-summary",
    formats={
        "Atlas proportion": lambda x: f"{100*x:.3f}%",
        "Crossing 15 proportion": lambda x: f"{x:.3f}",
        "Alternating proportion": lambda x: f"{x:.3f}",
        "Median mean log norm": lambda x: f"{x:.3f}",
        "Positive gap proportion": lambda x: f"{x:.3f}",
        "Large gap proportion": lambda x: f"{x:.3f}",
        "Mean gap": lambda x: f"{x:.3f}",
    },
    column_format="lrrrrrrrrrrr",
)


# ==================================================================
# TABLE 3 — Focused exact-stratified null tests
# ==================================================================
table3 = pd.DataFrame(
    [
        [
            "All five views", "At least 3 of 5", 413, 338,
            "Positive gap proportion",
            0.769231, 0.350172, 0.417476, 0.447368,
            5000, 0, 0.000200, 0.000236,
        ],
        [
            "All five views", "At least 3 of 5", 413, 338,
            "Mean gap",
            1.733728, 0.747633, 0.888889, 0.948469,
            5000, 0, 0.000200, 0.000236,
        ],
        [
            "All five views", "At least 3 of 5", 413, 338,
            "Large gap proportion",
            0.097633, 0.024309, 0.038462, 0.045455,
            5000, 0, 0.000200, 0.000236,
        ],
        [
            "Without Khovanov", "At least 3 of 4", 220, 167,
            "Positive gap proportion",
            0.868263, 0.342729, 0.436364, 0.480769,
            5000, 0, 0.000200, 0.000236,
        ],
        [
            "Without Khovanov", "At least 3 of 4", 220, 167,
            "Mean gap",
            2.059880, 0.742885, 0.941176, 1.041685,
            5000, 0, 0.000200, 0.000236,
        ],
        [
            "Without Khovanov", "At least 3 of 4", 220, 167,
            "Large gap proportion",
            0.161677, 0.028983, 0.051770, 0.065217,
            5000, 0, 0.000200, 0.000236,
        ],
        [
            "Polynomial only", "All 3 of 3", 35, 8,
            "Positive gap proportion",
            1.000000, 0.111260, 0.500000, 1.000000,
            4768, 232, 0.019711, 0.019711,
        ],
        [
            "Polynomial only", "All 3 of 3", 35, 8,
            "Mean gap",
            2.750000, 0.222437, 1.000000, 2.000000,
            4768, 232, 0.000210, 0.000236,
        ],
        [
            "Polynomial only", "All 3 of 3", 35, 8,
            "Large gap proportion",
            0.375000, 0.000105, 0.000000, 0.000000,
            4768, 232, 0.000210, 0.000236,
        ],
    ],
    columns=[
        "View family",
        "Selection rule",
        "Selected n",
        "Nonalternating n",
        "Metric",
        "Observed",
        "Null mean",
        "Null q95",
        "Null q99",
        "Valid nulls",
        "Missing nulls",
        "Empirical p",
        "FDR q",
    ],
)

files_table3 = save_paper_table(
    table3,
    stem="table3_exact_conditional_nulls",
    caption=(
        "Focused exact-stratified null tests for the nonalternating "
        "Rasmussen-signature gap. Null permutations preserve norm bin, "
        "crossing number, alternation status, and exact signature. "
        "FDR correction is applied across the nine focused tests."
    ),
    label="tab:exact-nulls",
    formats={
        "Observed": lambda x: f"{x:.3f}",
        "Null mean": lambda x: f"{x:.3f}",
        "Null q95": lambda x: f"{x:.3f}",
        "Null q99": lambda x: f"{x:.3f}",
        "Empirical p": lambda x: (
            f"{x:.2e}" if x < 0.001 else f"{x:.4f}"
        ),
        "FDR q": lambda x: (
            f"{x:.2e}" if x < 0.001 else f"{x:.4f}"
        ),
    },
    column_format="llrrlrrrrrrrr",
)


# ==================================================================
# TABLE 4 — Matched geometric inference
# ==================================================================
table4 = pd.DataFrame(
    [
        [
            "Amplitude-only matched subset",
            "Hyperbolic volume",
            109, 20.540225, 20.390978,
            0.149246, 0.065928,
            0.890590, 1.000000, False,
        ],
        [
            "Amplitude-only matched subset",
            "Census tetrahedra",
            109, 22.348624, 22.201835,
            0.146789, 0.064949,
            0.251647, 0.572563, False,
        ],
        [
            "Amplitude-only matched subset",
            "Symmetry-group order",
            109, 1.596330, 1.761468,
            -0.165138, -0.102373,
            0.286281, 0.572563, False,
        ],
        [
            "Conditional only",
            "Hyperbolic volume",
            355, 16.996740, 17.775405,
            -0.778665, -0.264309,
            3.739127e-7, 5.234778e-6, True,
        ],
        [
            "Conditional only",
            "Census tetrahedra",
            357, 18.551821, 19.305322,
            -0.753501, -0.247101,
            8.807891e-7, 6.165524e-6, True,
        ],
        [
            "Conditional only",
            "Symmetry-group order",
            355, 1.723944, 1.535211,
            0.188732, 0.133421,
            0.003280, 0.015307, True,
        ],
    ],
    columns=[
        "Regime",
        "Outcome",
        "Valid pairs",
        "Selected mean",
        "Control mean",
        "Paired difference",
        "Paired SMD",
        "p value",
        "FDR q",
        "FDR significant",
    ],
)

files_table4 = save_paper_table(
    table4,
    stem="table4_matched_geometric_inference",
    caption=(
        "Unique-control matched comparisons of geometric properties. "
        "Matching is exact on crossing number, alternation status, and "
        "signature stratum, followed by nearest matching on vector-norm "
        "covariates with caliper 0.50. The amplitude-only analysis applies "
        "to its common-support subset."
    ),
    label="tab:matched-geometry",
    formats={
        "Selected mean": lambda x: f"{x:.3f}",
        "Control mean": lambda x: f"{x:.3f}",
        "Paired difference": lambda x: f"{x:+.3f}",
        "Paired SMD": lambda x: f"{x:+.3f}",
        "p value": lambda x: (
            f"{x:.2e}" if x < 0.001 else f"{x:.4f}"
        ),
        "FDR q": lambda x: (
            f"{x:.2e}" if x < 0.001 else f"{x:.4f}"
        ),
        "FDR significant": lambda x: "Yes" if x else "No",
    },
    column_format="llrrrrrrrl",
)


# ==================================================================
# SUPPLEMENTARY TABLE — Universal six-knot core
# ==================================================================
table_s1 = pd.DataFrame(
    [
        [
            "15n159774", False, 8, 10, 2,
            5.445979, 7.530800, 5, 4, 3, 0.998084,
        ],
        [
            "15n164613", False, 8, 10, 2,
            5.683954, 8.252608, 5, 3, 4, 0.994733,
        ],
        [
            "15a58615", True, 12, 12, 0,
            10.154417, 12.584616, 5, 4, 4, 0.994400,
        ],
        [
            "15n119483", False, 8, 12, 4,
            6.921470, 10.114073, 5, 3, 5, 0.993457,
        ],
        [
            "15n143746", False, 8, 12, 4,
            6.459680, 9.643086, 5, 3, 5, 0.992976,
        ],
        [
            "15n159815", False, 8, 10, 2,
            5.279236, 7.691743, 5, 4, 3, 0.992020,
        ],
    ],
    columns=[
        "Knot",
        "Alternating",
        "Signature",
        "s invariant",
        "Gap",
        "Mean log norm",
        "Maximum log norm",
        "Raw PCA membership",
        "Conditional membership",
        "AE seed membership",
        "All-view conditional score",
    ],
)

files_table_s1 = save_paper_table(
    table_s1,
    stem="tableS1_universal_six_knot_core",
    caption=(
        "Universal six-knot case-study core recovered by full-data PCA, "
        "held-out PCA, held-out autoencoders, and norm-conditioned "
        "multiview hardness."
    ),
    label="tab:universal-core",
    formats={
        "Alternating": lambda x: "Yes" if x else "No",
        "Mean log norm": lambda x: f"{x:.3f}",
        "Maximum log norm": lambda x: f"{x:.3f}",
        "All-view conditional score": lambda x: f"{x:.4f}",
    },
    column_format="llrrrrrrrrr",
)


# ------------------------------------------------------------------
# Registry
# ------------------------------------------------------------------
registry_rows = []

for table_name, collection in [
    ("Table 1", files_table1),
    ("Table 2", files_table2),
    ("Table 3", files_table3),
    ("Table 4", files_table4),
    ("Table S1", files_table_s1),
]:
    registry_rows.append(
        {
            "table": table_name,
            "source_csv": str(collection["source_csv"]),
            "display_csv": str(collection["display_csv"]),
            "latex": str(collection["latex"]),
        }
    )

table_registry = pd.DataFrame(registry_rows)
table_registry.to_csv(TABLE_DIR / "paper_table_registry.csv", index=False)

print("\nSaved paper tables to:")
print(TABLE_DIR)
print("\nRegistry:")
display(table_registry)

In [ ]:
from pathlib import Path
import pandas as pd

TABLE_DIR = Path(
    (__import__("os").environ.get('KNOT_OUTPUT_DIR', '/content/drive/MyDrive/Colab Notebooks/data_invariants/Invariants/processed_consensus_hardness/corrected_run_20260819') + '/19_paper_tables')
)


def save_compact_table(df, stem, caption, label, column_format):
    display_path = TABLE_DIR / f"{stem}_display.csv"
    tex_path = TABLE_DIR / f"{stem}.tex"

    df.to_csv(display_path, index=False)

    latex = df.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
        position="t",
        column_format=column_format,
    )

    tex_path.write_text(latex, encoding="utf-8")

    print("\n" + "=" * 90)
    print(stem)
    print("=" * 90)
    display(df)


# ==============================================================
# TABLE 1 — Compact PCA design
# ==============================================================

t1 = pd.read_csv(
    TABLE_DIR / "table1_representation_pca_design_source.csv"
)

t1_paper = pd.DataFrame({
    "Representation": t1["Representation"],
    "Input dimension": t1["Input dimension"],
    "$k_{99}$": t1["Components k99"],
    "Explained variance": t1["Cumulative explained variance"].map(
        lambda x: f"{100*x:.2f}\\%"
    ),
    "Compression": t1["Compression ratio"].map(
        lambda x: f"{x:.2f}$\\times$"
    ),
})

save_compact_table(
    t1_paper,
    stem="table1_representation_pca_design_paper",
    caption=(
        "Dimensions and PCA compression of the five invariant "
        "representations. Here, $k_{99}$ is the smallest number of "
        "components explaining at least 99\\% of the variance."
    ),
    label="tab:pca-design",
    column_format="lrrrr",
)


# ==============================================================
# TABLE 2 — Compact regime summary
# ==============================================================

t2 = pd.read_csv(
    TABLE_DIR / "table2_hard_regime_summary_source.csv"
)

t2_paper = pd.DataFrame({
    "Regime": t2["Regime"],
    "$n$": t2["n"],
    "15 crossings": t2["Crossing 15 proportion"].map(
        lambda x: f"{100*x:.1f}\\%"
    ),
    "Alternating": t2["Alternating proportion"].map(
        lambda x: f"{100*x:.1f}\\%"
    ),
    "Median $\\sigma$": t2["Median signature"].map(
        lambda x: f"{x:g}"
    ),
    "Median $s$": t2["Median s"].map(
        lambda x: f"{x:g}"
    ),
    "Median log norm": t2["Median mean log norm"].map(
        lambda x: f"{x:.3f}"
    ),
    "Nonalt. $n$": t2["Nonalternating n"],
    "$P(\\Delta>0)$": t2["Positive gap proportion"].map(
        lambda x: f"{100*x:.1f}\\%"
    ),
    "$P(\\Delta\\geq4)$": t2["Large gap proportion"].map(
        lambda x: f"{100*x:.3f}\\%"
    ),
    "Mean $\\Delta$": t2["Mean gap"].map(
        lambda x: f"{x:.3f}"
    ),
})

save_compact_table(
    t2_paper,
    stem="table2_hard_regime_summary_paper",
    caption=(
        "Structural, amplitude, and concordance characteristics of the "
        "four regimes. Gap statistics are restricted to nonalternating "
        "knots and use $\\Delta=s-\\sigma$."
    ),
    label="tab:regime-summary",
    column_format="lrrrrrrrrrr",
)


# ==============================================================
# TABLE 3 — Compact exact-null results
# ==============================================================

t3 = pd.read_csv(
    TABLE_DIR / "table3_exact_conditional_nulls_source.csv"
)

family_labels = {
    "All five views": "All five ($\\geq3/5$)",
    "Without Khovanov": "Without Khovanov ($\\geq3/4$)",
    "Polynomial only": "Polynomial only ($3/3$)",
}

metric_labels = {
    "Positive gap proportion": "$P(\\Delta>0)$",
    "Mean gap": "Mean $\\Delta$",
    "Large gap proportion": "$P(\\Delta\\geq4)$",
}

t3_paper = pd.DataFrame({
    "View family": t3["View family"].map(family_labels),
    "Selected $n$": t3["Selected n"],
    "Nonalt. $n$": t3["Nonalternating n"],
    "Metric": t3["Metric"].map(metric_labels),
    "Observed": t3["Observed"].map(lambda x: f"{x:.3f}"),
    "Null mean": t3["Null mean"].map(lambda x: f"{x:.3f}"),
    "Null 95--99\\%": [
        f"[{q95:.3f}, {q99:.3f}]"
        for q95, q99 in zip(t3["Null q95"], t3["Null q99"])
    ],
    "$p_{\\mathrm{emp}}$": t3["Empirical p"].map(
        lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}"
    ),
    "FDR $q$": t3["FDR q"].map(
        lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}"
    ),
})

save_compact_table(
    t3_paper,
    stem="table3_exact_conditional_nulls_paper",
    caption=(
        "Focused exact-stratified null tests for the nonalternating "
        "concordance gap $\\Delta=s-\\sigma$. Permutations preserve norm "
        "bin, crossing number, alternation status, and exact signature. "
        "Tests used 5,000 permutations; polynomial-only metrics had "
        "4,768 valid null realizations."
    ),
    label="tab:exact-nulls",
    column_format="lrrlrrrrr",
)


# ==============================================================
# TABLE 4 — Matched geometric inference
# ==============================================================

t4 = pd.read_csv(
    TABLE_DIR / "table4_matched_geometric_inference_source.csv"
)

t4["Outcome"] = t4["Outcome"].replace({
    "Census tetrahedra": "Census triangulation tetrahedra"
})

t4_paper = pd.DataFrame({
    "Regime": t4["Regime"],
    "Outcome": t4["Outcome"],
    "Pairs": t4["Valid pairs"],
    "Selected": t4["Selected mean"].map(lambda x: f"{x:.3f}"),
    "Control": t4["Control mean"].map(lambda x: f"{x:.3f}"),
    "Difference": t4["Paired difference"].map(lambda x: f"{x:+.3f}"),
    "Paired SMD": t4["Paired SMD"].map(lambda x: f"{x:+.3f}"),
    "FDR $q$": t4["FDR q"].map(
        lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}"
    ),
})

save_compact_table(
    t4_paper,
    stem="table4_matched_geometric_inference_paper",
    caption=(
        "Unique-control matched geometric comparisons. Matching is exact "
        "on crossing number, alternation status, and signature stratum, "
        "with nearest matching on norm covariates and caliper 0.50. "
        "Census tetrahedron counts describe the stored triangulations "
        "and are not minimal-triangulation invariants."
    ),
    label="tab:matched-geometry",
    column_format="llrrrrrr",
)


print("\nPaper-facing tables saved to:")
print(TABLE_DIR)

In [ ]:
get_ipython().run_line_magic("run", '-i "' + str(PROJECT_DIR / 'notebooks/stage18A_mirror_invariant_gap.py') + '"')

In [ ]:
orientation_audit = pd.DataFrame([
    {
        "quantity": "signature < 0",
        "n": int((meta_mirror["signature"] < 0).sum()),
    },
    {
        "quantity": "signature = 0",
        "n": int((meta_mirror["signature"] == 0).sum()),
    },
    {
        "quantity": "signature > 0",
        "n": int((meta_mirror["signature"] > 0).sum()),
    },
    {
        "quantity": "delta < 0",
        "n": int((meta_mirror["delta_signed"] < 0).sum()),
    },
    {
        "quantity": "delta = 0",
        "n": int((meta_mirror["delta_signed"] == 0).sum()),
    },
    {
        "quantity": "delta > 0",
        "n": int((meta_mirror["delta_signed"] > 0).sum()),
    },
])

display(orientation_audit)

strata_comparison = pd.DataFrame([
    {
        "invariant": invariant,
        "signed_and_absolute_strata_identical": bool(
            np.array_equal(
                joint_strata_signed_sigma[invariant],
                joint_strata_abs_sigma[invariant],
            )
        ),
    }
    for invariant in INVARIANTS
])

display(strata_comparison)

In [ ]:
import numpy as np
import pandas as pd

assert "aligned" in globals()
assert "REPRESENTATION_SPECS" in globals()

cleaned_tables = aligned["cleaned_tables"]
aligned_tables = aligned["aligned_tables"]
common_ids = set(meta[CONFIG.universe.id_col].astype(str))

rows = []

for invariant, spec in REPRESENTATION_SPECS.items():
    source = spec["source"]
    df = cleaned_tables[source].copy()

    df["knot_id_base"] = df["knot_id_base"].astype(str)
    df = df[df["knot_id_base"].isin(common_ids)].copy()

    df["_is_mirror"] = (
        df["knot_id_clean"]
        .astype(str)
        .str.contains("!", regex=False)
    )

    sig = pd.to_numeric(df["signature"], errors="coerce")
    df["_sig"] = sig

    by_base = (
        df.groupby("knot_id_base", sort=False)
        .agg(
            n_rows=("knot_id_clean", "size"),
            has_plain=("_is_mirror", lambda x: (~x).any()),
            has_mirror=("_is_mirror", "any"),
            sig_min=("_sig", "min"),
            sig_max=("_sig", "max"),
        )
    )

    pairable = by_base["has_plain"] & by_base["has_mirror"]
    strict_sign_pair = (
        pairable
        & by_base["sig_min"].lt(0)
        & by_base["sig_max"].gt(0)
    )
    zero_signature_pair = (
        pairable
        & by_base["sig_min"].eq(0)
        & by_base["sig_max"].eq(0)
    )

    current = aligned_tables[source].copy()
    current["_is_mirror"] = (
        current["knot_id_clean"]
        .astype(str)
        .str.contains("!", regex=False)
    )

    rows.append({
        "invariant": invariant,
        "source": source,
        "common_base_knots": len(by_base),
        "bases_with_plain_and_mirror": int(pairable.sum()),
        "mirror_pair_coverage": float(pairable.mean()),
        "strict_positive_negative_pairs": int(strict_sign_pair.sum()),
        "zero_signature_pairs": int(zero_signature_pair.sum()),
        "currently_selected_mirror_rows":
            int(current["_is_mirror"].sum()),
    })

mirror_representation_preflight = pd.DataFrame(rows)

display(mirror_representation_preflight)


In [ ]:
import numpy as np
import pandas as pd

feature_cols_dict = aligned["feature_cols_dict"]
cleaned_tables = aligned["cleaned_tables"]

# ------------------------------------------------------------
# 1. Esquema de coordenadas de cada representación
# ------------------------------------------------------------
schema_rows = []

for invariant, spec in REPRESENTATION_SPECS.items():
    cols = list(feature_cols_dict[invariant])

    schema_rows.append({
        "invariant": invariant,
        "source": spec["source"],
        "n_features": len(cols),
        "first_features": " | ".join(map(str, cols[:12])),
        "last_features": " | ".join(map(str, cols[-12:])),
    })

feature_schema_audit = pd.DataFrame(schema_rows)

pd.set_option("display.max_colwidth", 500)
display(feature_schema_audit)


# ------------------------------------------------------------
# 2. ¿El espejo explícito es una permutación/reversión?
#    Diagnóstico para Jones y HOMFLY-PT
# ------------------------------------------------------------
def paired_mirror_diagnostic(invariant, sample_size=2000, seed=20260819):
    source = REPRESENTATION_SPECS[invariant]["source"]
    cols = list(feature_cols_dict[invariant])

    df = cleaned_tables[source].copy()
    df["knot_id_base"] = df["knot_id_base"].astype(str)
    df["_is_mirror"] = (
        df["knot_id_clean"]
        .astype(str)
        .str.contains("!", regex=False)
    )

    plain_ids = set(
        df.loc[~df["_is_mirror"], "knot_id_base"]
    )
    mirror_ids = set(
        df.loc[df["_is_mirror"], "knot_id_base"]
    )
    paired_ids = sorted(plain_ids & mirror_ids)

    rng = np.random.default_rng(seed)
    if len(paired_ids) > sample_size:
        paired_ids = sorted(
            rng.choice(
                paired_ids,
                size=sample_size,
                replace=False,
            )
        )

    plain = (
        df[
            (~df["_is_mirror"])
            & df["knot_id_base"].isin(paired_ids)
        ]
        .drop_duplicates("knot_id_base")
        .set_index("knot_id_base")
        .loc[paired_ids]
    )

    mirror = (
        df[
            df["_is_mirror"]
            & df["knot_id_base"].isin(paired_ids)
        ]
        .drop_duplicates("knot_id_base")
        .set_index("knot_id_base")
        .loc[paired_ids]
    )

    A = (
        plain[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .to_numpy(float)
    )

    B = (
        mirror[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .to_numpy(float)
    )

    norm_a = np.linalg.norm(A, axis=1)
    norm_b = np.linalg.norm(B, axis=1)

    same_rows = np.all(
        np.isclose(A, B, atol=1e-10, rtol=1e-10),
        axis=1,
    )

    reversed_rows = np.all(
        np.isclose(A[:, ::-1], B, atol=1e-10, rtol=1e-10),
        axis=1,
    )

    nonzero = (norm_a > 0) & (norm_b > 0)

    return {
        "invariant": invariant,
        "n_sample_pairs": len(paired_ids),
        "same_vector_prop": float(same_rows.mean()),
        "full_reverse_prop": float(reversed_rows.mean()),
        "norm_preserved_prop": float(
            np.isclose(
                norm_a,
                norm_b,
                atol=1e-8,
                rtol=1e-8,
            ).mean()
        ),
        "median_mirror_to_plain_norm_ratio": float(
            np.median(norm_b[nonzero] / norm_a[nonzero])
        ),
        "direct_flat_correlation": float(
            np.corrcoef(A.ravel(), B.ravel())[0, 1]
        ),
        "reversed_flat_correlation": float(
            np.corrcoef(A[:, ::-1].ravel(), B.ravel())[0, 1]
        ),
    }


paired_diagnostics = pd.DataFrame([
    paired_mirror_diagnostic("Jones"),
    paired_mirror_diagnostic("HOMFLY-PT"),
])

display(paired_diagnostics)

In [ ]:
import re
import numpy as np
import pandas as pd

# ============================================================
# A. Comprobar que la canonicalización actual coincide
#    exactamente con "elegir siempre la fila sin !"
# ============================================================
target_free_rows = []

common_order = (
    meta[CONFIG.universe.id_col]
    .astype(str)
    .tolist()
)

for invariant, spec in REPRESENTATION_SPECS.items():
    source = spec["source"]
    cols = list(feature_cols_dict[invariant])

    clean = aligned["cleaned_tables"][source].copy()
    clean["knot_id_base"] = clean["knot_id_base"].astype(str)

    plain = clean.loc[
        ~clean["knot_id_clean"]
        .astype(str)
        .str.contains("!", regex=False)
    ].copy()

    duplicate_plain = int(
        plain["knot_id_base"].duplicated().sum()
    )

    plain = (
        plain
        .drop_duplicates("knot_id_base")
        .set_index("knot_id_base")
        .reindex(common_order)
    )

    missing_plain = int(
        plain["knot_id_clean"].isna().sum()
    )

    X_plain = (
        plain[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .to_numpy(np.float32)
    )

    X_current = np.asarray(X_dict[invariant])

    target_free_rows.append({
        "invariant": invariant,
        "duplicate_plain_rows": duplicate_plain,
        "missing_plain_knots": missing_plain,
        "same_shape": X_plain.shape == X_current.shape,
        "exact_feature_matrix_match":
            bool(np.array_equal(X_plain, X_current)),
        "max_abs_difference": float(
            np.max(np.abs(X_plain - X_current))
        ),
    })

target_free_canonicalization_audit = pd.DataFrame(
    target_free_rows
)

display(target_free_canonicalization_audit)

assert (
    target_free_canonicalization_audit[
        "exact_feature_matrix_match"
    ].all()
), "La canonicalización target-free no reproduce la matriz actual."


# ============================================================
# B. Funciones para interpretar coordenadas
# ============================================================
def parse_homfly(col):
    match = re.fullmatch(r"a(-?\d+)_z(-?\d+)", str(col))
    return tuple(map(int, match.groups())) if match else None


def parse_theta(col):
    match = re.fullmatch(
        r"T1(-?\d+)_T2(-?\d+)",
        str(col),
    )
    return tuple(map(int, match.groups())) if match else None


def parse_khovanov(col):
    match = re.fullmatch(
        r"F_q(-?\d+)_t(-?\d+)",
        str(col),
    )
    return tuple(map(int, match.groups())) if match else None


def transformation_closure(
    invariant,
    parser,
    transform,
):
    cols = list(feature_cols_dict[invariant])
    coordinates = [parser(col) for col in cols]

    if any(coord is None for coord in coordinates):
        raise ValueError(
            f"No pude interpretar todas las columnas de {invariant}"
        )

    coordinate_set = set(coordinates)
    transformed = [transform(*coord) for coord in coordinates]
    missing = sorted(set(transformed) - coordinate_set)

    return {
        "invariant": invariant,
        "n_features": len(cols),
        "n_transformed_coordinates_missing": len(missing),
        "closed_under_candidate_transform":
            len(missing) == 0,
        "missing_examples": missing[:15],
    }


closure_audit = pd.DataFrame([
    transformation_closure(
        "HOMFLY-PT",
        parse_homfly,
        lambda a, z: (-a, z),
    ),
    transformation_closure(
        "Theta",
        parse_theta,
        lambda t1, t2: (-t1, -t2),
    ),
    transformation_closure(
        "Khovanov",
        parse_khovanov,
        lambda q, t: (-q, -t),
    ),
])

display(closure_audit)


# ============================================================
# C. Validar empíricamente la transformación HOMFLY
# ============================================================
def validate_homfly_mirror(sample_size=5000, seed=20260819):
    invariant = "HOMFLY-PT"
    source = REPRESENTATION_SPECS[invariant]["source"]
    cols = list(feature_cols_dict[invariant])

    coordinates = [parse_homfly(col) for col in cols]
    coordinate_to_index = {
        coordinate: index
        for index, coordinate in enumerate(coordinates)
    }

    permutation = np.array([
        coordinate_to_index[(-a, z)]
        for a, z in coordinates
    ])

    df = aligned["cleaned_tables"][source].copy()
    df["knot_id_base"] = df["knot_id_base"].astype(str)
    df["_mirror"] = (
        df["knot_id_clean"]
        .astype(str)
        .str.contains("!", regex=False)
    )

    plain_ids = set(
        df.loc[~df["_mirror"], "knot_id_base"]
    )
    mirror_ids = set(
        df.loc[df["_mirror"], "knot_id_base"]
    )
    paired_ids = sorted(plain_ids & mirror_ids)

    rng = np.random.default_rng(seed)
    if len(paired_ids) > sample_size:
        paired_ids = sorted(
            rng.choice(
                paired_ids,
                size=sample_size,
                replace=False,
            )
        )

    def matrix_for(mirror):
        return (
            df.loc[
                df["_mirror"].eq(mirror)
                & df["knot_id_base"].isin(paired_ids)
            ]
            .drop_duplicates("knot_id_base")
            .set_index("knot_id_base")
            .loc[paired_ids, cols]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0)
            .to_numpy(float)
        )

    plain = matrix_for(False)
    observed_mirror = matrix_for(True)
    predicted_mirror = plain[:, permutation]

    row_matches = np.all(
        np.isclose(
            predicted_mirror,
            observed_mirror,
            atol=1e-10,
            rtol=1e-10,
        ),
        axis=1,
    )

    return pd.DataFrame([{
        "invariant": invariant,
        "n_pairs_tested": len(paired_ids),
        "exact_row_match_prop": float(row_matches.mean()),
        "max_abs_error": float(
            np.max(
                np.abs(
                    predicted_mirror
                    - observed_mirror
                )
            )
        ),
        "transformation": "(a,z) -> (-a,z)",
    }])


homfly_mirror_validation = validate_homfly_mirror()
display(homfly_mirror_validation)

assert (
    homfly_mirror_validation[
        "exact_row_match_prop"
    ].iloc[0] == 1.0
)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 1. ¿Theta ya es simétrico bajo (T1,T2) -> (T1^-1,T2^-1)?
#    Se procesa por bloques para no duplicar la matriz completa.
# ============================================================
theta_cols = list(feature_cols_dict["Theta"])
theta_coords = [parse_theta(col) for col in theta_cols]
theta_index = {
    coord: i
    for i, coord in enumerate(theta_coords)
}

theta_inversion_perm = np.array([
    theta_index[(-t1, -t2)]
    for t1, t2 in theta_coords
])

X_theta = np.asarray(X_dict["Theta"])

theta_equal_rows = 0
theta_max_abs_difference = 0.0
chunk_size = 5000

for start in range(0, len(X_theta), chunk_size):
    stop = min(start + chunk_size, len(X_theta))

    original = X_theta[start:stop]
    inverted = original[:, theta_inversion_perm]

    difference = np.abs(original - inverted)

    theta_equal_rows += int(
        np.all(
            np.isclose(
                original,
                inverted,
                atol=1e-10,
                rtol=1e-10,
            ),
            axis=1,
        ).sum()
    )

    theta_max_abs_difference = max(
        theta_max_abs_difference,
        float(difference.max()),
    )

theta_inversion_audit = pd.DataFrame([{
    "n_knots": len(X_theta),
    "equal_under_simultaneous_inversion":
        theta_equal_rows,
    "equal_prop":
        theta_equal_rows / len(X_theta),
    "max_abs_difference":
        theta_max_abs_difference,
    "candidate_transform":
        "(T1,T2) -> (T1^-1,T2^-1)",
}])

display(theta_inversion_audit)


# ============================================================
# 2. Construir el espacio cerrado de Khovanov
# ============================================================
kh_cols = list(feature_cols_dict["Khovanov"])
kh_coords = [parse_khovanov(col) for col in kh_cols]

kh_original_set = set(kh_coords)
kh_reflected_set = {
    (-q, -t)
    for q, t in kh_coords
}
kh_union_set = kh_original_set | kh_reflected_set

# Orden reproducible: primero los 373 originales,
# después únicamente las coordenadas reflejadas faltantes.
kh_missing_reflections = sorted(
    kh_reflected_set - kh_original_set
)

kh_union_coords = (
    kh_coords
    + kh_missing_reflections
)

kh_union_index = {
    coord: i
    for i, coord in enumerate(kh_union_coords)
}

# Posiciones en el espacio ampliado para una fila original
kh_original_positions = np.array([
    kh_union_index[(q, t)]
    for q, t in kh_coords
])

# Posiciones en el espacio ampliado para su espejo
kh_mirror_positions = np.array([
    kh_union_index[(-q, -t)]
    for q, t in kh_coords
])

# La reflexión debe ser inyectiva e involutiva
assert len(np.unique(kh_original_positions)) == len(kh_coords)
assert len(np.unique(kh_mirror_positions)) == len(kh_coords)

for q, t in kh_union_coords:
    assert (-q, -t) in kh_union_index

kh_union_audit = pd.DataFrame([{
    "original_dimension": len(kh_coords),
    "missing_reflected_coordinates":
        len(kh_missing_reflections),
    "mirror_closed_dimension":
        len(kh_union_coords),
    "reflection_is_involution": True,
    "expected_dimension": 398,
}])

display(kh_union_audit)

print("Missing Khovanov coordinates:")
print(kh_missing_reflections)


# ============================================================
# 3. Verificar preservación de norma sin construir 313k x 398
# ============================================================
X_kh = np.asarray(X_dict["Khovanov"])

original_sq_norm = np.sum(
    X_kh.astype(np.float64) ** 2,
    axis=1,
)

# La reflexión solo recoloca los mismos coeficientes.
mirrored_sq_norm = original_sq_norm.copy()

kh_norm_audit = pd.DataFrame([{
    "n_knots": len(X_kh),
    "raw_norm_preserved_prop": float(
        np.isclose(
            original_sq_norm,
            mirrored_sq_norm,
            atol=1e-12,
            rtol=1e-12,
        ).mean()
    ),
    "maximum_sq_norm_difference": float(
        np.max(
            np.abs(
                original_sq_norm
                - mirrored_sq_norm
            )
        )
    ),
}])

display(kh_norm_audit)


# ============================================================
# 4. Guardar los audits
# ============================================================
MIRROR_B_DIR = (
    Path(OUTPUT_DIR)
    / "18B_mirror_representation_robustness"
)
MIRROR_B_DIR.mkdir(parents=True, exist_ok=True)

target_free_canonicalization_audit.to_csv(
    MIRROR_B_DIR
    / "target_free_canonicalization_audit.csv",
    index=False,
)

closure_audit.to_csv(
    MIRROR_B_DIR
    / "mirror_coordinate_closure_audit.csv",
    index=False,
)

homfly_mirror_validation.to_csv(
    MIRROR_B_DIR
    / "homfly_exact_mirror_validation.csv",
    index=False,
)

theta_inversion_audit.to_csv(
    MIRROR_B_DIR
    / "theta_inversion_symmetry_audit.csv",
    index=False,
)

kh_union_audit.to_csv(
    MIRROR_B_DIR
    / "khovanov_mirror_union_audit.csv",
    index=False,
)

print("Saved to:", MIRROR_B_DIR)

In [ ]:
get_ipython().run_line_magic("run", '-i "' + str(PROJECT_DIR / 'notebooks/stage18B_random_mirror_selection.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '-i "' + str(PROJECT_DIR / 'notebooks/stage18C_random_mirror_exact_nulls.py') + '"')

In [ ]:
display(mirror_random_decision)
display(global_positive_control_audit)

In [ ]:
get_ipython().run_line_magic("run", '-i "' + str(PROJECT_DIR / 'notebooks/stage19_conditional_heldout_validation.py') + '"')

In [ ]:
display(conditional_heldout_family_summary)
display(conditional_heldout_overlap)
display(conditional_heldout_phenotype)

In [ ]:
get_ipython().run_line_magic("run", '-i "' + str(PROJECT_DIR / 'notebooks/stage20_mathematical_phenotype_tests.py') + '"')

In [ ]:
display(mathematical_phenotype_by_regime)
display(matched_mathematical_phenotype)
display(knotinfo_phenotype_tests)

In [ ]:
get_ipython().run_line_magic("run", '-i "' + str(PROJECT_DIR / 'notebooks/stage20B_no_khovanov_external_thickness.py') + '"')

In [ ]:
display(no_khovanov_external_observed)
display(no_khovanov_external_decision)

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage21_final_mlst_artifacts.py') + '"')

In [ ]:
import numpy as np
import pandas as pd


# ============================================================
# Attach SnapPy symmetry order to geometry_match_base
# ============================================================

ID_CANDIDATES = [
    "knot_id_base",
    "internal_name",
    "knot_id_clean",
    "knot_id",
    "snappy_name",
]


def normalize_knot_id(values):
    return (
        values.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"^k", "", regex=True)
    )


# Find every in-memory DataFrame containing symmetry_order.
symmetry_sources = []

for object_name, obj in list(globals().items()):

    if not isinstance(obj, pd.DataFrame):
        continue

    if "symmetry_order" not in obj.columns:
        continue

    id_col = next(
        (
            col
            for col in ID_CANDIDATES
            if col in obj.columns
        ),
        None,
    )

    if id_col is None:
        continue

    tmp = obj[
        [id_col, "symmetry_order"]
    ].copy()

    tmp["knot_key"] = normalize_knot_id(
        tmp[id_col]
    )

    tmp["symmetry_order"] = pd.to_numeric(
        tmp["symmetry_order"],
        errors="coerce",
    )

    tmp["source_object"] = object_name

    symmetry_sources.append(
        tmp[
            [
                "knot_key",
                "symmetry_order",
                "source_object",
            ]
        ]
    )

    print(
        f"Symmetry source: {object_name}, "
        f"shape={obj.shape}, key={id_col}"
    )


if not symmetry_sources:
    raise RuntimeError(
        "No encontré en memoria ningún DataFrame con "
        "'symmetry_order'. Ejecuta primero los bloques de "
        "SnapPy para candidatos y controles."
    )


symmetry_long = pd.concat(
    symmetry_sources,
    ignore_index=True,
)

symmetry_long = symmetry_long.dropna(
    subset=[
        "knot_key",
        "symmetry_order",
    ]
)


# Check that repeated knots do not have conflicting values.
symmetry_conflicts = (
    symmetry_long
    .groupby("knot_key")["symmetry_order"]
    .nunique()
)

symmetry_conflicts = symmetry_conflicts[
    symmetry_conflicts > 1
]

if not symmetry_conflicts.empty:
    raise RuntimeError(
        "Hay valores contradictorios de symmetry_order para: "
        + str(
            symmetry_conflicts.index[
                :10
            ].tolist()
        )
    )


symmetry_lookup = (
    symmetry_long
    .drop_duplicates(
        subset="knot_key",
        keep="first",
    )
    .set_index("knot_key")[
        "symmetry_order"
    ]
)


# Identify the knot-ID column in the matching frame.
match_id_col = next(
    (
        col
        for col in ID_CANDIDATES
        if col in geometry_match_base.columns
    ),
    None,
)

if match_id_col is None:
    raise RuntimeError(
        "geometry_match_base no contiene una columna "
        "identificadora de nudo."
    )


geometry_match_base = geometry_match_base.copy()

geometry_match_base["knot_key"] = normalize_knot_id(
    geometry_match_base[match_id_col]
)

geometry_match_base["symmetry_order"] = (
    geometry_match_base["knot_key"]
    .map(symmetry_lookup)
)


# Check coverage only among knots required by the frozen matching.
required_indices = np.unique(
    np.concatenate(
        [
            pairs[
                [
                    "selected_index",
                    "control_index",
                ]
            ].to_numpy().ravel()
            for pairs in geometry_unique_pairs.values()
        ]
    )
)

required_geometry = geometry_match_base.loc[
    required_indices
]

n_missing = int(
    required_geometry[
        "symmetry_order"
    ].isna().sum()
)

print(
    "\nRequired matched knots:",
    len(required_geometry),
)

print(
    "With symmetry order:",
    len(required_geometry) - n_missing,
)

print(
    "Missing symmetry order:",
    n_missing,
)


if n_missing:
    print(
        "\nAdvertencia: los nudos anteriores tienen "
        "symmetry_order desconocido por triangulaciones "
        "no positivas. Se aplicará análisis por pares completos."
    )


print(
    "\n'symmetry_order' attached successfully."
)

In [ ]:
from scipy.stats import binomtest
import numpy as np
import pandas as pd


# ============================================================
# Binary endpoint: nontrivial symmetry
# ============================================================

SYMMETRY_REGIMES = [
    "amplitude_only",
    "conditional_only",
]

symmetry_rows = []

for regime_name in SYMMETRY_REGIMES:

    pairs = geometry_unique_pairs[
        regime_name
    ].copy()

    selected_order = pd.to_numeric(
        geometry_match_base.loc[
            pairs["selected_index"].to_numpy(),
            "symmetry_order",
        ],
        errors="coerce",
    ).reset_index(drop=True)

    control_order = pd.to_numeric(
        geometry_match_base.loc[
            pairs["control_index"].to_numpy(),
            "symmetry_order",
        ],
        errors="coerce",
    ).reset_index(drop=True)

    # Pairwise-complete analysis.
    valid = (
        selected_order.notna()
        &
        control_order.notna()
    )

    selected_order = selected_order.loc[
        valid
    ].to_numpy()

    control_order = control_order.loc[
        valid
    ].to_numpy()

    # Nontrivial symmetry means group order > 1.
    selected_nontrivial = (
        selected_order > 1
    )

    control_nontrivial = (
        control_order > 1
    )

    # Discordant pairs for exact McNemar.
    selected_yes_control_no = int(
        np.sum(
            selected_nontrivial
            &
            ~control_nontrivial
        )
    )

    selected_no_control_yes = int(
        np.sum(
            ~selected_nontrivial
            &
            control_nontrivial
        )
    )

    n_discordant = (
        selected_yes_control_no
        +
        selected_no_control_yes
    )

    if n_discordant == 0:
        p_value = 1.0
    else:
        p_value = binomtest(
            selected_yes_control_no,
            n=n_discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue

    selected_prop = float(
        selected_nontrivial.mean()
    )

    control_prop = float(
        control_nontrivial.mean()
    )

    symmetry_rows.append(
        {
            "regime": regime_name,
            "outcome": (
                "Nontrivial symmetry-group order"
            ),
            "definition": (
                "symmetry_order > 1"
            ),
            "n_total_pairs": len(pairs),
            "n_valid_pairs": int(valid.sum()),
            "n_excluded_pairs": int(
                (~valid).sum()
            ),
            "selected_prop": selected_prop,
            "control_prop": control_prop,
            "risk_difference": (
                selected_prop
                -
                control_prop
            ),
            "selected_yes_control_no":
                selected_yes_control_no,
            "selected_no_control_yes":
                selected_no_control_yes,
            "n_discordant": n_discordant,
            "p_value": p_value,
            "test": "exact paired McNemar",
        }
    )


binary_symmetry_results = pd.DataFrame(
    symmetry_rows
)


# ============================================================
# BH and Holm corrections across the two planned comparisons
# ============================================================

def bh_adjust(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    m = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]

    adjusted_ranked = (
        ranked
        *
        m
        /
        np.arange(1, m + 1)
    )

    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[::-1]
    )[::-1]

    adjusted = np.empty(m)
    adjusted[order] = np.minimum(
        adjusted_ranked,
        1.0,
    )

    return adjusted


def holm_adjust(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    m = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]

    adjusted_ranked = (
        ranked
        *
        (m - np.arange(m))
    )

    adjusted_ranked = np.maximum.accumulate(
        adjusted_ranked
    )

    adjusted = np.empty(m)
    adjusted[order] = np.minimum(
        adjusted_ranked,
        1.0,
    )

    return adjusted


binary_symmetry_results["q_bh"] = bh_adjust(
    binary_symmetry_results["p_value"]
)

binary_symmetry_results["p_holm"] = holm_adjust(
    binary_symmetry_results["p_value"]
)

binary_symmetry_results[
    "bh_significant"
] = (
    binary_symmetry_results["q_bh"]
    < 0.05
)

binary_symmetry_results[
    "holm_significant"
] = (
    binary_symmetry_results["p_holm"]
    < 0.05
)


display(binary_symmetry_results)

In [ ]:
SYMMETRY_OUTPUT_DIR = (
    OUTPUT_DIR
    / "21_final_mlst_artifacts"
)

SYMMETRY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

binary_symmetry_results.to_csv(
    SYMMETRY_OUTPUT_DIR
    / "binary_nontrivial_symmetry_matched.csv",
    index=False,
)

print(
    "Saved to:",
    SYMMETRY_OUTPUT_DIR
    / "binary_nontrivial_symmetry_matched.csv",
)

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage22_gap_thickness_dependence.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage23_anomaly_score_baselines.py') + '"')


In [ ]:
get_ipython().run_line_magic("run", '-i "' + str(PROJECT_DIR / "notebooks/stage23C_relative_withheld_khovanov.py") + '"')


In [ ]:
get_ipython().run_line_magic("run", '-i "' + str(PROJECT_DIR / "notebooks/stage23D_conditional_withheld_khovanov.py") + '"')


In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage24_mirror_support_directional_audit.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage23B_size_matched_score_sensitivity.py') + '"')

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    (__import__("os").environ.get('KNOT_OUTPUT_DIR', '/content/drive/MyDrive/Colab Notebooks/data_invariants/Invariants/processed_consensus_hardness/corrected_run_20260819') + '')
)

# ---------------------------------------------------------
# 1. Sensibilidad al número de bins
# ---------------------------------------------------------
bin_path = (
    ROOT
    / "23_anomaly_score_baselines"
    / "conditional_bin_sensitivity.csv"
)

bin_df = pd.read_csv(bin_path)

bin_phenotypes = bin_df[
    bin_df["comparison_bins"].isna()
].copy()

requested_columns = [
    "n_bins",
    "n",
    "nonalternating_n",
    "abs_gap_positive_prop",
    "mean_abs_gap",
    "abs_gap_ge_4_prop",
    "kh_diagonal_mean",
    "kh_diagonal_ge_3_prop",
    "kh_support_mean",
    "crossing_15_prop",
    "alternating_prop",
]

available_columns = [
    column
    for column in requested_columns
    if column in bin_phenotypes.columns
]

print("Conditional phenotype by number of bins:")
display(
    bin_phenotypes[
        available_columns
    ].sort_values("n_bins")
)

bin_jaccard = (
    bin_df[
        bin_df["comparison_bins"].notna()
    ]
    .pivot(
        index="n_bins",
        columns="comparison_bins",
        values="jaccard",
    )
    .sort_index()
    .sort_index(axis=1)
)

print("Jaccard across bin counts:")
display(bin_jaccard)

# ---------------------------------------------------------
# 2. Verificación final de artefactos
# ---------------------------------------------------------
required = [
    ROOT / "22_gap_thickness_dependence"
    / "gap_thickness_conditioned_null_summary.csv",

    ROOT / "22_gap_thickness_dependence"
    / "no_khovanov_external_with_kh_norm_summary.csv",

    ROOT / "23_anomaly_score_baselines"
    / "score_norm_correlation_by_view.csv",

    ROOT / "23_anomaly_score_baselines"
    / "score_family_external_phenotypes.csv",

    ROOT / "23_anomaly_score_baselines"
    / "figures"
    / "figure_score_baseline_comparison.png",

    ROOT / "23B_size_matched_score_sensitivity"
    / "size_matched_score_phenotypes.csv",

    ROOT / "23B_size_matched_score_sensitivity"
    / "size_matched_aggregate_norm_diagnostics.csv",

    ROOT / "23B_size_matched_score_sensitivity"
    / "figures"
    / "figure_size_matched_score_comparison.png",

    ROOT / "24_mirror_support_directional_audit"
    / "mirror_support_selection_bridge.csv",

    ROOT / "24_mirror_support_directional_audit"
    / "mirror_directional_gap_audit.csv",

    ROOT / "24_mirror_support_directional_audit"
    / "table3_mirror_nulls_with_canonical_support_rows.csv",
]

artifact_check = pd.DataFrame({
    "path": [str(path) for path in required],
    "exists": [path.exists() for path in required],
    "size_bytes": [
        path.stat().st_size if path.exists() else None
        for path in required
    ],
})

display(artifact_check)

assert artifact_check["exists"].all(), (
    "Faltan artefactos:\n"
    + "\n".join(
        artifact_check.loc[
            ~artifact_check["exists"], "path"
        ]
    )
)

artifact_check.to_csv(
    ROOT / "final_mlst_revision_artifact_check.csv",
    index=False,
)

print("FINAL COMPUTATIONAL REVISION COMPLETE")
print("You can safely close the Colab session.")

In [ ]:
%pip install -q snappy snappy_15_knots statsmodels

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage25A_revision_audit_tables.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage25_score_matched_geometry.py') + '"')

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    (__import__("os").environ.get('KNOT_OUTPUT_DIR', '/content/drive/MyDrive/Colab Notebooks/data_invariants/Invariants/processed_consensus_hardness/corrected_run_20260819') + '')
)

ATLAS_PATH = (
    ROOT / "17_final_paper_outputs/final_hard_regime_atlas.parquet"
)
PAIRS_PATH = (
    ROOT / "25_score_matched_geometry/score_geometry_pairs.csv"
)

ID_COL = "knot_id_base"
NORM_COLS = ["mean_log_sq_norm", "max_log_sq_norm"]

atlas = pd.read_parquet(
    ATLAS_PATH,
    columns=[ID_COL, *NORM_COLS],
)
atlas[ID_COL] = atlas[ID_COL].astype(str)

pairs = pd.read_csv(
    PAIRS_PATH,
    dtype={"selected_id": str, "control_id": str},
)

selected_lookup = atlas.rename(
    columns={
        ID_COL: "selected_id",
        **{c: f"selected_{c}" for c in NORM_COLS},
    }
)

control_lookup = atlas.rename(
    columns={
        ID_COL: "control_id",
        **{c: f"control_{c}" for c in NORM_COLS},
    }
)

matched = (
    pairs
    .merge(selected_lookup, on="selected_id", how="left")
    .merge(control_lookup, on="control_id", how="left")
)

def conventional_smd(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    pooled_sd = np.sqrt(
        (np.var(x, ddof=1) + np.var(y, ddof=1)) / 2
    )
    return (
        (np.mean(x) - np.mean(y)) / pooled_sd
        if pooled_sd > 0
        else np.nan
    )

rows = []

for method, group in matched.groupby("method", sort=False):
    row = {
        "method": method,
        "n_pairs": len(group),
    }

    smds = []

    for col in NORM_COLS:
        value = conventional_smd(
            group[f"selected_{col}"],
            group[f"control_{col}"],
        )
        row[f"smd_{col}"] = value
        smds.append(abs(value))

    row["max_abs_group_norm_smd"] = max(smds)
    rows.append(row)

balance = pd.DataFrame(rows)

display(balance)
balance.to_csv(
    ROOT
    / "25_score_matched_geometry/"
      "score_geometry_conventional_balance.csv",
    index=False,
)

In [ ]:
%env STAGE26_BOOTSTRAP_REPS=5000
%env STAGE26_BOOTSTRAP_SEED=20261226
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage26_bootstrap_confidence_intervals.py') + '"')

In [ ]:
%env STAGE27_NULL_REPS=5000
%env STAGE27_NULL_SEED=20261227
%env STAGE27_KH_NORM_BINS=100,50,25,20,10,5,4,3,2,1
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage27_khovanov_norm_bin_null_sensitivity.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage28_definition_provenance_audit.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage29_statistical_null_audit.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage30_tie_boundary_audit.py') + '"')

In [ ]:
import pandas as pd

p = (
    (__import__("os").environ.get('KNOT_OUTPUT_DIR', '/content/drive/MyDrive/Colab Notebooks/data_invariants/Invariants/processed_consensus_hardness/corrected_run_20260819') + '/29_statistical_null_audit/joint_fixed_cardinality_null_results.csv')
)

joint = pd.read_csv(p)

display(
    joint.sort_values(
        ["analysis", "joint_norm_bins_per_view", "metric"],
        ascending=[True, False, True]
    )
)

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage29b_final_joint_null.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage29c_continuous_amplitude_residualized.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage29c_continuous_amplitude_residualized_v2.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage32_representation_robustness_audits.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage32_representation_robustness_audits_v2.py') + '"')

In [ ]:
# 1. Verify Theta mirror action
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage31A_theta_mirror_action_audit.py') + '"')

In [ ]:
# 2. Correct mirror experiment, Khovanov completely withheld
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage31B_corrected_four_view_mirror_withheld_khovanov.py') + '"')

In [ ]:
# 3. Statistical repair
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage32_gcm_width_association.py') + '"')

In [ ]:
# 4. Representation/view ablation + fiber audit
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage33_view_ablation_and_fibers.py') + '"')

In [ ]:
# 5. Topological consequence audit
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage34_turaev_width_consequence_audit.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage32B_multiplier_gcm_calibration.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage33B_example_pair_extraction.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage35_khovanov_grading_width_audit.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage31_crossing_number_extrapolation.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage36_euler_cancellation_endpoint.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage38_residual_compression_sensitivity.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage37_mirror_symmetrized_score.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage39_per_norm_matching_balance.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage40_revision_audit_v2.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage41_theta_preprocessing.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage42_crossing15_grid.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage43_split_tuple_audit.py') + '"')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage44_theta_float64_verification.py') + '"')

In [ ]:
# Exact fibers precede diagram verification.
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / "notebooks/stage45_exact_fiber_khovanov_audit.py") + '"')


In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage46_independent_diagram_verification.py') + '" --theta required --install-sage')

In [ ]:
get_ipython().run_line_magic("run", '"' + str(PROJECT_DIR / 'notebooks/stage47_fiber_grouped_validation.py') + '"')

## Imagenes

Figures are generated from explicit saved tables using the cell below.


In [ ]:
subprocess.check_call([sys.executable, str(PROJECT_DIR / "scripts/make_paper_figures.py"),
    "--root", str(OUTPUT_DIR), "--out", str(PROJECT_DIR / "figures"), "--figures", "1"])


In [ ]:
subprocess.check_call([sys.executable, str(PROJECT_DIR / "scripts/make_paper_figures.py"),
    "--root", str(OUTPUT_DIR), "--out", str(PROJECT_DIR / "figures"), "--figures", "2"])


In [ ]:
subprocess.check_call([sys.executable, str(PROJECT_DIR / "scripts/make_paper_figures.py"),
    "--root", str(OUTPUT_DIR), "--out", str(PROJECT_DIR / "figures"), "--figures", "3"])


In [ ]:
subprocess.check_call([sys.executable, str(PROJECT_DIR / "scripts/make_paper_figures.py"),
    "--root", str(OUTPUT_DIR), "--out", str(PROJECT_DIR / "figures"), "--figures", "4"])
